# Hafta 14 · 2. bölüm · GPT, DeepSeek, LLaMA ve Açık Modeller

**Yapay Zekâya Giriş — Laboratuvar Defteri**  
Dr. Seda Bayat Toksöz · Iğdır Üniversitesi · 2026–2027 Güz

Bu defterde altı küçük proje var: (P1) bir modelin parametrelerini config.json'dan saymak, (P2) bellek/GPU hesaplayıcı, (P3) sohbet şablonu ve kendi tokenizer'ımız, (P4) ders notları üzerinde RAG, (P5) LoRA/QLoRA ince ayar tarifi, (P6) modeli kendi bilgisayarınızda çalıştırmak. İnternet gerektiren hücreler ayrıca işaretlidir.

> Colab'da eksik kütüphane olursa: `!pip install -q -U transformers peft` (P1 için transformers ≥ 4.56 gerekir).

Hücreleri yukarıdan aşağıya sırayla çalıştırın (**Shift + Enter**). Parametreleri değiştirip sonucun nasıl değiştiğini gözlemleyin.

In [1]:
# Grafik ayarları (ders notlarıyla aynı renk paleti)
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#0072B2", "#D55E00", "#009E73", "#E69F00", "#56B4E9", "#CC79A7"]),
})

## P1 · Bir modelin parametrelerini config.json'dan saymak
Değerler `meta-llama/Llama-3.1-8B` deposundaki config.json dosyasından alındı. Önce elle (formülle), sonra PyTorch ile sayıyoruz.

In [2]:
V, d, ff = 128256, 4096, 14336     # vocab_size, hidden, intermediate
L, h, kv = 32, 32, 8               # katman, Q başı, KV başı
hd = d // h                        # baş boyutu = 128
gomme  = V * d                     # token gömme tablosu
dikkat = 2*d*d + 2*d*(kv*hd)       # Q, O tam; K, V küçük (GQA)
ffn    = 3 * d * ff                # SwiGLU: gate, up, down
blok   = dikkat + ffn + 2*d        # + 2 RMSNorm
toplam = gomme + L*blok + d + V*d  # + son norm + çıkış katmanı
for ad, v in [("gömme", gomme), ("1 blok", blok),
              ("32 blok", L*blok), ("TOPLAM", toplam)]:
    print(f"{ad:8s}{v:>15,}".replace(",", " "))

gömme       525 336 576
1 blok      218 112 000
32 blok   6 979 584 000
TOPLAM    8 030 261 248


PyTorch ile aynı modeli **meta** cihazda kuruyoruz: ağırlıklar için bellek ayrılmaz, yalnızca şekiller oluşur. 8 milyar parametreli model bir saniyede kurulur ve sayılır.

In [3]:
import torch
from transformers import AutoModelForCausalLM, LlamaConfig
cfg = dict(vocab_size=V, hidden_size=d, intermediate_size=ff,
           num_hidden_layers=L, num_attention_heads=h,
           num_key_value_heads=kv, tie_word_embeddings=False)
with torch.device("meta"):         # bellek harcamadan kur
    m = AutoModelForCausalLM.from_config(LlamaConfig(**cfg))
n = sum(p.numel() for p in m.parameters())
print("PyTorch:", f"{n:,}".replace(",", " "), "| aynı mı?", n == toplam)

PyTorch: 8 030 261 248 | aynı mı? True


### Altı modelin mimarisi
Yoğun modellerde değerler resmî config.json dosyalarından girildi; DeepSeek-V3 ve gpt-oss-120b için `transformers`'ın varsayılan config değerleri resmî modellerinkiyle aynıdır (toplamlar 671 milyar ve 116.8 milyar çıkıyor).

In [4]:
from transformers import (Qwen2Config, MistralConfig, MixtralConfig,
                          DeepseekV3Config, GptOssConfig)
ortak = dict(hidden_size=4096, intermediate_size=14336,
             num_hidden_layers=32, num_attention_heads=32,
             num_key_value_heads=8, vocab_size=32000)
qwen = Qwen2Config(vocab_size=152064, hidden_size=3584,
                   intermediate_size=18944, num_hidden_layers=28,
                   num_attention_heads=28, num_key_value_heads=4)
modeller = {                      # (config, uzman sayısı E, seçilen k)
    "Llama-3.1-8B": (LlamaConfig(**cfg), 1, 1),
    "Qwen2.5-7B":   (qwen, 1, 1),
    "Mistral-7B":   (MistralConfig(**ortak), 1, 1),
    "Mixtral-8x7B": (MixtralConfig(**ortak, num_local_experts=8,
                                   num_experts_per_tok=2), 8, 2),
    "DeepSeek-V3":  (DeepseekV3Config(), 256, 8),
    "gpt-oss-120b": (GptOssConfig(), 128, 4),
}

### Toplam ve token başına aktif parametre
MoE katmanında E uzmandan yalnızca k tanesi çalışır: aktif = toplam − uzman parametreleri × (1 − k/E). Şirketler aktif sayıyı farklı sayar: kimi gömme tablosunu dahil eder (Mixtral 12.9 milyar, DeepSeek-V3 37 milyar), kimi etmez (gpt-oss 5.1 milyar).

In [5]:
print(f"{'model':13s}{'toplam':>8s}{'aktif':>8s}{'gömmesiz':>9s}")
for ad, (c, E, k) in modeller.items():
    with torch.device("meta"):
        m = AutoModelForCausalLM.from_config(c)
    top = sum(p.numel() for p in m.parameters())
    uzm = sum(p.numel() for n, p in m.named_parameters()
              if "mlp.experts" in n)       # yönlendirilen uzmanlar
    gom = m.get_input_embeddings().weight.numel()
    akt = top - uzm * (1 - k / E)
    print(f"{ad:13s}{top/1e9:7.1f}B{akt/1e9:7.2f}B{(akt-gom)/1e9:8.2f}B")

model          toplam   aktif gömmesiz
Llama-3.1-8B     8.0B   8.03B    7.50B
Qwen2.5-7B       7.6B   7.62B    7.07B
Mistral-7B       7.2B   7.24B    7.11B
Mixtral-8x7B    46.7B  12.88B   12.75B


DeepSeek-V3    671.0B  37.55B   36.63B
gpt-oss-120b   116.8B   5.71B    5.13B


### Notlardaki Soru 12(c): bir MoE bloğu
50 milyon dikkat parametresi, her biri 100 milyon parametreli 16 FFN uzmanı, k = 2.

In [6]:
dikkat_p, uzman_p, E, k = 50e6, 100e6, 16, 2
print("toplam:", (dikkat_p + E * uzman_p) / 1e6, "milyon")
print("aktif :", (dikkat_p + k * uzman_p) / 1e6, "milyon")

toplam: 1650.0 milyon
aktif : 250.0 milyon


## P2 · Bellek hesaplayıcı: hangi model hangi GPU'ya sığar?
Ağırlık belleği = parametre sayısı × parametre başına bayt. Üstüne KV önbelleği ve çalışma ek yükü (≈ %20) gelir. Hesaplar kaba tahmindir; gerçek değer kütüphaneye ve ayarlara göre değişir.

In [7]:
def agirlik_gb(N_milyar, bit):             # yalnızca ağırlıklar
    return N_milyar * 1e9 * bit / 8 / 1e9

gpu = {"8GB": 8, "T4-16GB": 16, "24GB": 24, "80GB": 80}
print("model   BF16  8bit  4bit  | 4 bit + %20 ek yük sığar mı?")
for N in [1, 3, 8, 14, 32, 70, 120, 671]:
    g16, g8, g4 = (agirlik_gb(N, b) for b in (16, 8, 4))
    sig = "  ".join(f"{ad}:{'E' if g4 * 1.2 <= gb else '-'}"
                    for ad, gb in gpu.items())
    print(f"{N:>4}B {g16:6.0f}{g8:6.0f}{g4:6.1f}  | {sig}")

model   BF16  8bit  4bit  | 4 bit + %20 ek yük sığar mı?
   1B      2     1   0.5  | 8GB:E  T4-16GB:E  24GB:E  80GB:E
   3B      6     3   1.5  | 8GB:E  T4-16GB:E  24GB:E  80GB:E
   8B     16     8   4.0  | 8GB:E  T4-16GB:E  24GB:E  80GB:E
  14B     28    14   7.0  | 8GB:-  T4-16GB:E  24GB:E  80GB:E
  32B     64    32  16.0  | 8GB:-  T4-16GB:-  24GB:E  80GB:E
  70B    140    70  35.0  | 8GB:-  T4-16GB:-  24GB:-  80GB:E
 120B    240   120  60.0  | 8GB:-  T4-16GB:-  24GB:-  80GB:E
 671B   1342   671 335.5  | 8GB:-  T4-16GB:-  24GB:-  80GB:-


### KV önbelleği bağlam uzadıkça büyür (Llama-3.1-8B, BF16)
Token başına 2 (K ve V) × katman × KV başı × baş boyutu × 2 bayt.

In [8]:
L, kv_bas, bas_boy = 32, 8, 128
tok_bayt = 2 * L * kv_bas * bas_boy * 2
print("token başına KV:", tok_bayt // 1024, "KB")
for baglam in [4_096, 32_768, 131_072]:
    print(f"{baglam:>7} token -> {tok_bayt*baglam/2**30:5.1f} GB")
print("GQA olmasaydı (32 KV başı) 128K:",
      round(tok_bayt * 4 * 131_072 / 2**30, 1), "GB")

token başına KV: 128 KB
   4096 token ->   0.5 GB
  32768 token ->   4.0 GB
 131072 token ->  16.0 GB
GQA olmasaydı (32 KV başı) 128K: 64.0 GB


## P3 · Sohbet şablonu: model aslında tek bir metin görür
Aşağıdaki iki fonksiyon Qwen (ChatML) ve Llama 3 biçimini sadeleştirerek elle kurar. Gerçek projede `tokenizer.apply_chat_template` kullanılır.

In [9]:
mesajlar = [{"role": "system", "content": "Kısa ve Türkçe cevap ver."},
            {"role": "user", "content": "LoRA nedir?"}]
def chatml(mesajlar):                         # Qwen biçimi
    s = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
                for m in mesajlar)
    return s + "<|im_start|>assistant\n"      # sıra modelde
def llama3(mesajlar):                         # Llama 3 biçimi
    s = "<|begin_of_text|>" + "".join(
        f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n"
        f"{m['content']}<|eot_id|>" for m in mesajlar)
    return s + "<|start_header_id|>assistant<|end_header_id|>\n\n"
print(chatml(mesajlar)); print("-" * 30); print(llama3(mesajlar))

<|im_start|>system
Kısa ve Türkçe cevap ver.<|im_end|>
<|im_start|>user
LoRA nedir?<|im_end|>
<|im_start|>assistant

------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Kısa ve Türkçe cevap ver.<|eot_id|><|start_header_id|>user<|end_header_id|>

LoRA nedir?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




### Ders notlarını dosyaya yazmak
Aşağıdaki hücre bu dersin notlarından alınmış düzyazıyı (≈150 bin karakter) `ders_notlari.txt` dosyasına yazar; P3 ve P4 bu dosyayı kullanır. Hücreyi bir kez çalıştırmanız yeterli.

In [10]:
NOTLAR = r"""Yapay zekâ, son on yılda bilgisayar biliminin en hızlı gelişen alanı oldu. Telefonunuzun kilidini açan yüz tanıma sistemi, e-posta kutunuzdaki istenmeyen posta filtresi, hastanelerde tümörleri işaretleyen görüntü analiz yazılımları ve sohbet ettiğiniz büyük dil modelleri aynı temel fikre dayanır: verilerden öğrenen modeller. Bu ders notları, Yazılım Mühendisliği 4. sınıf öğrencilerinin bu fikri hem kavramsal hem de hesaplama düzeyinde kavraması için hazırlandı. Notlar 14 haftalık ders planına göre düzenlenmiştir. İlk yedi hafta makine öğrenmesinin ve yapay sinir ağlarının temellerini kapsar: veri ve öğrenme türleri, regresyon, sınıflandırma, model değerlendirme, sinir ağları ve geri yayılım. İkinci yarıda bu temellerin üzerine görüntü işleme, evrişimli sinir ağları (CNN), görüntü segmentasyonu, doğal dil işleme, Transformer mimarisi ve büyük dil modelleri (LLM) inşa edilir. Her hafta bir öncekinin üzerine kurulduğu için konuları sırasıyla ça-
Bu dersteki amacımız formül ezberlemek değil, bir modelin neden ve nasıl çalıştığını anlamaktır. Bu yüzden her haftada adım adım çözülmüş sayısal örnekler bulacaksınız. Bu örnekleri yalnızca okumayın; kâğıt, kalem ve hesap makinesiyle kendiniz de çözün. Sınavlardaki hesaplama soruları bu örneklerin benzerleridir. Haftaların içindeki kısa Python kodları ise laboratuvar uygulamalarına hazırlık niteliğindedir; kodları kendi bilgisayarınızda ya da Google Colab üzerinde çalıştırarak sonuçları notlardaki elle yapılmış hesaplarla karşılaştırmanız, konuyu pekiştirmenin en iyi yoludur.
Vize ve final sınavları bu notlardaki konular, tanımlar, çözümlü örnekler ve her haftanın sonundaki değerlendirme soruları esas alınarak hazırlanır. Önerilen vize kapsamı 1–7. haftalardır; final sınavı tüm haftaları kapsar.
Her hafta aynı yapıyı izler. Haftanın başında o haftanın kazanımları ve anahtar kavramları listelenir; ardından konu anlatımı, şekiller ve çözümlü örnekler gelir. Hafta, kısa bir özet ve değerlendirme soruları ile biter. Sorular üç gruptur: (A) çoktan seçmeli sorular, (B) kısa cevaplı ve hesaplama soruları, (C) birden çok kavramı birleştiren ileri düzey sorular. Soruların cevapları Ek D’deki cevap anahtarında yer alır; önce kendiniz çözmeye çalışın, sonra kontrol edin. Metin içinde farklı amaçlara hizmet eden renkli kutular kullanılmıştır:
Konunun matematiksel arka planını, türetmeleri ve ileri düzey hesaplamaları içeren kutulardır. Temel anlatımı kesintiye uğratmadan bir adım daha derine inmek isteyenler içindir; sınavlardaki ileri düzey sorular bu kutulardaki içerikten yararlanır.
• Yuvarlama: Ara adımlarda genellikle 3–4 anlamlı basamak tutulmuştur. Sınavda kendi yuvarlamanızdan kaynaklanan küçük farklar sorun değildir. • Terimler: Türkçe terimlerin İngilizce karşılıkları ilk geçtikleri yerde parantez içinde verilmiştir. Tüm terimler Ek C’deki sözlükte alfabetik olarak toplanmıştır. • Görseller: Fotoğraf örnekleri, açık kaynaklı scikit-image kütüphanesinin örnek görüntülerinden alınmıştır. Diğer tüm şekil ve grafikler bu notlar için hazırlanmıştır.
Bu ders; yapay zekânın temel kavramlarını, makine öğrenmesi ve derin öğrenme yöntemlerini, görüntü ve metin verileri üzerinde çalışan modern mimarileri (CNN, Transformer, büyük dil modelleri) sezgisel ve hesaplamalı bir bakışla tanıtmayı amaçlar. Ders sonunda öğrencilerin bir yapay
zekâ problemini doğru öğrenme türüyle eşleştirebilmesi, temel modelleri elle ve kodla uygulayabilmesi ve modellerin başarımını doğru metriklerle yorumlayabilmesi hedeflenir.
• Her dersten önce ilgili haftanın kazanımlarını ve özetini okuyun; dersten sonra anlatımı ve çözümlü örnekleri ayrıntılı çalışın. • Çözümlü örnekleri önce kendiniz çözün, sonra çözümle karşılaştırın. • Değerlendirme sorularını cevap anahtarına bakmadan çözün; yanlış yaptığınız soruların konu-
• Python kodlarını çalıştırın, parametreleri değiştirerek sonucun nasıl değiştiğini gözlemleyin. • Ek B’deki formül özetini sınavdan önce tekrar listesi olarak kullanın.
Bu haftanın sonunda öğrenciler; • yapay zekânın farklı tanımlarını açıklayabilecek ve dar, genel ve süper yapay zekâ ayrımını
• geleneksel programlama ile makine öğrenmesi yaklaşımını karşılaştırabilecek, • yapay zekânın tarihsel gelişimindeki kilometre taşlarını sıralayabilecek, • akıllı ajan kavramını PEAS çerçevesiyle tanımlayabilecek, • A* gibi arama algoritmalarının çalışma mantığını açıklayıp küçük bir haritada uygulaya-
“Zekâ” kelimesini tanımlamak bile kolay değildir: Bir satranç ustasının hamle seçmesi, bir çocuğun yürümeyi öğrenmesi, bir doktorun röntgen filminde kırığı fark etmesi ya da bir arkadaşınızın şakanızı anlaması farklı zekâ biçimleridir. Yapay zekâ alanı bu yeteneklerin bir bölümünü bilgisayarlarla gerçekleştirmeyi amaçlar. Terimi ilk kez kullanan John McCarthy yapay zekâyı “akıllı makineler, özellikle de akıllı bilgisayar programları yapma bilimi ve mühendisliği” olarak tanımlamıştır.
Yapay zekâ (artificial intelligence, AI), normalde insan zekâsı gerektiren görevleri —algılama, öğrenme, akıl yürütme, problem çözme, karar verme ve dil anlama gibi— yerine getirebilen bilgisayar sistemleri geliştirmeyi amaçlayan bilim ve mühendislik dalıdır.
Yapay zekâ ders kitaplarının en bilineni olan Russell ve Norvig’in kitabı, alandaki tanımları iki soruya göre dört gruba ayırır: Sistem düşünmeye mi yoksa davranmaya mı odaklanıyor? Başarı ölçütü insana benzemek mi yoksa rasyonel (en doğru) olmak mı? Tablo 1.1 bu dört yaklaşımı özetler.
Alan Turing, 1950 yılında yayımladığı “Computing Machinery and Intelligence” makalesinde “Makineler düşünebilir mi?” sorusunun belirsiz olduğunu savunarak yerine ölçülebilir bir deney önermiştir: taklit oyunu (imitation game). Bugün Turing testi (Turing test) olarak bilinen bu deneyde bir sorgulayıcı, göremediği iki tarafla yalnızca yazılı mesajlarla konuşur (Şekil 1.1). Taraflardan biri insan, diğeri makinedir. Sorgulayıcı hangisinin makine olduğunu güvenilir biçimde ayırt edemiyorsa makine testi geçmiş sayılır.
Filozof John Searle (1980), Turing testine karşı ünlü “Çin Odası” düşünce deneyini önermiştir: Çince bilmeyen bir kişi, bir kural kitabını izleyerek kendisine verilen Çince sorulara kusursuz Çince cevaplar üretebilir; ancak yine de söylediklerinin hiçbirini anlamaz. Searle’e göre davranışı taklit etmek, anlamak demek değildir. Bu tartışma, büyük dil modelleri çağında yeniden
Yapay zekâ sistemleri yeteneklerinin genişliğine göre üç düzeyde incelenir (Tablo 1.2). Günümüzde kullanılan sistemlerin büyük çoğunluğu belirli bir görev için tasarlanmış dar yapay zekâ (narrow AI) örnekleridir. Büyük dil modelleri çok sayıda farklı görevi yerine getirebildikleri için “genel amaçlı” olarak nitelendirilse de insan düzeyinde genel yapay zekâya (artificial general intelligence, AGI) ulaşılıp ulaşılmadığı hâlâ tartışma konusudur.
Bu üç terim günlük dilde çoğu zaman birbirinin yerine kullanılır; oysa aralarında iç içe bir ilişki vardır (Şekil 1.2). Yapay zekâ en geniş kavramdır ve öğrenmeyen sistemleri de (örneğin el ile yazılmış kurallarla çalışan uzman sistemler veya arama algoritmaları) kapsar. Makine öğrenmesi yapay zekânın, derin öğrenme de makine öğrenmesinin bir alt alanıdır.
Makine öğrenmesi (machine learning, ML), bilgisayarların açıkça programlanmadan, verilerden ve deneyimden öğrenerek bir görevdeki başarımlarını artırmasını sağlayan yöntemleri incele-
Tom Mitchell (1997) daha biçimsel ve sınavlarda sıkça sorulan bir tanım vermiştir: Bir bilgisayar programı, T görevlerindeki P ölçütüyle ölçülen başarımı E deneyimiyle artıyorsa, bu program E deneyiminden öğreniyor denir. Burada T görev (task), P performans ölçütü (performance), E ise deneyimdir
• E (deneyim): Ziraat uzmanlarınca etiketlenmiş binlerce yaprak fotoğrafı. • P (performans): Daha önce görülmemiş test fotoğraflarında doğru sınıflandırılanların
Geleneksel programlamada yazılım mühendisi problemi çözen kuralları kendisi yazar; bilgisayar bu kuralları verilere uygulayarak cevap üretir. Makine öğrenmesinde ise bilgisayara veriler ve bu verilere ait doğru cevaplar (etiketler) verilir; bilgisayar bu ikisi arasındaki ilişkiyi, yani kuralları, kendisi çıkarır (Şekil 1.3). Öğrenilen bu kurallar kümesine model (model) denir ve model daha sonra yeni veriler için tahmin yapmakta kullanılır.
Bu farkı bir istenmeyen e-posta (spam) filtresi üzerinden düşünelim. Kural tabanlı bir filtre “konu satırında kazandınız geçiyorsa spam olarak işaretle” gibi yüzlerce kural içerir. Bu yaklaşımın üç temel sorunu vardır: (i) kurallar zamanla çoğalır ve bakımı zorlaşır, (ii) spam gönderenler kelimeleri değiştirerek (“kaz@ndınız”) kuralları kolayca atlatır, (iii) her dil ve her yeni saldırı türü için kural yazmak gerekir. Makine öğrenmesi tabanlı bir filtre ise kullanıcıların “spam” olarak işaretlediği binlerce e-postadan hangi kelime ve özelliklerin spam ile ilişkili olduğunu kendisi öğrenir ve yeni örnekler geldikçe yeniden eğitilerek güncellenir.
Derin öğrenme (deep learning, DL), çok sayıda katmandan oluşan yapay sinir ağlarıyla verinin basitten karmaşığa doğru hiyerarşik temsillerini otomatik olarak öğrenen makine öğrenmesi
Klasik makine öğrenmesi ile derin öğrenme arasındaki en önemli fark, öznitelik (feature) adı verilen ve modele girdi olarak verilen ölçülebilir özelliklerin nasıl elde edildiğidir. Klasik yaklaşımda uzman bir insan, örneğin bir görüntüden kenar yoğunluğu, renk histogramı veya doku ölçüleri gibi öznitelikleri elle tasarlar (öznitelik mühendisliği (feature engineering)). Derin öğrenmede ise ağ, ham pikselleri girdi olarak alır ve hangi özniteliklerin önemli olduğunu eğitim sırasında kendisi öğrenir
Son yıllarda en çok konuşulan alt alan ise üretken yapay zekâdır (generative AI): Bu modeller yalnızca sınıflandırma veya tahmin yapmakla kalmaz, eğitim verisine benzeyen yeni metin, görüntü, ses ve kod üretir. ChatGPT, Gemini ve Claude gibi sohbet asistanlarının temelindeki büyük dil modellerini 14. haftada ayrıntılı olarak inceleyeceğiz.
Yapay zekânın tarihi düz bir çizgide ilerlememiş; büyük beklentilerle geçen “bahar” dönemleri ile hayal kırıklığı ve fon kesintileriyle geçen YZ kışları (AI winters) birbirini izlemiştir. Şekil 1.5 başlıca kilometre taşlarını özetlemektedir. Tarihçeden çıkarılacak en önemli ders şudur: Derin öğrenmenin 2012’den sonraki yükselişi tek bir buluşa değil, üç etkenin aynı anda olgunlaşmasına dayanır.
2. Hesaplama gücü: Başlangıçta bilgisayar oyunları için geliştirilen grafik işlem birimleri (GPU), sinir ağlarının ihtiyaç duyduğu paralel matris çarpımlarını çok hızlı yapabiliyordu. 3. Algoritmik gelişmeler: ReLU aktivasyonu, dropout, daha iyi ağırlık başlatma ve optimizasyon yöntemleri derin ağların eğitimini kararlı hâle getirdi.
2024 Nobel Fizik Ödülü, yapay sinir ağlarıyla makine öğrenmesini mümkün kılan temel keşifleri nedeniyle John Hopfield ve Geoffrey Hinton’a verildi. Aynı yıl Nobel Kimya Ödülü’nün yarısı protein yapısı tahmini yapan AlphaFold modeli için Demis Hassabis ve John Jumper’a, diğer yarısı ise hesaplamalı protein tasarımı için David Baker’a verildi. Yapay zekâ böylece temel bilimlerin en prestijli ödüllerine konu oldu.
Modern yapay zekâda sistemler genellikle ajan (agent) kavramıyla modellenir. Ajan, çevresini algılayıcılar aracılığıyla algılayan ve çevresi üzerinde eyleyiciler aracılığıyla eylemde bulunan her şeydir (Şekil 1.6). Bir robot süpürge, bir satranç programı ya da bir sohbet asistanı birer ajandır. Bir ajan tasarlarken önce görev ortamını tanımlamak gerekir. Bunun için kullanılan çerçeveye, İngilizce bileşen adlarının baş harflerinden oluşan PEAS (Performance, Environment, Actuators, Sensors) denir. Bu çerçeve dört soruyu yanıtlar: Başarı neyle ölçülecek (performans ölçütü)? Ajan nerede çalışacak (ortam)? Hangi araçlarla eylemde bulunacak (eyleyiciler)? Çevreyi nasıl algılayacak
Ajanın çalıştığı ortamın özellikleri tasarım zorluğunu belirler. Ortam tam gözlemlenebilir (satranç tahtasının tamamı görünür) ya da kısmi gözlemlenebilir (taksi, köşenin arkasını göremez); deterministik (aynı hamle hep aynı sonucu verir) ya da stokastik (yayanın ne yapacağı kesin değildir); statik ya da dinamik olabilir. Otonom sürüş; kısmi gözlemlenebilir, stokastik ve dinamik bir ortam olduğu için en zor yapay zekâ problemlerinden biridir.
Makine öğrenmesi yaygınlaşmadan önce yapay zekânın temel aracı arama (search) idi ve bugün de rota planlama, oyun oynama ve otomatik planlama gibi birçok problem arama ile çözülür. Bir arama probleminde bir başlangıç durumu, her durumda uygulanabilecek eylemler, eylemlerin maliyetleri ve bir hedef vardır. Amaç, başlangıçtan hedefe giden en düşük maliyetli eylem dizisini bulmaktır. Haritada iki şehir arasındaki en kısa yol, 8-bulmacasının çözümü veya bir robotun engellerin arasından geçişi bu biçimde modellenir. Arama algoritmaları iki gruba ayrılır. Bilgisiz (uninformed) algoritmalar hedefe ne kadar yakın olduklarını bilmeden ilerler: Genişlik öncelikli arama (BFS) durumları başlangıçtan uzaklıklarına göre katman katman gezer; derinlik öncelikli arama (DFS) bir dalı sonuna kadar izler; tek biçimli maliyet araması (UCS) her adımda o ana kadarki toplam maliyeti g(n) en küçük olan durumu genişletir. Bilgili (informed) algoritmalar ise bir sezgisel fonksiyon (heuristic function) h(n) kullanır: h(n), n durumundan hedefe kalan maliyetin bir tahminidir (ör. haritada kuş uçuşu uzaklık). En ünlü bilgili arama algoritması olan A* araması (A* search), her adımda
değeri en küçük olan durumu genişletir. Sezgisel fonksiyon gerçek kalan maliyeti hiçbir zaman aşmıyorsa (kabul edilebilir sezgisel), A* en düşük maliyetli çözümü bulmayı garanti eder.
Aşağıdaki haritada şehirler arası yol maliyetleri ve her şehrin hedef G’ye kuş uçuşu uzaklık tahmini h verilmiştir. A* algoritmasıyla S’den G’ye en ucuz yolu bulunuz.
dönüşür ve aynı sonucu genellikle daha fazla düğüm genişleterek bulur (bu küçük haritada her iki yöntem de beş düğümün tamamını genişletir; fark büyük haritalarda ortaya çıkar).
Arama, problemin kurallarının açıkça bilindiği durumlarda (harita, satranç, bulmaca) çok güçlüdür; öğrenme ise kuralların veriden çıkarılması gerektiğinde devreye girer. Modern sistemler ikisini birleştirir: AlphaGo, hamle olasılıklarını derin bir ağla tahmin eder, en iyi hamleyi ise bu tahminlerle yönlendirilen bir ağaç araması bulur.
Yapay zekâ bugün hemen her sektörde kullanılmaktadır. Şekil 1.7 bu dönem boyunca inceleyeceğimiz yöntemlerin gerçek hayattaki karşılıklarından bazılarını göstermektedir.
Yapay zekâ sistemleri insanların hayatını doğrudan etkileyen kararlarda (işe alım, kredi onayı, tıbbi teşhis) kullanıldıkça etik sorular da önem kazanmaktadır. Bir yazılım mühendisi olarak geliştireceğiniz sistemlerde aşağıdaki ilkeleri göz önünde bulundurmanız gerekir:
• Yanlılık ve adalet: Model, eğitim verisindeki önyargıları öğrenir ve büyütebilir. Örneğin geçmişte çoğunlukla erkek adayların işe alındığı bir şirketin verileriyle eğitilen bir özgeçmiş değerlendirme sistemi, kadın adayları sistematik olarak düşük puanlayabilir. Bu duruma algoritmik yanlılık (algorithmic bias) denir. • Gizlilik ve veri koruma: Kişisel verilerin toplanması, saklanması ve işlenmesi yasal düzenlemelere uygun olmalıdır. Türkiye’de bu konuda 6698 sayılı Kişisel Verilerin Korunması Kanunu
• Şeffaflık ve açıklanabilirlik: Özellikle derin öğrenme modelleri “kara kutu” gibi davranabilir. Bir kredi başvurusunu reddeden modelin bu kararı neden verdiğini açıklayabilmesi gerekir. Bu ihtiyaç açıklanabilir yapay zekâ (explainable AI, XAI) alanını doğurmuştur. • Sorumluluk: Otonom bir aracın karıştığı kazada sorumluluğun kime (üretici, yazılımcı, kullanıcı) ait olduğu hukuki bir sorudur. Avrupa Birliği’nin Yapay Zekâ Yasası (AI Act), sistemleri risk düzeyine göre sınıflandıran ilk kapsamlı düzenlemelerden biridir. • Kötüye kullanım: Gerçeğinden ayırt edilmesi zor sahte ses ve videolar (deepfake), dezenformasyon ve otomatik siber saldırılar ciddi riskler oluşturur. • Akademik dürüstlük: Ödev ve projelerde üretken yapay zekâ araçlarını kullanırken ders kurallarına uymalı, kullanımı açıkça belirtmeli ve üretilen içeriğin doğruluğunu mutlaka kontrol
“Çöp girer, çöp çıkar” (garbage in, garbage out) ilkesi yapay zekâda da geçerlidir. Hatalı, eksik veya yanlı verilerle eğitilen bir model, ne kadar karmaşık olursa olsun güvenilir sonuçlar
Bu derste göreceğimiz hemen her model; girdileri sayı listeleri, yani vektörler (vectors) olarak alır ve bunlar üzerinde matris işlemleri yapar. Bir görüntü piksel değerlerinden, bir cümle kelime temsillerinden, bir hasta kaydı ise ölçüm değerlerinden oluşan bir vektöre dönüştürülür. Bu nedenle ilk haftada iki temel işlemi hatırlayalım (ayrıntılı tekrar için Ek A’ya bakınız). İç çarpım (dot product): Aynı boyuttaki iki vektörün karşılıklı elemanları çarpılıp toplanır:
(a) şıkkındaki işlem, 6. haftada göreceğimiz yapay nöronun yaptığı hesabın ta kendisidir: girdileri ağırlıklarıyla çarp, topla ve bias ekle. (c) şıkkı ise aynı anda iki nöronun hesabını yapar.
✓ Yapay zekâ, insan zekâsı gerektiren görevleri yerine getiren sistemler geliştirmeyi amaçlar; modern yaklaşım rasyonel ajan yaklaşımıdır. ✓ Turing testi, makinenin yazışmalarda insandan ayırt edilemeyecek davranış sergileyip ser-
✓ Derin öğrenme devrimi; büyük veri, GPU’larla gelen hesaplama gücü ve algoritmik gelişmelerin birleşimiyle 2012’de başladı. ✓ Ajan, algılayıcılarıyla çevreyi algılar ve eyleyicileriyle çevreye etki eder; görev ortamı PEAS
✓ Yanlılık, gizlilik, açıklanabilirlik ve sorumluluk, yapay zekâ sistemleri tasarlanırken gözetilmesi gereken temel etik ilkelerdir.
3. Makine öğrenmesi yaklaşımında sisteme verilenler ve sistemin ürettiği çıktı aşağıdakilerin hangisinde doğru eşleştirilmiştir?
4. Mitchell’in makine öğrenmesi tanımına göre, bir spam filtresi için “doğru sınıflandırılan epostaların yüzdesi” hangi bileşene karşılık gelir?
A Abartılı beklentilerin karşılanamaması B Dönemin bilgisayarlarının hesaplama gücünün yetersiz olması C Tek katmanlı perceptronun XOR problemini çözemediğinin gösterilmesi D Araştırma fonlarının kesilmesi E Transformer modellerinin eğitim maliyetinin çok yüksek olması
7. Aşağıdakilerden hangisi dar (zayıf) yapay zekâya örnektir? A Her alanda en yetenekli insanları aşan bir sistem B Yalnızca yaprak fotoğraflarından bitki hastalığı tespit eden bir model C İnsan gibi her türlü görevi öğrenip bilgisini başka alanlara aktarabilen bir sistem D Bilinç sahibi olduğu kanıtlanmış bir robot
12. Mitchell’in T–P–E tanımını, bir e-ticaret sitesinde kullanıcılara ürün öneren bir sistem için yazınız. Bu sistemin başarımını yalnızca “tıklama oranı” ile ölçmenin hangi etik ve teknik sorunlara yol açabileceğini tartışınız.
Bu haftanın sonunda öğrenciler; • bir veri setinde örnek, öznitelik ve etiket kavramlarını ayırt edebilecek, • denetimli, denetimsiz ve pekiştirmeli öğrenmeyi örneklerle açıklayabilecek; bir problemi doğru öğrenme türüyle eşleştirebilecek, • sınıflandırma ile regresyon problemlerini ayırt edebilecek, • makine öğrenmesi iş akışının adımlarını sıralayabilecek, • veriyi eğitim, doğrulama ve test kümelerine ayırmanın amacını ve veri sızıntısı riskini açık-
• aşırı ve eksik öğrenmeyi tanımlayabilecek, • min-maks normalizasyonu, z-skoru standardizasyonu ve one-hot kodlamayı elle uygula-
• Pearson korelasyon katsayısını hesaplayıp yorumlayabilecek; dengesiz veri setleri için yeniden örnekleme ve sınıf ağırlığı yaklaşımlarını açıklayabilecektir.
Makine öğrenmesi modelleri kurallarını veriden çıkardığı için verinin yapısını doğru anlamak her şeyin başlangıcıdır. Tablo biçimindeki bir veri setinde (dataset) her satır bir örnek (sample, instance) (gözlem), her sütun bir öznitelik ve tahmin etmek istediğimiz sütun ise etiket (label) ya da hedef
Belirleyici olan girdinin değil, çıktının (etiketin) türüdür. “Bir hastanın tümörü iyi huylu mu, kötü huylu mu?” sorusu sınıflandırma; “tümörün çapı kaç milimetre?” sorusu ise regresyondur. Çıktı sayı olsa bile sınıf belirtiyorsa (ör. rakam tanımada “7”) problem sınıflandırmadır.
Denetimsiz öğrenmede (unsupervised learning) verilerin etiketi yoktur. Model, verinin içindeki yapıyı, grupları veya örüntüleri kendi başına keşfetmeye çalışır.
En yaygın denetimsiz görevler şunlardır: benzer örnekleri gruplayan kümeleme (clustering) (ör. müşteri segmentasyonu), çok boyutlu veriyi daha az boyutla temsil eden boyut indirgeme (dimensionality reduction) ve olağan dışı örnekleri bulan anomali tespiti (anomaly detection). Şekil 2.2 denetimli ve denetimsiz öğrenmeyi karşılaştırmaktadır.
Pekiştirmeli öğrenmede (reinforcement learning) bir ajan, çevresiyle deneme–yanılma yoluyla etkileşime girer. Her eyleminin ardından bir ödül (reward) (ya da ceza) alır ve uzun vadede toplam ödülü en üst düzeye çıkaran davranış stratejisini (policy) öğrenir.
Pekiştirmeli öğrenmede doğru cevap önceden verilmez; ajan yalnızca yaptığı şeyin ne kadar iyi olduğunu ödül sinyalinden anlar (Şekil 2.3). Oyun oynayan programlar (AlphaGo), robotların yürümeyi öğrenmesi ve büyük dil modellerinin insan tercihleriyle hizalanması (RLHF, 14. hafta) bu
Etiketleme çoğu zaman pahalıdır: Bir radyoloğun binlerce MR görüntüsünü tek tek işaretlemesi haftalar sürer. Yarı denetimli öğrenme (semi-supervised learning), az sayıda etiketli örneği çok sayıda etiketsiz örnekle birlikte kullanır. Öz-denetimli öğrenmede (self-supervised learning) ise etiket verinin kendisinden üretilir. Örneğin bir cümlenin sonraki kelimesini tahmin etmek için etikete gerek yoktur, çünkü sonraki kelime metnin içinde zaten vardır. Büyük dil modellerinin ön eğitimi tam
Gerçek bir makine öğrenmesi projesi yalnızca “model eğitmekten” ibaret değildir. Şekil 2.4’deki adımlar bir yazılım projesinin yaşam döngüsüne benzer ve çoğu zaman tekrarlıdır: Değerlendirme sonuçları beklentiyi karşılamazsa veri toplama veya öznitelik adımlarına geri dönülür.
2. Veri toplama: Yeterli miktarda, problemi temsil eden ve mümkünse etiketli veri toplanır. 3. Veri ön işleme: Eksik ve hatalı değerler düzeltilir, veri ölçeklenir ve kodlanır (Bölüm 2.6). 4. Öznitelik mühendisliği: Modelin işini kolaylaştıracak yeni öznitelikler türetilir (ör. doğum ta-
5. Model eğitimi: Uygun algoritma seçilir ve eğitim verisiyle eğitilir. 6. Değerlendirme: Model, eğitimde hiç görmediği test verisi üzerinde uygun metriklerle ölçülür
7. Dağıtım ve izleme: Model bir uygulamaya entegre edilir; zamanla verinin değişmesi (data drift) nedeniyle başarımı izlenir ve gerektiğinde yeniden eğitilir. Bu adımların yazılım mühendisliği disipliniyle yönetilmesine MLOps (machine learning operations) denir.
Sektörde yapılan anketlerde veri bilimcilerin zamanlarının önemli bir bölümünü model geliştirmeye değil, veriyi toplamaya, temizlemeye ve düzenlemeye harcadıkları sıkça dile getirilir. Gerçek projelerde en çok emek isteyen adım genellikle veridir.
Bir modelin gerçek başarısı, eğitim sırasında hiç görmediği veriler üzerindeki başarısıdır. Bu yeteneğe genelleme (generalization) denir. Genellemeyi ölçebilmek için veri seti en baştan üç parçaya ayrılır
• Eğitim kümesi (training set): Modelin parametrelerini öğrenmek için kullanılır. • Doğrulama kümesi (validation set): Model seçimi ve hiperparametre ayarı (ör. polinom derecesi, öğrenme oranı) için kullanılır. • Test kümesi (test set): Yalnızca en sonda, nihai başarımı raporlamak için bir kez kullanılır.
mede kaç görüntü bulunur? (b) Bölme katmanlı (sınıf oranları korunarak) yapılırsa test kümesinde kaç zatürre görüntüsü olur?
Test verisine ait herhangi bir bilginin eğitim sürecine karışmasına veri sızıntısı (data leakage) denir. Sızıntı, modelin başarısını olduğundan yüksek gösterir ve model gerçek hayatta beklenenden kötü çalışır. Sık görülen örnekler:
• Aynı hastaya ait farklı görüntülerin hem eğitim hem test kümesine düşmesi, • Test kümesine bakarak model veya hiperparametre seçmek.
Bir model eğitim verisindeki gürültüyü ve rastlantısal ayrıntıları da ezberlerse eğitim verisinde çok başarılı, yeni verilerde ise başarısız olur. Bu duruma aşırı öğrenme (overfitting) denir. Tersine, model verideki ilişkiyi yakalayamayacak kadar basitse hem eğitim hem test verisinde başarısız olur; buna
Sınava yalnızca geçmiş yılların sorularını cevaplarıyla ezberleyerek hazırlanan bir öğrenci, aynı sorular gelirse tam puan alır; soruların sayıları biraz değiştirildiğinde ise çözemez. Bu öğrenci aşırı öğrenmiştir. Konuyu hiç çalışmayan öğrenci ise her durumda başarısızdır (eksik öğrenme). İyi bir model, konuyu anlayan öğrenci gibi yeni sorularda da başarılı olandır.
Aşırı öğrenmeyle mücadele yöntemlerini (düzenlileştirme, çapraz doğrulama) 5. haftada, derin ağlara özgü yöntemleri (dropout, veri artırma) 7. ve 10. haftalarda ayrıntılı olarak inceleyeceğiz.
Gerçek veri setlerinde bazı hücreler boş olabilir (anket sorusunu cevaplamayan kişi, arızalanan sensör). Başlıca çözümler: (i) eksik değer içeren satırları silmek (veri azsa kayıp büyüktür), (ii) sayısal sütunlarda eksik değeri sütunun ortalaması veya medyanı ile doldurmak, (iii) kategorik sütunlarda en sık görülen değerle (mod) doldurmak. Aykırı değerlerin bulunduğu sütunlarda medyan, ortalamaya göre daha güvenilirdir.
arasında değişir. Uzaklık hesabına dayanan yöntemlerde (k-NN, k-means) ve gradyan inişle eğitilen modellerde büyük ölçekli öznitelik baskın hâle gelir. Bu nedenle öznitelikler benzer aralıklara
yonu ve (b) z-skoru standardizasyonu uygulayınız. (c) Test kümesinden gelen 50 yaşındaki bir kişinin her iki yöntemle dönüştürülmüş değerini bulunuz.
Modeller sayılarla çalıştığı için “Iğdır”, “Kars”, “Van” gibi kategorik değerlerin sayıya dönüştürülmesi gerekir. İlk akla gelen yöntem her kategoriye bir tam sayı vermektir (etiket kodlama (label
veriler için doğru yöntem one-hot kodlamadır (one-hot encoding): Her kategori için ayrı bir sütun açılır ve yalnızca ilgili sütun 1, diğerleri 0 olur.
Aşağıdaki tabloda bir değer eksiktir. (a) Eksik “Yaş” değerini ortalama ile doldurunuz. (b) “Şehir” sütununu one-hot kodlayınız.
Ön işlemenin son adımı, özniteliklerin birbiriyle ve etiketle nasıl ilişkili olduğunu incelemektir. İki sayısal değişken arasındaki doğrusal ilişkinin yönünü ve gücünü Pearson korelasyon katsayısı (Pe-
gösterir. Korelasyon matrisi (her öznitelik çifti için r), iki durumda yol göstericidir: Etiketle güçlü korelasyonu olan öznitelikler iyi birer aday tahmin edicidir; birbiriyle çok yüksek korelasyonlu iki öznitelik (ör. metre ve santimetre cinsinden boy) ise aynı bilgiyi taşıdığından biri atılabilir.
olarak açıklanabildiğini söyler; 3. haftada aynı veriyle kuracağımız regresyon doğrusunun R2 değeri de tam olarak 0.6 çıkacaktır.
ama doğrusal olmayan bir ilişkide r sıfıra yakın çıkabilir. Bu nedenle korelasyon her zaman saçılım grafiğiyle birlikte yorumlanmalıdır.
Sınıflandırma problemlerinde sınıfların örnek sayıları çoğu zaman çok farklıdır: Kredi kartı işlemlerinin binde biri sahtedir, tıbbi taramaların küçük bir kısmı hastalık içerir. Bu tür dengesiz veri (imbalanced data) setlerinde model, çoğunluk sınıfını tahmin ederek yüksek doğruluğa ulaşabilir ve azınlık sınıfını hiç öğrenmeyebilir. Başlıca çözümler:
• Yeniden örnekleme: Çoğunluk sınıfından örnek atmak (undersampling) veya azınlık sınıfını çoğaltmak (oversampling). SMOTE yöntemi, azınlık örneklerinin arasında ara noktalar üreterek sentetik örnekler oluşturur. Yeniden örnekleme yalnızca eğitim kümesine uygulanır; test kümesi gerçek dağılımı yansıtmalıdır. • Sınıf ağırlıkları: Kayıp fonksiyonunda azınlık sınıfının hataları daha ağır cezalandırılır. Yaygın
Yüzlerce özniteliği olan bir veri setinde öznitelikler çoğu zaman birbirine bağımlıdır ve veri aslında daha düşük boyutlu bir yapıda yaşar. Temel bileşen analizi (principal component analysis, PCA), veriyi varyansın en büyük olduğu yönlere (temel bileşenlere) izdüşürür: İlk bileşen verinin en çok yayıldığı doğrultudur, ikinci bileşen ona dik ve kalan varyansı en çok açıklayan doğrultudur; bu böyle devam eder. Bileşenler, standartlaştırılmış verinin kovaryans matrisinin özvektörleridir; her bileşenin açıkladığı varyans oranı ilgili özdeğerle orantılıdır. Uygulamada ilk k bileşen alınarak (ör. toplam varyansın %95’ini açıklayan sayıda) veri d boyuttan k boyuta indirgenir. PCA; görselleştirme (2–3 boyuta indirgeme), gürültü azaltma ve uzaklık tabanlı yöntemleri hızlandırma amacıyla kullanılır. Doğrusal olmayan yapılar için t-SNE ve UMAP gibi görselleştirme yöntemleri tercih edilir.
✓ Veri setinde satırlar örnek, sütunlar öznitelik, tahmin edilen sütun etikettir. ✓ Denetimli öğrenmede etiketler bilinir; etiket kategorikse sınıflandırma, sürekliyse regresyon
✓ Denetimsiz öğrenme etiketsiz veride yapı arar (kümeleme, boyut indirgeme); pekiştirmeli öğrenmede ajan ödül sinyaliyle öğrenir.
melerine ayrılır. Test bilgisinin eğitime karışması veri sızıntısıdır. ✓ Aşırı öğrenmede eğitim hatası çok düşük, test hatası yüksektir; eksik öğrenmede ikisi de
eğitim kümesinden hesaplanır. ✓ Nominal kategorik değişkenler one-hot kodlanır; etiket kodlama sahte bir sıra ilişkisi yara-
✓ Dengesiz veride yeniden örnekleme (yalnızca eğitim kümesinde), sınıf ağırlıkları ve doğruluk dışı metrikler kullanılır. PCA, veriyi en çok varyans taşıyan yönlere izdüşürerek boyut
1. Bir bankanın müşterilerini, herhangi bir etiket kullanmadan harcama alışkanlıklarına göre gruplara ayırması hangi tür bir problemdir?
2. Aşağıdakilerden hangisi bir regresyon problemidir? A Bir tweet’in olumlu ya da olumsuz olduğunu belirlemek B Bir yaprak fotoğrafındaki hastalık türünü belirlemek
6. Aşağıdakilerden hangisi bir veri sızıntısı örneğidir? A Veriyi eğitim ve test kümelerine katmanlı olarak bölmek B Standardizasyon için ortalama ve standart sapmayı, veriyi bölmeden önce tüm veri
C Hiperparametreleri doğrulama kümesine göre seçmek D Eksik değerleri eğitim kümesinin medyanıyla doldurmak E Test kümesini yalnızca en sonda bir kez kullanmak
A Eksik öğrenme; modeli daha karmaşık hâle getirmek B Aşırı öğrenme; daha fazla veri toplamak veya modeli basitleştirmek C Eksik öğrenme; eğitimi erken durdurmak D Aşırı öğrenme; test kümesini eğitime eklemek E Model idealdir; değişiklik gerekmez
11. Eğitim, doğrulama ve test kümelerinin her birinin görevini birer cümleyle açıklayınız. Test kümesi neden model seçiminde kullanılmamalıdır?
ruluğu nedir ve bu model neden işe yaramaz? (b) Çoğunluk sınıfı, sahte işlemlerin dört katı olacak şekilde alt örneklenirse eğitim kümesi kaç örnek içerir? (c) Yeniden örnekleme neden test kümesine uygulanmaz?
Bu haftanın sonunda öğrenciler; • basit ve çoklu doğrusal regresyon modelini yazabilecek ve parametrelerini yorumlayabile-
• artık (hata) ve kayıp fonksiyonu kavramlarını açıklayabilecek, • en küçük kareler yöntemiyle regresyon katsayılarını elle hesaplayabilecek, • MSE, RMSE, MAE ve R2 metriklerini hesaplayıp yorumlayabilecek, • gradyan iniş algoritmasının adımlarını uygulayabilecek ve öğrenme oranının etkisini açık-
• en küçük kareler çözümünü normal denklemle matris formunda bulabilecek ve ölçeklemenin gradyan inişin yakınsamasına etkisini açıklayabilecektir.
Regresyon, sürekli bir hedef değişkeni tahmin etmeyi amaçlayan denetimli öğrenme görevidir. Bu ailenin en basit ve en çok kullanılan üyesi doğrusal regresyondur (linear regression). Tek öznitelikli durumda model bir doğrudur:
linde modellenebilir. Bu modelde bina yaşının katsayısının negatif olması, diğer her şey sabitken eski binaların daha ucuz olduğunu söyler. Katsayıların bu şekilde doğrudan yorumlanabilmesi, doğrusal regresyonun en büyük avantajlarından biridir.
bir parabol çizse de w1 , w2 , b parametrelerine göre doğrusal olduğu için yine doğrusal regresyonla çözülür. Buna polinom regresyon (polynomial regression) denir; 2. haftadaki Şekil 2.6 bu
Artıkları tek bir sayıya indirgeyen fonksiyona kayıp fonksiyonu (loss function) denir. Regresyonda en yaygın kayıp, ortalama kare hatadır (mean squared error, MSE):
Ayrıca kare fonksiyonunun türevi kolay alındığından optimizasyon kolaylaşır. Alternatif olarak hataların mutlak değerinin ortalaması (MAE) da kullanılabilir (Şekil 3.2).
Basit doğrusal regresyonda MSE’yi en küçük yapan w ve b değerleri kapalı formda (doğrudan formülle) bulunabilir. Bu yönteme en küçük kareler (least squares) yöntemi denir.
Formül sezgisel olarak şunu söyler: x ortalamanın üzerindeyken y de ortalamanın üzerinde olma eğilimindeyse pay pozitif çıkar ve eğim pozitif olur. b’nin formülü ise en iyi doğrunun her zaman
kendisidir. MSE, (w, b)’ye göre dışbükey bir paraboloit olduğundan bu tek durağan nokta aynı zamanda küresel en küçük noktadır.
sayısı on binleri bulduğunda ya da iki öznitelik birbirinin katı olduğunda (matris tekil olur) gradyan iniş tercih edilir. Ayrıca bu kapalı çözüm yalnızca MSE kaybına ve doğrusal modele özgüdür; sinir ağlarında böyle bir formül yoktur ve tek seçenek gradyan iniştir.
R2 formülündeki pay hata kareler toplamı (sum of squared errors, SSE), payda ise toplam kareler toplamıdır (total sum of squares, SST). SST, hiçbir öznitelik kullanmadan her örnek için ȳ tahmin eden en basit modelin hatasıdır. Dolayısıyla R2 , modelimizin bu “tembel” modele göre ne kadar iyi oldu-
Yorum: Model, proje puanlarındaki değişkenliğin %60’ını açıklamaktadır. Tahminler gerçek puanlardan tipik olarak yaklaşık 0.7 puan sapmaktadır.
ortalamayı tahmin etmekten bile kötüyse (SSE > SST) R2 negatif çıkar. Negatif R2 , modelin işe yaramadığının güçlü bir işaretidir. Ayrıca yüksek R2 tek başına modelin doğru olduğunu kanıtlamaz; artıkların grafiğine de bakılmalıdır.
En küçük kareler formülü yalnızca doğrusal regresyon gibi özel modeller için vardır. Lojistik regresyonda ve sinir ağlarında kaybı en küçük yapan parametreleri veren kapalı bir formül yoktur. Bu durumda kullanılan genel amaçlı yöntem gradyan iniştir (gradient descent). Derin öğrenmedeki milyarlarca parametreli modeller de gradyan inişin türevleriyle eğitilir; bu nedenle bu bölüm dersin geri kalanı için kritik öneme sahiptir.
Yoğun siste bir dağın tepesinde olduğunuzu ve vadiye inmek istediğinizi düşünün. Etrafı göremiyorsunuz; ancak ayağınızın altındaki zeminin hangi yöne doğru eğimli olduğunu hissedebiliyorsunuz. Mantıklı strateji, en dik iniş yönünde küçük bir adım atmak ve bunu tekrarlamaktır. Gradyan iniş tam olarak budur: Dağ kayıp fonksiyonu, konumunuz parametreler, eğim gradyan, adım uzunluğunuz ise öğrenme oranıdır.
Burada η (eta) öğrenme oranıdır (learning rate). Adımlar, parametreler artık anlamlı ölçüde değişmeyene (yakınsama) ya da belirlenen iterasyon sayısına ulaşılana kadar tekrarlanır.
tif olduğu için (fonksiyon bu bölgede azalan) w artırılmıştır. En küçük noktaya yaklaştıkça gradyan küçüldüğünden adımlar da kendiliğinden küçülür.
Öğrenme oranı, eğitimin başarısını belirleyen en önemli hiperparametredir (hyperparameter). Hiperparametre, modelin veriden öğrendiği bir değer değil, eğitimden önce bizim belirlediğimiz bir ayardır. Şekil 3.3 aynı fonksiyonda üç farklı öğrenme oranının davranışını göstermektedir.
öğrenme oranıyla ilerleme çok yavaştır. (b) Uygun öğrenme oranıyla birkaç adımda en küçük noktaya ulaşılır. (c) Çok büyük öğrenme oranıyla adımlar en küçük noktanın üzerinden atlar ve
• η çok küçükse: Algoritma doğru yönde ilerler ama yakınsamak için çok fazla iterasyon gerekir. • η uygunsa: Kayıp hızla ve kararlı biçimde azalır. • η çok büyükse: Algoritma vadinin bir yamacından diğerine sıçrar; salınım yapar ya da ıraksar
Öznitelikler farklı ölçeklerdeyse kayıp yüzeyi bir yönde dik, diğer yönde düz uzamış bir vadiye dönüşür. Gradyan dik yönde büyük olduğu için adımlar vadinin duvarları arasında zikzak çizer; öğrenme oranı dik yöne göre küçük seçilmek zorunda kalır ve düz yönde ilerleme çok yavaşlar.
1/a gerekir. Öznitelik 10 kat büyütülürse a 100 kat büyür ve izin verilen en büyük öğrenme oranı 100 kat küçülür. Z-skoru standardizasyonu (2. hafta) tüm yönlerdeki eğrilikleri birbirine yaklaştırır; böylece tek bir öğrenme oranı her yönde iyi çalışır ve yakınsama hızlanır. Bu nedenle gradyan inişle eğitilen her modelde öznitelik ölçekleme zorunlu bir ön adımdır.
• Yığın (batch) gradyan iniş: Her adımda tüm eğitim verisi kullanılır. Kararlıdır ancak büyük veride her adım çok pahalıdır. • Stokastik gradyan iniş (stochastic gradient descent, SGD): Her adımda rastgele seçilen tek örnek kullanılır. Adımlar ucuz ama gürültülüdür.
Gradyan iniş fikri yeni değildir: Fransız matematikçi Augustin-Louis Cauchy 1847’de denklem sistemlerini çözmek için bu yöntemi önermiştir. Bugün ChatGPT gibi modelleri eğitirken kullanılan Adam optimizasyon algoritması da özünde Cauchy’nin fikrinin akıllıca geliştirilmiş bir
A Model tüm örnekleri hatasız tahmin etmektedir. B Model, her örnek için hedefin ortalamasını tahmin etmekten daha iyi değildir. C Modelin MSE değeri sıfırdır. D Model aşırı öğrenmiştir. E Öznitelikler ile hedef arasında mükemmel doğrusal ilişki vardır.
4. Gradyan iniş algoritmasında parametreler hangi yönde güncellenir? A Gradyan ile aynı yönde B Gradyanın tersi yönünde
10. Öğrenme oranının çok küçük ve çok büyük seçilmesinin sonuçlarını birer cümleyle açıklayınız. Öğrenme oranı neden “parametre” değil “hiperparametre” olarak adlandırılır?
Bu haftanın sonunda öğrenciler; • ikili, çok sınıflı ve çok etiketli sınıflandırma problemlerini ayırt edebilecek, • lojistik regresyonda sigmoid fonksiyonuyla olasılık hesaplayıp karar verebilecek ve ikili çapraz entropi kaybını hesaplayabilecek, • k-en yakın komşu algoritmasını uzaklık hesabıyla elle uygulayabilecek, • karar ağaçlarında entropi, Gini ve bilgi kazancı hesaplayarak bölme özniteliğini seçebile-
• Naive Bayes sınıflandırıcısıyla sonsal olasılık hesaplayabilecek, • destek vektör makinelerinin maksimum marj fikrini açıklayabilecek ve algoritmaları güçlü–zayıf yönleriyle karşılaştırabilecek, • çapraz entropi kaybının gradyanını türetebilecek, sürekli bir öznitelik için Gini ile en iyi eşiği bulabilecek ve SVM marjını hesaplayabilecektir.
Sınıflandırmada amaç, bir örneği önceden tanımlı sınıflardan birine (ya da birkaçına) atamaktır. Model, öznitelik uzayını sınıflara karşılık gelen bölgelere ayırır; bu bölgeleri ayıran yüzeye karar sınırı (decision boundary) denir. Sınıflandırma problemleri üç türe ayrılır:
• İkili sınıflandırma (binary classification): İki sınıf vardır. Spam/normal, hasta/sağlıklı, geçti/kaldı. Sınıflar genellikle 1 (pozitif) ve 0 (negatif) olarak kodlanır. • Çok sınıflı sınıflandırma (multi-class classification): İkiden fazla sınıf vardır ve her örnek tam olarak bir sınıfa aittir. El yazısı rakam tanıma (10 sınıf), bitki hastalığı türü. • Çok etiketli sınıflandırma (multi-label classification): Bir örnek aynı anda birden fazla etikete sahip olabilir. Bir fotoğrafta hem “kedi” hem “kanepe” bulunabilir; bir film hem “komedi” hem
Bu hafta dört temel yaklaşımı inceleyeceğiz: olasılık üreten doğrusal bir model (lojistik regresyon), benzerliğe dayanan bir yöntem (k-NN), kurallar üreten bir yöntem (karar ağacı) ve olasılık ku-
lenmiştir. (a) Haftada 3 saat ve 5 saat çalışan öğrencilerin geçme olasılıklarını hesaplayınız ve 0.5 eşiğine göre sınıflandırınız. (b) Karar sınırını bulunuz.
Lojistik regresyonda MSE yerine ikili çapraz entropi (binary cross-entropy) kaybı kullanılır. Bu kayıp, doğru sınıfa verilen olasılığın negatif logaritmasıdır:
Yani güncelleme miktarı, tahmin edilen olasılık ile gerçek etiket arasındaki farkla orantılıdır. Model gradyan inişle (3. hafta) eğitilir. İkiden fazla sınıf olduğunda sigmoid yerine softmax fonksiyonu kullanılır; bunu 7. haftada göreceğiz.
Sigmoid çıktısı ile MSE birlikte kullanıldığında kayıp yüzeyi dışbükey (convex) olmaz ve birçok yerel minimum oluşabilir; ayrıca model çok yanlış olduğunda gradyanlar çok küçülür ve öğrenme yavaşlar. Çapraz entropi bu iki sorunu da ortadan kaldırır.
k-en yakın komşu (k-nearest neighbors, k-NN) algoritması, “bana arkadaşını söyle, sana kim olduğunu söyleyeyim” ilkesine dayanır. Yeni bir örneği sınıflandırmak için eğitim kümesinde ona en yakın k örnek bulunur ve bu komşular arasında en çok görülen sınıf (çoğunluk oylaması) tahmin edilir. Regresyonda ise komşuların hedef değerlerinin ortalaması alınır.
1. Yeni örnek q ile eğitim kümesindeki her örnek arasındaki uzaklığı hesapla. 2. Uzaklıkları küçükten büyüğe sırala ve en yakın k örneği seç. 3. Bu k komşunun sınıflarına bak; en sık görülen sınıfı tahmin olarak döndür.
karar sınırı girintili çıkıntılı olur ve aşırı öğrenme görülür. Büyük k ise karar sınırını aşırı yumuşatır ve küçük sınıfları “yutar” (eksik öğrenme). Uygun k değeri doğrulama kümesi veya çapraz doğrulama (5. hafta) ile seçilir. İki sınıflı problemlerde oyların eşit çıkmaması için k genellikle tek sayı
• Tahmin yavaştır: k-NN eğitim sırasında hiçbir şey öğrenmez, yalnızca veriyi saklar (tembel öğrenme (lazy learning)). Her tahminde tüm eğitim örneklerine uzaklık hesaplan-
dığından büyük veride yavaştır. • Boyut laneti: Öznitelik sayısı çok arttığında tüm noktalar birbirine benzer uzaklıklarda kalır ve “en yakın komşu” kavramı anlamını yitirir (boyutluluğun laneti (curse of dimensio-
Karar ağacı (decision tree), veriyi “evet/hayır” sorularıyla art arda bölerek sınıflandıran, akış şemasına benzeyen bir modeldir. Ağacın en üstündeki düğüme kök, iç düğümlere karar düğümü, en alttaki düğümlere ise yaprak denir. Her yaprak bir sınıf tahmini içerir. Karar ağaçlarının en büyük avantajı yorumlanabilir olmalarıdır: Bir tahminin neden yapıldığı kökten yaprağa giden yol izlenerek açıkla-
Ağaç kurulurken her düğümde “hangi öznitelikle bölersem ortaya çıkan gruplar en saf olur?” sorusu sorulur. Saf bir düğümde örneklerin hepsi aynı sınıftandır. Saflığı ölçmek için iki ölçü kulla-
Bir A özniteliğine göre bölmenin bilgi kazancı (information gain), ebeveyn düğümün entropisinden çocuk düğümlerin ağırlıklı ortalama entropisinin çıkarılmasıyla bulunur:
On öğrencinin devam durumu, ödev teslim durumu ve dersi geçip geçmediği aşağıda verilmiştir. Kök düğümde hangi öznitelikle bölme yapılmalıdır?
Sonuç: IG(Devam) > IG(Ödev) olduğundan kök düğümde Devam özniteliği kullanılır. Aynı işlem alt düğümlerde tekrarlanarak Şekil 4.4’deki ağaç elde edilir. (Gini ile de aynı sonuç çıkar:
Yaş, gelir gibi sürekli öznitelikler “x < t” biçiminde ikili sorulara dönüştürülür. Aday eşikler, sıralanmış değerlerin ardışık çiftlerinin orta noktalarıdır (yalnızca etiketin değiştiği yerlere bakmak yeterlidir); her aday için safsızlık hesaplanır ve en iyi eşik seçilir. Bu işlem her düğümde, her öznite-
lik için tekrarlanır; CART algoritması (scikit-learn’ün kullandığı) safsızlık ölçüsü olarak varsayılan biçimde Gini’yi kullanır.
Sınırlandırılmayan bir karar ağacı, her yaprakta tek bir örnek kalana kadar bölmeye devam ederek eğitim verisini ezberleyebilir. Bunu önlemek için en fazla derinlik (max_depth), bir yapraktaki en az örnek sayısı gibi sınırlar konur ya da ağaç sonradan budanır (pruning). Tek bir ağacın kararsızlığını gidermenin güçlü bir yolu, çok sayıda ağacı bir araya getirmektir. Rastgele orman (random forest), her biri verinin rastgele bir alt örneklemiyle ve özniteliklerin rastgele bir alt kümesiyle eğitilmiş yüzlerce ağacın oylamasıyla karar verir. Birden çok modeli birleştiren bu tür yöntemlere topluluk öğrenmesi (ensemble learning) denir. Rastgele ormanlar ve gradyan artırma (XGBoost, LightGBM) yöntemleri, tablo verilerinde bugün hâlâ en başarılı yöntemler arasındadır.
İki sınıfı ayıran sonsuz sayıda doğru çizilebilir. Destek vektör makineleri (support vector machines, SVM), bu doğrular arasından iki sınıfa da en uzak olanı, yani sınıflar arasındaki boşluğu (marj (margin)) en büyük yapanı seçer (Şekil 4.6). Karar sınırına en yakın olan ve sınırın konumunu belirleyen örneklere destek vektörleri (support vectors) denir; diğer örnekler silinse bile karar sınırı değişmez. Geniş marj, yeni örneklerde daha iyi genelleme sağlar.
ile az hata arasındaki dengeyi ayarlar: Büyük C hataları ağır cezalandırır (dar marj, aşırı öğrenme riski), küçük C marjı genişletir. Veri doğrusal olarak ayrılamıyorsa SVM, çekirdek hilesi (kernel trick) ile veriyi örtük olarak daha yüksek boyutlu bir uzaya taşır ve orada doğrusal bir sınır arar. Bu sınır, orijinal uzayda eğri bir sınıra karşılık gelir. En yaygın çekirdek RBF (Gauss) çekirdeğidir. SVM’ler derin öğrenme öncesi dönemde (1995–2012) en başarılı sınıflandırıcılardandı ve küçük–orta boyutlu veri setlerinde hâlâ
✓ Sınıflandırma ikili, çok sınıflı veya çok etiketli olabilir; model öznitelik uzayını karar sınırlarıyla bölgelere ayırır.
✓ k-NN, en yakın k komşunun çoğunluk oyuyla karar verir; ölçekleme şarttır, küçük k aşırı öğrenmeye, büyük k eksik öğrenmeye yol açar.
4. k-NN algoritması için aşağıdaki ifadelerden hangisi yanlıştır? A Uzaklık hesabına dayandığı için özniteliklerin ölçeklenmesi önemlidir. B Eğitim aşamasında parametre öğrenmez, veriyi saklar. C k değeri büyüdükçe karar sınırı daha düzgün (yumuşak) hâle gelir. D Tahmin aşaması, eğitim kümesi büyüdükçe yavaşlar.
6. Karar ağacı kurulurken bir düğümde bölme yapılacak öznitelik hangi ölçüte göre seçilir? A En düşük bilgi kazancı B En yüksek bilgi kazancı (veya en düşük ağırlıklı safsızlık) C En fazla kategorisi olan öznitelik
8. Naive Bayes sınıflandırıcısının “naif” olarak adlandırılmasının nedeni nedir? A Yalnızca iki sınıfla çalışabilmesi B Sınıf bilindiğinde özniteliklerin birbirinden bağımsız olduğunu varsayması C Önsel olasılıkları kullanmaması D Gradyan inişle eğitilmesi E Karar sınırının her zaman doğrusal olması
10. Bir düğümde 9 “Evet” ve 5 “Hayır” örneği vardır. Bir öznitelikle bölündüğünde birinci çocukta 6 Evet–2 Hayır, ikinci çocukta 3 Evet–3 Hayır oluşmaktadır. (a) Ebeveyn düğümün entropisini, (b) bu bölmenin bilgi kazancını hesaplayınız.
şilerin ise %5’inde (yanlışlıkla) pozitif sonuç vermektedir. Testi pozitif çıkan birinin gerçekten hasta olma olasılığını Bayes teoremiyle hesaplayınız ve sonucu yorumlayınız.
yazınız, (b) her aday için ağırlıklı Gini safsızlığını hesaplayıp en iyi eşiği seçiniz, (c) tek bir eşikle tüm örneklerin neden doğru sınıflandırılamayacağını açıklayınız.
Bu haftanın sonunda öğrenciler; • karışıklık matrisini oluşturup doğruluk, kesinlik, duyarlılık, özgüllük ve F1 skorunu he-
• dengesiz veri setlerinde doğruluğun neden yanıltıcı olduğunu açıklayabilecek, • ROC eğrisi ve AUC değerini yorumlayabilecek, karar eşiğinin etkisini tartışabilecek, • k-katlı çapraz doğrulamayı uygulayabilecek, • yanlılık–varyans ödünleşimini ve L1/L2 düzenlileştirmenin amacını açıklayabilecek, • k-ortalamalar (k-means) algoritmasını elle uygulayabilecek ve küme sayısını dirsek yönte-
• kesinlik–duyarlılık eğrisinden ortalama kesinliği (AP) hesaplayabilecek, hiperparametre aramasını çapraz doğrulamayla planlayabilecek ve siluet skorunu yorumlayabilecektir.
Bir sınıflandırıcının başarısını ölçmenin akla gelen ilk yolu, doğru tahminlerin oranı olan doğruluktur (accuracy). Ancak doğruluk tek başına yanıltıcı olabilir. Bin kişide 50’sinin hasta olduğu bir tarama programı düşünün. Hiçbir şey öğrenmeden herkese “sağlıklı” diyen bir model %95 doğruluk elde eder; ama tek bir hastayı bile yakalayamaz. Sınıfların sayıca çok farklı olduğu bu tür veri setlerine dengesiz veri seti (imbalanced dataset) denir ve gerçek hayattaki problemlerin çoğu (hastalık teşhisi, dolandırıcılık, arıza tespiti) bu türdendir. Bu nedenle sınıflandırıcıları farklı açılardan ölçen metriklere ihtiyaç vardır.
İkili sınıflandırmada tahminler, gerçek sınıfla karşılaştırıldığında dört gruba ayrılır. Bu dört sayıyı gösteren tabloya karışıklık matrisi (confusion matrix) denir (Şekil 5.1).
Kesinlik (precision) ve duyarlılık (recall, sensitivity) birbirini tamamlayan iki sorunun cevabıdır. Duyarlılık tıpta “hassasiyet” (sensitivity) veya “doğru pozitif oranı” (TPR) olarak da bilinir. F1 skoru (F1 score), iki metriği tek sayıda birleştirir. Harmonik ortalama kullanıldığı için metriklerden biri çok düşükse F1 de düşük çıkar; böylece “yalnızca birinde iyi” olan model ödüllendirilmez.
doğru tespit etmiş, ayrıca 30 sağlıklı kişiye yanlışlıkla “hasta” demiştir. (a) Karışıklık matrisini oluşturunuz. (b) Doğruluk, kesinlik, duyarlılık, özgüllük ve F1 değerlerini hesaplayınız. (c) Herkese “sağlıklı” diyen modelin doğruluğu ve duyarlılığı nedir?
• Duyarlılık önceliklidir (FN pahalıdır): kanser taraması, hırsızlık alarmı, salgın tespiti. Bir hastayı kaçırmak, birkaç yanlış alarmdan çok daha maliyetlidir. • Kesinlik önceliklidir (FP pahalıdır): spam filtresi (önemli bir e-postayı spam klasörüne atmak), bir müşteriyi haksız yere dolandırıcı ilan etmek, öneri sistemleri. • Sınıflar dengesizse doğruluk yerine F1, kesinlik–duyarlılık ve AUC birlikte raporlanmalı-
minleri gösterir ve köşegen doğru tahminleri içerir. Kesinlik, duyarlılık ve F1 her sınıf için “o sınıf pozitif, diğerleri negatif” kabul edilerek ayrı ayrı hesaplanır. Sonra bu değerlerin basit ortalaması alınır (makro ortalama (macro average)) ya da sınıfların örnek sayısına göre ağırlıklı ortalaması alınır (ağırlıklı ortalama). Makro ortalama, küçük sınıflara da büyük sınıflar kadar önem verdiği için dengesiz veride tercih edilir.
Lojistik regresyon gibi modeller bir olasılık üretir ve sınıf kararı bir eşikle verilir. Eşiği düşürmek modeli daha “cesur” yapar: Daha fazla örneğe “pozitif” der, duyarlılık artar ama kesinlik genellikle düşer. Eşiği yükseltmek ise tersini yapar (Şekil 5.2). Varsayılan 0.5 eşiğinin probleme göre en iyi
• AUC’nin olasılıksal anlamı: Rastgele seçilen bir pozitif örneğe, rastgele seçilen bir negatif örnekten daha yüksek skor verilme olasılığıdır.
ROC eğrisi, negatif sınıf çok büyük olduğunda iyimser görünebilir: Binlerce negatif arasında birkaç yüz yanlış pozitif, FPR’yi neredeyse hiç değiştirmez ama kesinliği çökertir. Dengesiz problemlerde bu yüzden kesinlik–duyarlılık eğrisi (precision–recall curve) tercih edilir: Eşik düşürüldükçe elde edilen (duyarlılık, kesinlik) çiftleri çizilir. Eğrinin altındaki alan ortalama kesinlik (average precision, AP) olarak özetlenir ve pratikte, her yeni doğru pozitifin yakalandığı andaki kesinliklerin ortala-
Rastgele bir modelin AP’si pozitif sınıfın oranına eşittir (%1 pozitif içeren veride 0.01); bu yüzden AP, aynı verideki AUC’den çok daha sert bir ölçüttür.
Aynı modelin AUC’si 0.8125 idi; iki metrik farklı soruları yanıtlar. Nesne tespitinde kullanılan mAP (10. hafta) de bu hesabın sınıflar üzerinden ortalamasıdır.
Veriyi bir kez eğitim ve doğrulama kümelerine ayırmak, özellikle küçük veri setlerinde şansa bağlı sonuçlar verebilir: Doğrulama kümesine kolay örnekler düşerse model olduğundan iyi görünür. k-katlı çapraz doğrulama (k-fold cross-validation) bu sorunu şöyle çözer (Şekil 5.4): Eğitim verisi k eşit parçaya (kata) bölünür. Model k kez eğitilir; her seferinde bir kat doğrulama, kalan k − 1 kat eğitim için kullanılır. Sonuçta k başarım değerinin ortalaması ve standart sapması raporlanır. Yaygın
Çapraz doğrulamanın en yaygın kullanımı hiperparametre (hyperparameter) seçimidir: k-NN’deki k, SVM’deki C, ağacın en büyük derinliği, öğrenme oranı gibi eğitimle öğrenilmeyen ayarlar. Izgara arama (grid search) her hiperparametre için bir değer listesi alır ve tüm kombinasyonları çapraz
sayısı hızla patladığından rastgele arama, belirli bir bütçe kadar rastgele kombinasyon dener ve aynı bütçede genellikle ızgaradan daha iyi sonuç verir; çünkü önemsiz hiperparametrelere eşit zaman harcamaz. Önceki denemelerden öğrenerek umut vadeden bölgelere yönelen Bayesçi optimizasyon araçları (Optuna gibi) da yaygındır.
Yüzlerce kombinasyon denendiğinde, seçilen en iyi ayar kısmen doğrulama kümesinin şansını da yansıtır; raporlanan doğrulama skoru iyimserdir. Bu yüzden nihai başarım, hiperparametre seçiminde hiç kullanılmamış test kümesinde ölçülür. Küçük veride iç içe çapraz doğrulama (dış döngü başarımı, iç döngü hiperparametreleri belirler) kullanılır.
Bir modelin test verisindeki hatası iki ana kaynaktan gelir. Yanlılık (bias), modelin gerçek ilişkiyi yakalayamayacak kadar basit olmasından kaynaklanan sistematik hatadır (eksik öğrenme). Varyans (variance) ise modelin eğitim verisindeki küçük değişikliklere aşırı duyarlı olmasından kaynak-
lanan hatadır (aşırı öğrenme). Model karmaşıklaştıkça yanlılık azalır, varyans artar. Amaç, toplam hatayı en küçük yapan dengeyi bulmaktır (Şekil 5.5).
model karmaşıklığı (polinom derecesi) Şekil 5.5Model karmaşıklığı arttıkça eğitim hatası sürekli düşer; doğrulama hatası ise önce düşer, sonra yükselir. En iyi model, doğrulama hatasının en küçük olduğu noktadadır. (Eğriler 300 farklı eğitim örneklemi üzerinden ortalanmıştır.)
Yüksek yanlılıklı bir okçunun okları hep hedefin aynı yanına, merkezden uzağa saplanır (sistematik hata). Yüksek varyanslı bir okçunun okları ise ortalamada merkeze yakın olsa da tahtanın her yanına dağılır. İyi bir model, okları merkezin etrafında sıkı bir şekilde toplayan ok-
Aşırı öğrenmeyi azaltmanın en etkili yollarından biri düzenlileştirmedir (regularization). Bu yaklaşımda kayıp fonksiyonuna büyük ağırlıkları cezalandıran bir terim eklenir:
resyona Ridge (ridge regression), L1 düzenlileştirmeli olana Lasso (lasso regression) denir. L2 tüm ağırlıkları sıfıra doğru küçültürken, L1 bazı ağırlıkları tam olarak sıfır yapar ve böylece öznitelik seçimi de gerçekleştirir. Derin öğrenmede L2 düzenlileştirme ağırlık sönümü (weight decay) olarak bilinir.
Eğitim, artık yalnızca MSE’yi değil bu toplamı küçültmeye çalışır; bu yüzden aynı uyumu daha küçük ağırlıklarla sağlayan modeller tercih edilir.
Şimdi denetimsiz öğrenmenin en bilinen algoritmasına geçelim. k-ortalamalar (k-means) algoritması, etiketsiz veriyi önceden belirlenen k sayıda kümeye ayırır. Her küme bir merkez (centroid) ile temsil edilir ve her nokta en yakın merkezin kümesine atanır.
1. k küme merkezini başlat (ör. rastgele seçilen k veri noktası). 2. Atama adımı: Her noktayı en yakın merkeze (Öklid uzaklığı) ata. 3. Güncelleme adımı: Her merkezi, kendisine atanan noktaların ortalamasına taşı. 4. Atamalar değişmeyene (merkezler sabitlenene) kadar 2. ve 3. adımları tekrarla.
Algoritma, küme içi kareler toplamını (WCSS: her noktanın kendi merkezine uzaklığının karelerinin toplamı) adım adım küçültür.
nedenle WCSS’yi en küçük yapan k aranmaz. Bunun yerine dirsek yönteminde (elbow method) WCSS, k’ye karşı çizilir ve eğrinin belirgin biçimde kırıldığı, sonrasında azalmanın yavaşladığı nokta seçilir (Şekil 5.8). Siluet skoru gibi başka ölçütler de kullanılabilir.
k-means, kümelerin yaklaşık küresel ve benzer büyüklükte olduğunu varsayar; uzun, iç içe veya farklı yoğunluktaki kümeleri iyi ayıramaz. Sonuç başlangıç merkezlerine bağlıdır; bu yüz-
den algoritma farklı başlangıçlarla birkaç kez çalıştırılır (n_init) ve en düşük WCSS’yi veren sonuç seçilir. Ölçekleme de k-NN’de olduğu gibi şarttır.
Kümeleme sonucunun kalitesini etiketsiz ölçmenin yaygın yolu siluet skorudur (silhouette score). Bir i örneği için ai kendi kümesindeki diğer örneklere ortalama uzaklık, bi ise en yakın başka kümenin örneklerine ortalama uzaklık olsun:
Tüm örneklerin ortalama siluet skoru, farklı k değerlerini karşılaştırmak için dirsek yöntemine iyi bir alternatiftir: Skoru en büyük yapan k seçilir. k-means dışında iki yöntem daha bilinmelidir. DBSCAN, yoğunluğa dayalıdır: Yarıçapı ε olan komşuluğunda en az minPts örnek bulunan “çekirdek” noktalar birbirine zincirlenerek kümeleri oluşturur. Küme sayısı önceden verilmez, keyfi biçimli kümeler bulunabilir ve seyrek bölgelerdeki noktalar gürültü olarak ayrılır. Hiyerarşik (birleştirici) kümeleme her örneği ayrı bir küme olarak başlatır ve en yakın iki kümeyi art arda birleştirir; sonuç, istenen seviyede kesilerek küme sayısı belirlenen ağaç biçimli bir dendrogramdır. “En yakın” tanımı bağlantı ölçütüne (tek, tam, ortalama,
✓ Yanlılık basit modellerin, varyans karmaşık modellerin hatasıdır; L1/L2 düzenlileştirme büyük ağırlıkları cezalandırarak aşırı öğrenmeyi azaltır.
✓ Dengesiz veride kesinlik–duyarlılık eğrisi ve AP, ROC/AUC’den daha bilgilendiricidir; AP, her doğru pozitif yakalandığındaki kesinliklerin ortalamasıdır. ✓ Hiperparametreler ızgara/rastgele aramayla ve çapraz doğrulamayla seçilir; test kümesi
A Yanlış pozitif; kesinlik B Yanlış negatif; duyarlılık C Doğru negatif; özgüllük D Doğru pozitif; doğruluk E Yanlış pozitif; doğruluk
4. Bir modelin ROC eğrisinin altında kalan alan 0.5 olarak ölçülmüştür. Bu ne anlama gelir? A Model mükemmel ayrım yapmaktadır. B Model, sınıfları rastgele tahminden daha iyi ayıramamaktadır.
6. L1 (Lasso) düzenlileştirmeyi L2 (Ridge) düzenlileştirmeden ayıran temel özellik nedir? A L1 ağırlıkları büyütür. B L1 bazı ağırlıkları tam olarak sıfır yaparak öznitelik seçimi yapar. C L1 yalnızca sınıflandırmada kullanılır. D L1 eğitim hatasını her zaman sıfırlar. E L1’de λ hiperparametresi yoktur.
7. k-means algoritması için aşağıdakilerden hangisi doğrudur? A Etiketli veri gerektirir. B Küme sayısı k algoritma tarafından otomatik bulunur. C Her iterasyonda merkezler, kümeye atanan noktaların ortalamasına taşınır. D Sonuç başlangıç merkezlerinden hiçbir zaman etkilenmez. E WCSS’yi en küçük yapan k değeri her zaman en iyi seçimdir.
işaretlemiş, bunların 30’u gerçekten spamdir. Karışıklık matrisini oluşturup doğruluk, kesinlik, duyarlılık ve F1 değerlerini hesaplayınız.
Hatanın baskın bileşenine göre modeli iyileştirmek için ne önerirsiniz? Daha fazla veri toplamak bu durumda ne kadar yardımcı olur?
Bu haftanın sonunda öğrenciler; • biyolojik nöron ile yapay nöron arasındaki benzerliği açıklayabilecek, • perceptronun çıktısını hesaplayabilecek ve perceptron öğrenme kuralını uygulayabilecek, • tek katmanlı perceptronun XOR problemini neden çözemediğini gösterebilecek, • başlıca aktivasyon fonksiyonlarını tanıyıp doğrusal olmayan aktivasyonun gerekliliğini
• çok katmanlı bir ağda ileri yayılım hesabını matris formunda elle yapabilecek, • bir ağın parametre sayısını hesaplayabilecek ve çıkış katmanını probleme göre seçebilecek, • ileri yayılımı bir mini-yığın için matris formunda yazabilecek ve Xavier/He ağırlık başlatmasının gerekçesini açıklayabilecektir.
İnsan beyninde yaklaşık 86 milyar sinir hücresi (nöron) bulunur ve her biri binlerce başka nöronla bağlantı kurar. Bir nöron, dendritleri aracılığıyla diğer nöronlardan elektrokimyasal sinyaller alır. Bu sinyaller hücre gövdesinde birikir; toplam uyarı belirli bir eşiği aşarsa nöron “ateşlenir” ve akson boyunca bir sinyal göndererek bağlantılı olduğu nöronları uyarır. Bağlantı noktalarına sinaps denir ve öğrenme, büyük ölçüde sinapsların güçlenmesi ya da zayıflamasıyla gerçekleşir. Yapay nöron (artificial neuron), bu davranışın çok basitleştirilmiş matematiksel bir modelidir (Şekil 6.1): Girdiler xi dendritlere, ağırlıklar (weights) wi sinaps güçlerine, toplama işlemi hücre gövdesine, aktivasyon fonksiyonu (activation function) f ise ateşleme eşiğine karşılık gelir:
Burada z ağırlıklı toplam (ön aktivasyon), b bias, a ise nöronun çıktısıdır (aktivasyon). Bias terimi, nöronun ne kadar kolay “ateşleneceğini” ayarlar; geometrik olarak karar sınırının orijinden kaydı-
Yapay sinir ağları beyinden ilham almıştır, ancak beynin bir kopyası değildir. Gerçek nöronlar zamanlamaya dayalı ateşleme örüntüleri, kimyasal ileticiler ve çok daha karmaşık dinamiklerle çalışır. Modern derin öğrenme, biyolojiden çok matematik ve mühendislik ilkeleriyle
Frank Rosenblatt’ın 1958’de geliştirdiği perceptron (perceptron), aktivasyon fonksiyonu olarak basamak fonksiyonu kullanan tek bir yapay nörondur:
bu kuralın sonlu sayıda adımda tüm örnekleri doğru sınıflandıran ağırlıkları bulacağını kanıtlamıştır (perceptron yakınsama teoremi).
Perceptronun sınıfları bir doğruyla ayırabilmesi için verinin doğrusal olarak ayrılabilir (linearly separable) olması gerekir. AND ve OR böyledir; ancak girdilerden yalnızca biri 1 olduğunda 1 üreten
lılığı göstermesi, sinir ağı araştırmalarına olan ilginin uzun süre azalmasına yol açmıştır (1. hafta). Çözüm, nöronları katmanlar hâlinde art arda bağlamaktır. XOR, “OR olsun ama AND olmasın” şek-
Gizli katmanlar, veriyi problemin daha kolay çözülebildiği yeni bir temsile dönüştürür. Derin öğrenmenin gücü, bu temsillerin elle tasarlanmak yerine veriden öğrenilmesinden gelir.
Basamak fonksiyonunun türevi her yerde sıfırdır (sıfır noktasında ise tanımsızdır); bu nedenle gradyan inişle eğitim için kullanılamaz. Modern ağlarda türevlenebilir aktivasyon fonksiyonları
Aktivasyon fonksiyonu kullanmasaydık (ya da doğrusal bir fonksiyon kullansaydık) katman eklemenin hiçbir faydası olmazdı. Çünkü doğrusal dönüşümlerin bileşkesi yine doğrusal bir dönü-
Yani yüz katmanlı doğrusal bir ağ, tek katmanlı doğrusal bir modelden daha güçlü değildir ve XOR’u yine çözemez. Doğrusal olmayan aktivasyonlar, ağın eğri karar sınırları ve karmaşık ilişkiler
Nöronların katmanlar hâlinde düzenlendiği ve her katmanın bir sonrakine tam bağlı olduğu ağlara çok katmanlı perceptron (multilayer perceptron, MLP) ya da tam bağlı ileri beslemeli ağ (fully connected feedforward network) denir (Şekil 6.4). İlk katman girdi katmanı, son katman çıkış katmanı, aradakiler ise gizli katmanlardır (hidden layers). “Derin” öğrenmedeki derinlik, gizli katman sayısını ifade eder.
Girdiden başlayarak katman katman çıktıların hesaplanmasına ileri yayılım (forward propagation) denir. l. katmanın hesabı matris formunda şöyle yazılır:
Tam bağlı bir katmanda ngirdi nöronlu bir katmandan nçıktı nöronlu bir katmana geçerken her çıkış nöronunun ngirdi ağırlığı ve bir bias değeri vardır:
Bias vektörü her satıra ayrı ayrı eklenir (NumPy’daki yayınlama, broadcasting). Bu yazım, GPU’ların matris çarpımını çok hızlı yapmasının derin öğrenmede neden belirleyici olduğunu da açıklar: Döngü yerine tek bir büyük çarpım vardır.
malı normal dağılımdan çeker. Bias değerleri sıfırla başlatılır. Keras varsayılan olarak Glorot başlatmasını kullanır; PyTorch’un varsayılanı da aynı mertebede (1/ngirdi ölçeğinde) bir başlatmadır, He başlatması ise açıkça seçilir. Derin ağların 2010’lara kadar eğitilememesinin nedenlerinden biri bu basit ayarın bilinmemesiydi.
Evrensel yaklaşım teoremi (universal approximation theorem), doğrusal olmayan aktivasyonlu ve tek gizli katmanlı bir ağın, yeterince nöron içermesi koşuluyla, sürekli herhangi bir fonksiyona istenilen doğrulukta yaklaşabileceğini söyler. Ancak teorem gereken nöron sayısının ne kadar büyük olacağını veya ağın bu ağırlıkları eğitimle bulup bulamayacağını garanti etmez. Pratikte derin (çok katmanlı) ağlar, aynı fonksiyonu geniş ama sığ ağlardan çok daha az parametreyle öğrenebilir; çünkü her katman bir öncekinin öğrendiği özellikleri birleştirerek hiyerarşik temsiller kurar. Şekil 6.5, gizli katman eklemenin ve nöron sayısını artırmanın karar sınırına etkisini göstermektedir.
Gizli katmanlarda genellikle ReLU kullanılırken çıkış katmanının aktivasyonu ve kayıp fonksiyonu, problemin türüne göre seçilir (Tablo 6.2). Bu tablo, derin öğrenme modellerinin tasarımında en sık başvurulan kurallardan biridir.
✓ Doğrusal olmayan aktivasyon şarttır; aksi hâlde çok katmanlı ağ tek katmanlı doğrusal modele indirgenir. Gizli katmanlarda varsayılan seçim ReLU’dur.
✓ Çıkış katmanı göreve göre seçilir: regresyonda doğrusal + MSE, ikili sınıflandırmada sigmoid + ikili çapraz entropi, çok sınıflıda softmax + kategorik çapraz entropi.
9. OR kapısını gerçekleştiren bir perceptron için ağırlık ve bias değerleri öneriniz ve dört girdi için çıktıları tablo hâlinde göstererek doğrulayınız.
olan bir dağılımdan çekilirse ön aktivasyon z’nin standart sapması yaklaşık kaç olur ve bu neden sorundur? (b) He başlatmasında ağırlıkların standart sapması ne olmalıdır? (c) Tüm ağırlıklar aynı sabit değerle başlatılırsa ne olur?
Bu haftanın sonunda öğrenciler; • softmax fonksiyonuyla sınıf olasılıklarını ve kategorik çapraz entropi kaybını hesaplayabi-
• epok, yığın (batch) ve iterasyon kavramlarını ayırt edebilecek, • hesaplama grafiği üzerinde zincir kuralını uygulayarak gradyanları bulabilecek, • küçük bir ağda geri yayılım algoritmasını adım adım uygulayabilecek, • SGD, momentum ve Adam optimizasyon algoritmalarını karşılaştırabilecek, • gradyan sönümlenmesi, dropout, toplu normalizasyon ve erken durdurma kavramlarını açıklayabilecek ve PyTorch ile basit bir eğitim döngüsü yazabilecek, • Adam güncellemesini ve toplu normalizasyonu sayısal olarak uygulayabilecek; öğrenme oranı takvimlerini ve gradyan kırpmayı açıklayabilecektir.
Geçen hafta bir ağın girdiden çıktıya nasıl hesap yaptığını gördük. Bu hafta ağın ağırlıklarının veriden nasıl öğrenildiğini inceleyeceğiz. Önce çok sınıflı sınıflandırmada çıkış katmanını tamamlaya-
C sınıflı bir problemde ağın son katmanı C adet ham skor üretir; bunlara logit (logit) denir. Softmax (softmax) fonksiyonu bu skorları toplamı 1 olan pozitif olasılıklara dönüştürür:
Üstel fonksiyon her skoru pozitif yapar ve büyük skorları daha da belirgin hâle getirir; paydadaki toplam ise olasılıkların toplamını 1’e eşitler.
One-hot vektörde yalnızca doğru sınıfın elemanı 1 olduğundan toplamdan tek terim kalır: Kayıp, modelin doğru sınıfa verdiği olasılığın negatif logaritmasıdır. İki sınıflı durumda bu ifade 4. haftadaki ikili çapraz entropiye indirgenir.
tir. (a) Softmax olasılıklarını hesaplayınız. (b) Görüntü gerçekte bir kediyse kayıp nedir? (c) Görüntü gerçekte bir kuşsa kayıp nedir?
Bir sinir ağının eğitimi, aynı dört adımın binlerce kez tekrarlanmasından oluşur (Şekil 7.2): (1) bir grup örnek için ileri yayılım ile tahmin yap, (2) kaybı hesapla, (3) geri yayılım ile kaybın her parametreye göre gradyanını bul, (4) optimizasyon algoritmasıyla parametreleri güncelle.
• İterasyon (iteration): Bir yığın için yapılan tek bir ileri + geri yayılım + güncelleme adımı. • Epok (epoch): Eğitim kümesindeki tüm örneklerin modele bir kez gösterilmesi. Bir epoktaki iterasyon sayısı ⌈n/yığın boyutu⌉’dur.
Bir ağdaki milyonlarca parametrenin gradyanını tek tek elle türetmek imkânsızdır. Bunun yerine hesaplama, basit işlemlerden oluşan bir hesaplama grafiği (computational graph) olarak düşünülür ve türevler, analizdeki zincir kuralıyla (chain rule) grafiğin sonundan başına doğru aktarılır:
Her düğüm yalnızca kendi yerel türevini bilir; bu yerel türev, sonraki düğümden gelen gradyanla çarpılarak bir önceki düğüme iletilir.
Toplama düğümü gelen gradyanı girdilerine aynen dağıtır. Çarpma düğümü her girdiye, gelen gradyanın diğer girdiyle çarpımını iletir. ReLU düğümü ileri geçişte girdisi pozitifse grad-
Geri yayılım (backpropagation), zincir kuralının bir sinir ağına katman katman, verimli biçimde uygulanmasıdır. 1986’da Rumelhart, Hinton ve Williams tarafından yaygınlaştırılan bu algoritma, ileri yayılımda hesaplanan ara değerleri saklar ve gradyanları çıkıştan girdiye doğru tek bir geçişte hesaplar. Böylece tüm parametrelerin gradyanı, yaklaşık bir ileri yayılım maliyetinde elde edilir. Sigmoid çıkış ve ikili çapraz entropi (veya softmax ve kategorik çapraz entropi) birlikte kullanıldığında çıkış katmanındaki gradyan son derece sadedir:
PyTorch, TensorFlow ve JAX gibi derin öğrenme kütüphaneleri, yukarıda elle yaptığımız işlemi otomatik türev (automatic differentiation) ile yapar: İleri geçiş sırasında hesaplama grafiğini kaydeder ve loss.backward() çağrıldığında tüm gradyanları hesaplar. Bu sayede araştırmacılar yalnızca modelin ileri geçişini yazar.
• RMSProp: Her parametre için gradyan karelerinin hareketli ortalamasını tutar ve adımı bununla böler; böylece büyük gradyanlı parametrelerde adım küçülür, küçük gradyanlılarda büyür (parametreye özgü öğrenme oranı). • Adam: Momentum ile RMSProp’u birleştirir. Birinci moment (gradyanların ortalaması) yönü,
t. adımdaki gradyan gt olmak üzere Adam, iki hareketli ortalama tutar ve bunları başlangıçtaki sıfır değerinin etkisini gidermek için düzeltir:
İki gözlem: (i) Adam’ın adımı gradyanın büyüklüğünden neredeyse bağımsızdır ve η mertebesindedir; büyük gradyanlı parametrelerde adım otomatik olarak küçülür. (ii) Üçüncü adımda gradyan işaret değiştirdiği hâlde momentum sayesinde hareket aynı yönde, ama yavaşlayarak
Öğrenme oranı takvimi: Eğitim boyunca öğrenme oranını sabit tutmak yerine kademeli olarak azaltmak (learning rate schedule) genellikle daha iyi sonuç verir. Başta büyük adımlarla hızlı ilerlenir, sona doğru küçük adımlarla en küçük noktaya yerleşilir. Büyük modellerde ilk adımlarda öğrenme oranını sıfırdan yavaşça artırmak (warm-up) ve ardından kosinüs eğrisiyle azaltmak yaygındır. İki
Geri yayılımda gradyan, her katmandan geçerken o katmanın aktivasyon türeviyle ve ağırlıklarıyla çarpılır. Sigmoid fonksiyonunun türevi en fazla 0.25’tir (Şekil 7.4). On sigmoid katmanından geçen
ve bu katmanlar öğrenemez. Bu soruna gradyan sönümlenmesi (vanishing gradient) denir. Tersine, ağırlıklar büyükse gradyanlar katmanlar boyunca katlanarak büyüyebilir (gradyan patlaması (exp-
Başlıca çözümler: gizli katmanlarda ReLU ailesi aktivasyonlar kullanmak, uygun ağırlık başlatma (ReLU için He, tanh için Xavier/Glorot başlatma), toplu normalizasyon, artık bağlantılar (ResNet, 10. hafta) ve gradyan patlamasına karşı gradyan kırpma (gradient clipping). Gradyan kırpma, gradyan vektörünün normu bir c eşiğini aşarsa vektörü c g/∥g∥ ile değiştirir;
Tüm ağırlıklar aynı değerle (ör. sıfırla) başlatılırsa bir katmandaki tüm nöronlar aynı çıktıyı üretir, aynı gradyanı alır ve aynı şekilde güncellenir. Nöronlar hiçbir zaman farklılaşamaz (simetri sorunu). Bu nedenle ağırlıklar küçük rastgele değerlerle başlatılır; bias değerleri ise sıfırla
Toplu normalizasyon (batch normalization), bir katmanın çıktılarını her mini-yığın içinde ortalaması 0, varyansı 1 olacak şekilde normalleştirir ve ardından öğrenilebilir bir ölçek (γ) ve kaydırma (β) parametresi uygular. Katmanlara gelen girdilerin dağılımını kararlı tuttuğu için daha büyük öğrenme oranlarıyla daha hızlı eğitim sağlar. Transformer modellerinde bunun yerine her örneği kendi içinde normalleştiren katman normalizasyonu (layer normalization) kullanılır. Matematiksel olarak, bir mini-yığındaki N örnek için katmanın her j özniteliği ayrı ayrı normalleş-
Test sırasında tek bir örnek için yığın istatistiği olmadığından eğitim boyunca biriktirilen hareketli ortalamalar kullanılır; PyTorch’ta model.eval() çağrısı bu geçişi yapar. Katman genellikle doğrusal/evrişim katmanından sonra, aktivasyondan önce yerleştirilir.
Dropout (dropout), eğitim sırasında her iterasyonda nöronların rastgele bir kısmını (ör. %20–50) geçici olarak devre dışı bırakır (Şekil 7.5). Böylece ağ belirli nöronlara aşırı güvenemez ve bilgiyi daha dağıtık biçimde temsil etmeyi öğrenir. Bu, her adımda farklı bir alt ağ eğitmeye ve test sırasında bu alt ağların ortalamasını almaya benzer. Test sırasında dropout kapatılır ve tüm nöronlar kullanılır.
Eğitim ilerledikçe eğitim kaybı sürekli azalır; doğrulama kaybı ise bir noktadan sonra yükselmeye başlar. Bu, modelin eğitim verisini ezberlemeye başladığının işaretidir. Erken durdurma (early stopping), doğrulama kaybını her epokta izler; kayıp belirli sayıda epok (patience) boyunca iyileşmezse eğitimi durdurur ve doğrulama kaybının en düşük olduğu andaki ağırlıkları geri yükler (Şekil 7.6).
Günümüzde derin öğrenme modelleri çoğunlukla PyTorch (Meta) veya TensorFlow/Keras (Google) ile geliştirilir. Bu kütüphaneler tensör (çok boyutlu dizi) işlemlerini GPU’da çalıştırır, otomatik türev alır ve hazır katmanlar, kayıp fonksiyonları ve optimizasyon algoritmaları sunar. Aşağı-
verisinin bir kez kullanılmasıdır. ✓ Geri yayılım, zincir kuralını hesaplama grafiğinde çıkıştan girdiye uygular; çıkış katmanın-
✓ Momentum salınımları sönümler; Adam, momentum ile parametreye özgü adım büyüklüğünü birleştirir ve en yaygın optimizasyon algoritmasıdır. ✓ Sigmoid türevi en fazla 0.25 olduğundan derin sigmoid ağlarında gradyan sönümlenir; ReLU, iyi başlatma, toplu normalizasyon ve artık bağlantılar bu sorunu azaltır. ✓ Dropout, erken durdurma, ağırlık sönümü ve veri artırma derin ağlarda aşırı öğrenmeyi
✓ Adam: m̂t yönü, v̂t ölçeği belirler; adım büyüklüğü yaklaşık η mertebesindedir. Öğrenme oranı takvimleri (basamaklı, kosinüs) sona doğru adımı küçültür; gradyan kırpma normu
✓ Toplu normalizasyon her özniteliği yığın içinde standartlaştırıp γ, β ile yeniden ölçekler; testte hareketli ortalamalar kullanılır.
6. Derin bir ağın ilk katmanlarındaki ağırlıkların eğitim boyunca neredeyse hiç değişmediği gözlenmiştir. Gizli katmanlarda sigmoid kullanıldığı biliniyorsa en olası neden ve çözüm nedir? A Aşırı öğrenme; dropout eklemek B Gradyan sönümlenmesi; ReLU aktivasyonu ve uygun başlatma kullanmak C Gradyan patlaması; öğrenme oranını artırmak D Veri sızıntısı; test kümesini değiştirmek E Eksik öğrenme; epok sayısını azaltmak
7. Dropout için aşağıdakilerden hangisi doğrudur? A Hem eğitimde hem testte aynı oranda nöron kapatılır. B Yalnızca eğitim sırasında, her adımda rastgele nöronlar kapatılır. C Ağırlıkları kalıcı olarak sıfırlar. D Eğitimi hızlandırmak için kullanılır, düzenlileştirme etkisi yoktur. E Yalnızca çıkış katmanına uygulanır.
mış gradyanı bulunuz. (c) Toplu normalizasyon neden test sırasında yığın istatistikleri yerine hareketli ortalamaları kullanır?
Bu haftanın sonunda öğrenciler; • sayısal bir görüntünün piksellerden oluşan bir matris olarak nasıl temsil edildiğini açıklayabilecek ve bellek ihtiyacını hesaplayabilecek, • RGB ve gri ton görüntüler arasında dönüşüm yapabilecek, • histogramı yorumlayabilecek; parlaklık, kontrast ve histogram eşitleme işlemlerini açıkla-
• eşikleme ile ikili görüntü elde edebilecek, • bir görüntüye çekirdek (filtre) ile evrişim uygulayabilecek ve sonucu elle hesaplayabilecek, • bulanıklaştırma, keskinleştirme, medyan filtresi ve Sobel/Canny kenar bulma yöntemlerini karşılaştırabilecek, • temel morfolojik işlemleri (aşındırma, genişletme, açma, kapama) açıklayabilecek, • histogram eşitleme dönüşümünü elle hesaplayabilecek ve geometrik dönüşümleri homojen koordinatlarda ifade edip çift doğrusal enterpolasyon uygulayabilecektir.
Bilgisayar için bir görüntü, sayılardan oluşan bir matristen ibarettir. Görüntünün her bir küçük karesine piksel (pixel, picture element) denir ve her piksel o noktanın parlaklığını ya da rengini sayısal
Görüntü işlemede orijin sol üst köşededir. Bir piksel I (y, x ) veya I [satır, sütun] şeklinde gösterilir; satır indeksi aşağı, sütun indeksi sağa doğru artar. NumPy ve OpenCV’de bir görüntünün
Renkli bir görüntü gri tona, insan gözünün yeşile en duyarlı, maviye en az duyarlı olmasını hesaba katan ağırlıklı bir ortalamayla dönüştürülür:
RGB dışında, renk tonunu parlaklıktan ayıran HSV (ton, doygunluk, değer) gibi renk uzayları da vardır; örneğin “kırmızı nesneleri bul” gibi görevlerde HSV daha kullanışlıdır.
çözünürlüklü, kanal başına 8 bitlik sıkıştırılmamış bir renkli görüntü kaç bayt yer kaplar? (c) Saniyede 30 kare kaydeden bir kamera bir dakikada ne kadar ham veri üretir?
neden JPEG, PNG, H.264 gibi yöntemlerle sıkıştırıldığını ve görüntülerle çalışan modellerin neden çok fazla hesaplama gücü gerektirdiğini açıklar.
Her pikselin yeni değerinin yalnızca kendi eski değerine bağlı olduğu işlemlere nokta işlemleri denir. Girdi piksel değeri r, çıktı s olmak üzere:
Bir görüntünün histogramı (histogram), her yoğunluk değerine sahip piksel sayısını gösteren grafiktir. Histogram görüntünün içeriği hakkında değil, ton dağılımı hakkında bilgi verir: Sola yığılmış histogram karanlık, sağa yığılmış aydınlık, dar bir aralığa sıkışmış histogram ise düşük kontrastlı bir görüntüyü gösterir. Histogram eşitleme (histogram equalization), yoğunluk değerlerini histogram tüm aralığa yayılacak şekilde yeniden dağıtarak kontrastı otomatik olarak artırır (Şekil 8.3). Tıbbi görüntülerde ve düşük ışıkta çekilmiş fotoğraflarda sıkça kullanılır. Histogram eşitleme, kümülatif histogramı dönüşüm fonksiyonu olarak kullanır. L gri seviyeli ve N piksellik bir görüntüde n j , j seviyesindeki piksel sayısı olmak üzere her k seviyesi şu yeni değere
narak komşularından ayrılmıştır. Yeni histogram tam düz olmaz (3 ve 4 seviyeleri aynı değere gider), ancak aralık çok daha iyi kullanılır. Eşitleme tüm görüntüye tek bir dönüşüm uyguladığından yerel kontrastı artırmak için görüntüyü parçalara bölüp her parçayı ayrı eşitleyen
Eşikleme, nesneyi arka plandan ayırmanın en basit yoludur ve 11. haftada göreceğimiz segmentasyonun en temel biçimidir. Eşiği elle seçmek zordur. Otsu yöntemi (Otsu’s method), histogramı iki sınıfa (nesne ve arka plan) ayırırken sınıflar içi varyansı en küçük yapan eşiği otomatik olarak bulur (Şekil 8.4). Aydınlatmanın görüntü boyunca değiştiği durumlarda her bölge için ayrı eşik hesaplayan uyarlamalı eşikleme daha iyi sonuç verir.
Nokta işlemlerinin aksine çoğu görüntü işleme yöntemi, bir pikselin yeni değerini onun komşuluğuna bakarak hesaplar. Bunun temel aracı evrişim (convolution) işlemidir: Çekirdek (kernel) ya da
Bu işlem, 9. haftada göreceğimiz evrişimli sinir ağlarının (CNN) yapı taşıdır. Tek fark, CNN’lerde çekirdek değerlerinin elle seçilmek yerine veriden öğrenilmesidir.
karşılıklı elemanlar çarpılıp toplanarak çıktının sol üst elemanı (mavi) bulunur. Çekirdek bir piksel sağa kaydırılarak işlem tekrarlanır.
Matematikte ve klasik sinyal işlemede “evrişim”, çekirdek 180° döndürüldükten sonra uygulanır. Denklem (8.4)’deki döndürmesiz işlemin doğru adı çapraz korelasyondur. Simetrik çekirdeklerde (ör. Gauss) iki işlem aynı sonucu verir. Derin öğrenme kütüphaneleri de döndürmesiz işlemi kullanır ve buna “evrişim” der; çekirdek zaten öğrenildiği için bu fark önemsizdir. Bu notlarda da bu kullanımı izliyoruz.
• Ortalama (kutu) filtresi: Tüm elemanları 1/k2 olan çekirdek. Her pikseli komşularının ortalamasıyla değiştirir; gürültüyü azaltır ama kenarları da bulanıklaştırır.
• Gauss filtresi: Merkeze yakın piksellere daha büyük ağırlık verir; daha doğal bir bulanıklık üretir. Bulanıklığın derecesi standart sapma σ ile ayarlanır. Kenar bulmadan önce gürültüyü azaltmak için sıkça kullanılır. • Keskinleştirme: Merkez pikseli güçlendirip komşuları çıkaran çekirdek (ör. merkez 5, dört komşu −1) ayrıntıları belirginleştirir.
değiyle yaklaşık hesaplanır; düz bölgelerde 0, kenarların iki yanında zıt işaretli değerler verir. Keskinleştirme maskesi (unsharp masking) görüntüden bulanık kopyasını çıkararak ayrıntı-
Görüntüdeki rastgele parlak ve karanlık noktalara tuz-biber gürültüsü denir. Bu tür gürültüde medyan filtresi (median filter) çok etkilidir: Her piksel, komşuluğundaki değerlerin ortancasıyla (medyanıyla) değiştirilir. Medyan bir evrişim değildir (doğrusal değildir); ancak aykırı değerlerden etkilenmediği için hem gürültüyü temizler hem de kenarları korur (Şekil 8.7).
Ortalama filtresi gürültü noktasının etkisini komşularına yayarken (40 civarında soluk bir leke kalır), medyan filtresi aykırı değeri tamamen yok sayar ve komşulara uygun bir değer üretir.
Kenarlar, parlaklığın hızlı değiştiği yerlerdir ve nesnelerin sınırlarını belirler. Parlaklık değişimi, görüntünün gradyanı (türevi) ile ölçülür. Sobel operatörü (Sobel operator), yatay ve dikey türevleri yaklaşık olarak hesaplayan iki çekirdekten oluşur:
Gx soldan sağa parlaklık değişimini (dikey kenarları), Gy yukarıdan aşağıya değişimi (yatay kenarları) yakalar. G ise kenarın gücünü (gradyan büyüklüğünü) verir. Kenarın yönü
Canny kenar dedektörü (Canny edge detector), en yaygın kullanılan kenar bulma algoritmasıdır ve dört adımdan oluşur: (1) Gauss filtresiyle gürültüyü azalt, (2) Sobel ile gradyan büyüklüğünü ve yönünü bul, (3) kenarları tek piksel kalınlığa incelt (maksimum olmayanların bastırılması), (4) iki eşikle (histerezis) güçlü kenarları ve onlara bağlı zayıf kenarları tut. Sonuç ince, sürekli ve gürül-
• Aşındırma (erosion): Nesneleri inceltir; yapı elemanı nesnenin içine tamamen sığmıyorsa piksel silinir. Küçük beyaz gürültüleri yok eder, ince bağlantıları koparır. • Genişletme (dilation): Nesneleri kalınlaştırır; küçük delikleri doldurur, kopuk parçaları birleş-
Morfolojik işlemler özellikle eşikleme veya segmentasyon sonrasında maskeleri temizlemek için kullanılır. Örneğin bir hücre görüntüsünde eşikleme sonrası oluşan küçük gürültüler açma ile temizlenir, ardından bağlantılı bileşen analiziyle hücreler sayılır.
gulamada dönüşüm tersine uygulanır: Çıktı görüntüsündeki her piksel için kaynak konum hesaplanır ve oradaki değer alınır. Kaynak konum genellikle tam sayı değildir; değer, komşu piksellerden enterpolasyon (interpolation) ile elde edilir. En yakın komşu enterpolasyonu en yakın pikselin değerini alır (hızlı, ama basamaklı kenarlar üretir); çift doğrusal (bilinear) enterpolasyon dört komşunun uzaklıkla ağırlıklı ortalamasını alır. Sol üst köşesi (0, 0) olan birim karede köşe değerleri
olur. Geometrik dönüşümler, 10. haftada göreceğimiz veri artırma yönteminin temelidir: Eğitim görüntüleri rastgele döndürülüp ölçeklenerek modelin bu değişimlere dayanıklı olması sağlanır.
Bu hafta gördüğümüz yöntemlerde tüm çekirdekler ve eşikler insan tarafından tasarlanır. 2012 öncesinde bilgisayarlı görü sistemleri, kenar, köşe ve doku gibi bu tür elle tasarlanmış özniteliklerden (ör. SIFT, HOG) yararlanırdı. Evrişimli sinir ağları ise aynı evrişim işlemini kullanır, ancak hangi çekirdeklerin işe yarayacağını veriden öğrenir. İlginç biçimde, eğitilmiş CNN’lerin ilk katmanlarındaki çekirdekler kendiliğinden Sobel benzeri kenar dedektörlerine ve Gauss benzeri filtrelere
✓ Histogram ton dağılımını gösterir; histogram eşitleme kontrastı artırır. Eşikleme ikili görüntü üretir; Otsu yöntemi eşiği otomatik seçer. ✓ Evrişimde çekirdek görüntü üzerinde kaydırılır, eleman eleman çarpım toplamı alınır. Dol-
A Görüntü çok karanlıktır; negatif alınmalıdır. B Görüntünün kontrastı düşüktür; histogram eşitleme uygulanmalıdır. C Görüntü gürültülüdür; medyan filtresi uygulanmalıdır. D Görüntü ikilidir; eşikleme gerekmez. E Görüntü renklidir; gri tona çevrilmelidir.
Bu haftanın sonunda öğrenciler; • tam bağlı ağların görüntülerdeki sınırlılıklarını ve CNN’lerin yerel bağlantı ile ağırlık paylaşımı fikirlerini açıklayabilecek, • çok kanallı girdiler için evrişim katmanının çıktısını elle hesaplayabilecek, • çekirdek boyutu, adım ve dolgu verildiğinde çıktı boyutunu hesaplayabilecek, • evrişim katmanının parametre sayısını hesaplayıp tam bağlı katmanla karşılaştırabilecek, • maksimum ve ortalama havuzlamayı uygulayabilecek, • tipik bir CNN mimarisini katman katman çözümleyebilecek ve PyTorch ile basit bir CNN
• bir CNN’de alıcı alanı katman katman hesaplayabilecek; GAP ile düzleştirmeyi ve bir evrişim katmanının işlem sayısını (MAC) karşılaştırabilecektir.
• Uzamsal yapı kaybolur: Görüntü düzleştirildiğinde (vektöre çevrildiğinde) hangi pikselin hangisine komşu olduğu bilgisi atılır. • Konum bağımlılığı: Sol üst köşedeki bir kediyi tanımayı öğrenen ağ, aynı kediyi sağ alt köşede gördüğünde bunu yeniden öğrenmek zorunda kalır.
Evrişimli sinir ağları (convolutional neural networks, CNN), 8. haftada gördüğümüz evrişim işlemini bir katman olarak kullanarak bu sorunları üç fikirle çözer:
Bir kenar dedektörü görüntünün her yerinde aynı işe yaradığı için ayrı ayrı ağırlıklar öğrenmeye gerek yoktur. Parametre sayısı görüntü boyutundan bağımsız hâle gelir. 3. Öteleme eşdeğerliği (translation equivariance): Girdideki bir örüntü kaydığında, öznitelik haritasındaki karşılığı da aynı miktarda kayar. Havuzlama katmanları ise küçük kaymalara karşı
Bir evrişim katmanı, her biri ayrı bir örüntüyü arayan çok sayıda filtreden (filter) oluşur. Her filtre girdi üzerinde kaydırılır ve bir öznitelik haritası (feature map) (aktivasyon haritası) üretir. Öznitelik haritası, filtrenin aradığı örüntünün görüntünün neresinde ne kadar güçlü bulunduğunu gösterir (Şekil 9.2). Katmandaki filtre sayısı, çıktının kanal (derinlik) sayısını belirler.
eleman çarpılıp tüm kanallar üzerinden toplanır ve bias eklenir; sonuç tek bir sayıdır. Dolayısıyla her filtre, girdi kaç kanallı olursa olsun tek bir öznitelik haritası üretir. Cçıktı filtre kullanılırsa çıktı
İki kanallı girdiden tek kanallı bir öznitelik haritası elde edilmiştir. Ardından genellikle ReLU uygulanır (burada tüm değerler pozitif olduğu için değişmez).
yüksekliğini yaklaşık yarıya indirir. • Dolgu (padding) P: Girdinin kenarlarına eklenen sıfır çerçevesinin kalınlığı. Dolgu olmadan her
Tam bağlı katman yaklaşık 5 400 kat daha fazla parametre gerektirir; üstelik evrişim katmanı her konumda 64 farklı öznitelik üretirken tam bağlı katman yalnızca 64 sayı üretir.
Havuzlama (pooling) katmanı öznitelik haritalarının uzamsal boyutunu küçültür. Böylece hesaplama azalır, sonraki katmanların daha geniş bir bölgeyi “görmesi” sağlanır ve küçük kaymalara karşı dayanıklılık kazanılır. Havuzlamanın öğrenilen parametresi yoktur ve her kanala ayrı ayrı uygulanır; kanal sayısını değiştirmez.
ve yükseklik yarıya iner. • Ortalama havuzlama (average pooling): Her pencerenin ortalaması alınır. • Global ortalama havuzlama (global average pooling): Her öznitelik haritasının tamamının ortala-
ve yükseklik azalır, kanal sayısı artar: Görüntü “küçük ama derin” bir temsile dönüşür. 2. Sınıflandırıcı: Son öznitelik haritaları düzleştirilir (flatten) veya global ortalama havuzlamayla vektöre çevrilir; ardından bir veya birkaç tam bağlı katman ve softmax çıkışı gelir.
Dikkat çekici bir sonuç: Evrişim katmanları toplam parametrelerin yalnızca yaklaşık %4’ünü oluşturur; parametrelerin büyük kısmı tam bağlı katmanlardadır. Modern mimariler bu yüzden düzleştirme yerine global ortalama havuzlama kullanır.
boyutlu tensörü HWC uzunluğunda bir vektöre açar; ardından gelen tam bağlı katmanın parametre sayısı bu uzunlukla çarpıldığı için çok büyür ve model girdinin boyutuna bağımlı hâle gelir. Global ortalama havuzlama (global average pooling, GAP) ise her kanalın tüm uzamsal değerlerinin ortalamasını alarak C uzunluğunda bir vektör üretir; parametresi yoktur ve girdi boyutu de-
havuzlama katmanları bu büyümeyi hızlandırır. Bu nedenle derin ağlar hiyerarşik öznitelikler öğrenir (Şekil 9.6): İlk katmanlar kenar ve renk geçişlerini, orta katmanlar doku ve basit şekilleri (göz, tekerlek, yaprak damarı), son katmanlar ise nesne parçalarını ve bütün nesneleri temsil eder. Alıcı alan katman katman hesaplanabilir. l. katmanın çekirdek boyutu k l , adımı sl olsun; rl alıcı alanı, jl ise bu katmandaki komşu iki nöronun girdide kaç piksel aralıkla yerleştiğini (etkin adım)
yalnızca 1 piksel büyütmüş, ancak etkin adımı ikiye katlayarak sonraki evrişimin katkısını iki katına çıkarmıştır; derin ağlarda alıcı alanın hızla büyümesinin nedeni budur. Alıcı alanı parametre eklemeden büyütmenin bir başka yolu genişletilmiş evrişimdir (dilated convolution):
Parametre sayısı belleği, işlem sayısı ise hızı belirler ve ikisi çok farklı olabilir. Bir evrişim katmanı, çıktının her konumunda ve her çıktı kanalında K · K · Cgirdi çarpma-toplama (multiply–
metresi vardır. Parametrelerin çoğu derin katmanlarda ve tam bağlı katmanlarda, işlemlerin çoğu ise yüksek çözünürlüklü ilk katmanlardadır. Bu yüzden ilk katmanlarda adım 2 veya havuzlama kullanılarak çözünürlük hızla düşürülür; toplam işlem sayısı (GFLOP), bir modelin telefonda çalışıp çalışamayacağını belirleyen ana ölçüttür.
PyTorch görüntüleri (yığın, kanal, yükseklik, genişlik) yani NCHW sırasıyla bekler. OpenCV ve NumPy ise görüntüyü (yükseklik, genişlik, kanal) sırasıyla tutar. Görüntüyü modele vermeden önce img.permute(2, 0, 1) ile eksenlerin yeri değiştirilmelidir; torchvision.transforms.ToTensor() bunu otomatik yapar.
✓ CNN’ler yerel bağlantı ve ağırlık paylaşımı sayesinde görüntülerde tam bağlı ağlardan çok daha az parametreyle çalışır ve uzamsal yapıyı korur. ✓ Her filtre girdinin tüm kanallarını kapsar ve tek bir öznitelik haritası üretir; çıktının kanal sayısı filtre sayısına eşittir.
kanal sayısını değiştirir. İşlem sayısı Oh Ow K2 Cgirdi Cçıktı MAC’tır; parametre sayısıyla aynı şey değildir ve çıktı çözünürlüğüyle ölçeklenir.
5. Maksimum havuzlama katmanı için aşağıdakilerden hangisi yanlıştır? A Öğrenilebilir parametresi yoktur. B Her kanala ayrı ayrı uygulanır. C Uzamsal boyutu küçültür. D Kanal sayısını iki katına çıkarır. E Küçük kaymalara karşı dayanıklılık sağlar.
Bu haftanın sonunda öğrenciler; • AlexNet, VGG, GoogLeNet ve ResNet mimarilerinin getirdiği temel yenilikleri açıklayabi-
• artık bağlantıların gradyan akışına katkısını açıklayabilecek, • transfer öğrenmede öznitelik çıkarma ile ince ayar arasındaki farkı ve hangisinin ne zaman seçileceğini tartışabilecek, • veri artırma yöntemlerini sayabilecek ve amacını açıklayabilecek, • nesne tespitinde IoU değerini hesaplayabilecek, maksimum olmayanların bastırılması (NMS) algoritmasını uygulayabilecek ve tek/iki aşamalı dedektörleri karşılaştırabilecek, • derinlemesine ayrılabilir evrişimin tasarrufunu ve bir sınıf için AP değerini hesaplayabilecek; çapa kutuları ve odak kaybının amacını açıklayabilecektir.
fazla etiketli görüntü içeren bir veri setidir. 2010–2017 yılları arasında düzenlenen ImageNet Büyük Ölçekli Görsel Tanıma Yarışması (ILSVRC), 1000 sınıflı yaklaşık 1.2 milyon eğitim görüntüsüyle bilgisayarlı görünün en önemli ölçütü oldu. Yarışmada top-5 hata, yani doğru sınıfın modelin en olası beş tahmini arasında yer almadığı görüntülerin oranı ölçülür. Şekil 10.1, 2012’de AlexNet ile başlayan derin CNN’lerin bu hatayı birkaç yıl içinde insan düzeyinin altına indirdiğini gösterir.
Krizhevsky, Sutskever ve Hinton’ın geliştirdiği AlexNet, LeNet’in fikirlerini çok daha büyük ölçekte uyguladı: 5 evrişim ve 3 tam bağlı katman, yaklaşık 60 milyon parametre. Başarısının ardındaki
yenilikler bugün standart hâle gelmiştir: sigmoid/tanh yerine ReLU, aşırı öğrenmeye karşı dropout ve veri artırma, eğitimin iki GPU üzerinde paralel yürütülmesi.
araya ikinci bir ReLU girdiği için model daha ifade gücüne sahip olur. VGG-16 yaklaşık 138 milyon parametreyle oldukça ağırdır; ancak sade yapısı nedeniyle hâlâ öğretimde ve transfer öğrenmede
evrişimlerden önce kanal sayısını azaltmak için kullanılır (darboğaz, bottleneck). GoogLeNet, AlexNet’ten yaklaşık 12 kat az parametreyle (yaklaşık 7 milyon) daha yüksek doğruluk elde etmiştir.
Darboğaz tasarımı yaklaşık 8.5 kat daha az parametre kullanır. Bu tasarım ResNet-50 ve daha derin ResNet modellerinde de kullanılır.
Ağları derinleştirmek her zaman daha iyi sonuç vermez. Kaiming He ve arkadaşları, 56 katmanlı düz bir ağın 20 katmanlı bir ağdan yalnızca test kümesinde değil eğitim kümesinde de daha kötü olduğunu gözlemledi. Sorun aşırı öğrenme değil, çok derin ağların optimizasyonunun zorlaşmasıydı (bozulma problemi). Çözüm olarak önerilen ResNet, her bloğun girdisini çıktısına doğrudan ekleyen artık bağlantıları (residual / skip connection) kullanır (Şekil 10.2):
Sıfırdan bir derin CNN eğitmek için milyonlarca etiketli görüntü ve günlerce GPU zamanı gerekir. Oysa çoğu gerçek projede (ör. birkaç yüz MR görüntüsü veya yaprak fotoğrafı) bu kadar veri yoktur. Transfer öğrenme (transfer learning), büyük bir veri setinde (genellikle ImageNet) önceden eğitilmiş bir modelin öğrendiği bilgiyi yeni bir göreve aktarır. Bu işe yarar, çünkü CNN’in ilk katmanlarının öğrendiği kenar, doku ve şekil dedektörleri neredeyse tüm görüntü problemlerinde ortaktır.
• Öznitelik çıkarma: Önceden eğitilmiş evrişim tabanının ağırlıkları dondurulur (güncellenmez); yalnızca yeni eklenen sınıflandırıcı katmanı eğitilir. Hızlıdır ve çok az veriyle bile çalışır. • İnce ayar (fine-tuning): Sınıflandırıcı eğitildikten sonra evrişim tabanının son katmanları (veya
Büyük modellerin tüm parametrelerini ince ayarla güncellemek çok pahalı olabilir. Son yıllarda yalnızca küçük bir parametre alt kümesini veya modele eklenen küçük modülleri eğiten parametre-verimli ince ayar (PEFT) yöntemleri geliştirilmiştir. Bunların en bilineni olan LoRA’yı 14. haftada büyük dil modelleri bağlamında göreceğiz; aynı yöntem görüntü modellerinde (ör. ViT) de kullanılmaktadır.
Veri artırma (data augmentation), eğitim görüntülerine anlamı değiştirmeyen rastgele dönüşümler uygulayarak her epokta modele biraz farklı örnekler gösterir (Şekil 10.4). Böylece model “kedinin kedi olması için sağa bakması gerekmediğini” öğrenir; aşırı öğrenme azalır ve genelleme artar. Dönüşümler yalnızca eğitim sırasında, rastgele olarak uygulanır; doğrulama ve test görüntüleri de-
çevirir; trafik işaretlerinde yatay çevirme “sola dön” işaretini “sağa dön”e çevirir; tıbbi görüntülerde renk değişimi tanısal bilgiyi bozabilir. Artırma yöntemleri alan bilgisiyle seçilmelidir.
Şimdiye kadar bir görüntüye tek bir etiket veren sınıflandırma problemini inceledik. Gerçek uygulamalarda görüntüde birden fazla nesne bulunur ve her birinin ne olduğunu ve nerede olduğunu bilmemiz gerekir. Bilgisayarlı görüde görevler, verdikleri ayrıntı düzeyine göre sıralanabilir (Şe-
merkez, genişlik ve yükseklikle ( xc , yc , w, h) biçiminde temsil edilir. Kutu koordinatlarını tahmin etmek bir regresyon, sınıfı tahmin etmek ise bir sınıflandırma problemidir; dedektörler ikisini birlikte
Tahmin edilen kutunun gerçek kutuya ne kadar iyi oturduğu kesişim/birleşim oranı (Intersection over Union, IoU) ile ölçülür:
Dedektörler genellikle aynı nesne için birbirine çok yakın birçok kutu üretir. Maksimum olmayanların bastırılması (Non-Maximum Suppression, NMS) bu tekrarları temizler:
• İki aşamalı dedektörler (R-CNN, Fast R-CNN, Faster R-CNN): Önce nesne içerebilecek aday bölgeler önerilir (region proposal), sonra her aday bölge sınıflandırılır ve kutusu düzeltilir. Genellikle daha doğrudur ama daha yavaştır. • Tek aşamalı dedektörler (YOLO, SSD, RetinaNet): Görüntüyü bir ızgaraya böler ve her hücre için kutuları ve sınıfları tek bir ileri geçişte doğrudan tahmin eder. “You Only Look Once” adından da anlaşılacağı gibi gerçek zamanlı çalışabilir (saniyede onlarca kare). Otonom araçlar, güvenlik kameraları ve tarım robotlarında yaygındır. • Transformer tabanlı dedektörler (DETR): Nesne tespitini bir küme tahmini problemi olarak ele alır; NMS ve elle tasarlanmış çapa kutularına ihtiyaç duymaz.
Dedektörler ortalama kesinlik ortalaması (mean Average Precision, mAP) ile değerlendirilir: Her sınıf için kesinlik–duyarlılık eğrisinin altındaki alan (AP) hesaplanır ve sınıflar üzerinden ortalaması
Kaçırılan nesne AP’yi doğrudan 0.25 düşürmüştür; sıralamanın sonundaki yanlış pozitifler ise hiç zarar vermemiştir. mAP, bu değerin tüm sınıflar (ve COCO’da IoU eşikleri) üzerinden
Tek aşamalı dedektörler her ızgara hücresinde önceden tanımlı birkaç çapa kutusu (anchor box) (farklı boyut ve en-boy oranlarında şablon kutular) kullanır; ağ her çapa için bir nesnelik skoru,
on binlerce çapa olup yalnızca birkaçı nesne içerdiğinden eğitim verisi aşırı dengesizdir: Kolay arka plan örnekleri kaybın tamamına hâkim olur. RetinaNet’in odak kaybı (focal loss) çözümü, çapraz entropiyi kolay örneklerde küçülten bir çarpan ekler:
✓ Transfer öğrenme önceden eğitilmiş modelin bilgisini yeni göreve aktarır: Az veride taban dondurulur, çok veride ince ayar yapılır. ✓ Veri artırma eğitimde anlamı koruyan rastgele dönüşümlerle genellemeyi artırır.
kat azaltır; mobil mimarilerin temelidir. ✓ AP, enterpolasyonlu kesinlik–duyarlılık eğrisinin alanıdır; kaçırılan nesneler AP’yi doğrudan düşürür. Çapa kutuları şablon kutulardır; odak kaybı (1 − p̂)γ çarpanıyla kolay örnek-
B Paralel dallardan oluşan Inception modülü C Bloğun girdisini çıktısına ekleyen artık (atlama) bağlantıları D Sigmoid yerine ReLU kullanması E Derinlemesine ayrılabilir evrişim
3. Elinizde 300 adet göğüs röntgeni vardır ve ImageNet’te eğitilmiş bir modeli kullanmak istiyorsunuz. En uygun başlangıç stratejisi hangisidir? A Modeli sıfırdan, rastgele ağırlıklarla eğitmek B Evrişim tabanını dondurup yalnızca yeni sınıflandırıcıyı eğitmek, ardından gerekirse küçük öğrenme oranıyla sınırlı ince ayar yapmak C Tüm ağı büyük bir öğrenme oranıyla eğitmek D Son katmanı değiştirmeden ImageNet sınıflarıyla tahmin yapmak E Veri artırmayı test kümesine uygulamak
5. Maksimum olmayanların bastırılması (NMS) algoritmasının amacı nedir? A Görüntüdeki gürültüyü temizlemek B Aynı nesne için üretilen birbirine çok benzeyen kutulardan yalnızca en yüksek skorlu
6. Aşağıdaki veri artırma işlemlerinden hangisi el yazısı rakam (MNIST) sınıflandırmada etiketin anlamını değiştirebileceği için uygun değildir?
9. Transfer öğrenmede “öznitelik çıkarma” ile “ince ayar” arasındaki farkı açıklayınız. İnce ayarda neden küçük öğrenme oranı kullanılır?
Bu haftanın sonunda öğrenciler; • anlamsal, örnek ve panoptik segmentasyonu ayırt edebilecek, • eşikleme, bölge büyütme, havza (watershed) ve k-means gibi klasik segmentasyon yöntemlerini açıklayabilecek, • segmentasyonu piksel düzeyinde sınıflandırma olarak formüle edebilecek, • kodlayıcı–kod çözücü yapısını, transpoze evrişimi ve U-Net’teki atlama bağlantılarının ro-
• IoU (Jaccard) ve Dice katsayılarını maskeler üzerinde elle hesaplayabilecek ve piksel doğruluğunun neden yanıltıcı olabileceğini gösterebilecek, • segmentasyonda kullanılan kayıp fonksiyonlarını (piksel bazlı çapraz entropi, Dice kaybı)
• çok sınıflı karışıklık matrisinden mIoU hesaplayabilecek ve bağlantılı bileşen analizi gibi son işleme adımlarını uygulayabilecektir.
Görüntü segmentasyonu (image segmentation), bir görüntüyü anlamlı bölgelere ayırma işlemidir. Sınıflandırma “görüntüde ne var?”, nesne tespiti “nesneler nerede?” sorusunu kutularla cevaplarken segmentasyon her pikselin hangi nesneye veya sınıfa ait olduğunu belirler. Sonuç, görüntüyle aynı boyutta bir maskedir (mask). Bu ayrıntı düzeyi, bir tümörün hacmini ölçmek, bir tarladaki hastalıklı alanın yüzdesini hesaplamak veya otonom bir aracın “sürülebilir yol” bölgesini bulmak gibi görevler için zorunludur.
• Anlamsal segmentasyon (semantic segmentation): Her piksele bir sınıf etiketi verilir (“para”, “arka plan”, “yol”, “araba”). Aynı sınıftaki farklı nesneler ayırt edilmez; tüm paralar aynı renktedir. • Örnek segmentasyonu (instance segmentation): Her nesne ayrı ayrı bulunur ve maskelenir. Yan yana duran iki para iki ayrı nesne olarak çıkar; böylece nesneler sayılabilir.
• Panoptik segmentasyon (panoptic segmentation): İkisini birleştirir: Sayılabilen nesneler (araba, insan) örnek bazında, sayılamayan bölgeler (gökyüzü, yol, çimen) anlamsal olarak etiketlenir.
Derin öğrenme öncesinde segmentasyon, görüntünün düşük seviyeli özelliklerine dayanan yöntemlerle yapılırdı. Bu yöntemler bugün de veri az olduğunda, ön işleme aşamasında veya etiket
• Eşikleme (8. hafta): Nesne ile arka planın parlaklıkları belirgin biçimde farklıysa (ör. koyu zemin üzerinde parlak hücreler) en hızlı ve en basit yöntemdir. • Bölge büyütme (region growing): Kullanıcının seçtiği bir “tohum” pikselden başlanır; benzer yoğunluktaki komşu pikseller adım adım bölgeye eklenir. • Havza (watershed) algoritması (watershed): Gradyan büyüklüğü bir arazi yüzeyi gibi düşünülür; işaretçi noktalarından “su basılır” ve farklı kaynaklardan gelen suların buluştuğu sırtlar bölge sınırı olur. Birbirine değen nesneleri ayırmada etkilidir. Şekil 11.1’deki paralar bu yön-
Klasik yöntemler parlaklık ve renk gibi düşük seviyeli bilgiye dayanır. “Kedi” ile “kediyle aynı renkteki halı” arasındaki farkı bilmezler. Anlamsal düzeyde segmentasyon için görüntünün içeriğini anlayan, yani veriden öğrenen modellere ihtiyaç vardır.
pikselde softmax uygulanır ve en yüksek olasılıklı sınıf o pikselin etiketi olur. İkili segmentasyonda (ör. lezyon / arka plan) tek kanallı çıktı ve piksel başına sigmoid yeterlidir.
Segmentasyonda ise çıktının girdiyle aynı çözünürlükte olması gerekir. Bu nedenle segmentasyon ağları iki bölümden oluşan bir kodlayıcı–kod çözücü (encoder–decoder) yapısı kullanır: • Kodlayıcı (encoder): Evrişim ve havuzlamayla görüntüyü küçültüp derinleştirir; “ne var?” sorusunu cevaplayan anlamsal öznitelikler çıkarır. Çoğu zaman ImageNet’te önceden eğitilmiş bir sınıflandırma ağıdır (ör. ResNet). • Kod çözücü (decoder): Öznitelik haritalarını adım adım büyüterek tekrar girdi çözünürlüğüne çıkarır; “nerede?” sorusunu cevaplar.
Kod çözücüde öznitelik haritasını büyütmek (upsampling) için iki yol vardır: (i) en yakın komşu veya çift doğrusal (bilinear) enterpolasyon gibi sabit yöntemler, (ii) öğrenilebilir transpoze evrişim (transposed convolution). Transpoze evrişimde her girdi değeri çekirdekle çarpılarak çıktıya “yayılır”; adım (S) çekirdeğin çıktıda ne kadar aralıkla yerleştirileceğini belirler. Örtüşen bölgelerde
U-Net, 2015 yılında Ronneberger ve arkadaşları tarafından biyomedikal görüntü segmentasyonu için önerilmiş ve kısa sürede tıbbi görüntülemede standart hâline gelmiştir. Adını, kodlayıcı ve kod çözücünün oluşturduğu “U” biçiminden alır (Şekil 11.3). En önemli özelliği, kodlayıcının her seviyesindeki öznitelik haritalarını kod çözücüdeki aynı çözünürlükteki seviyeye doğrudan aktaran atlama bağlantılarıdır (skip connections). Kodlayıcıda havuzlama sırasında kaybolan ince uzamsal ayrıntılar (kenarlar, küçük yapılar) bu bağlantılar sayesinde kod çözücüye geri taşınır; böylece maske sınırları keskinleşir. U-Net’teki atlama bağlantıları ResNet’teki gibi toplama değil, kanal ekseninde birleştirme (concatenation) yapar.
• FCN (Fully Convolutional Network, 2015): Sınıflandırma ağlarındaki tam bağlı katmanları evrişimle değiştirerek uçtan uca segmentasyonu başlatan ilk model. • DeepLab ailesi: Genişletilmiş (atrous) evrişim ile çözünürlüğü düşürmeden alıcı alanı büyütür; farklı ölçeklerdeki bağlamı paralel dallarla toplar.
• nnU-Net: Veri setine göre ön işleme, mimari ve eğitim ayarlarını otomatik yapılandıran U-Net tabanlı bir çerçeve; birçok tıbbi segmentasyon yarışmasında güçlü bir taban çizgisidir. • SAM (Segment Anything Model, 2023): Milyarlarca maskeyle eğitilmiş bir temel model. Kullanıcının verdiği bir nokta, kutu veya metin istemiyle (prompt) hemen her görüntüde nesneleri sıfır-
Tahmin edilen maske, uzman tarafından çizilmiş gerçek maskeyle (ground truth) piksel piksel karşılaştırılır. Her piksel TP (iki maskede de nesne), FP (yalnızca tahminde nesne), FN (yalnızca ger-
kisi vardır; dolayısıyla Dice her zaman IoU’dan büyük veya ona eşittir. Çok sınıflı problemlerde her sınıfın IoU değerinin ortalaması (mIoU) raporlanır.
Üç sınıflı (arka plan, yol, araç) bir segmentasyon modelinin 100 piksellik bir test görüntüsündeki karışıklık matrisi aşağıdadır (satırlar gerçek sınıf, sütunlar tahmin). Her sınıfın IoU değerini, mIoU’yu ve piksel doğruluğunu hesaplayınız.
Piksel doğruluğu büyük arka plan sınıfının başarısını yansıtır; mIoU ise her sınıfa eşit ağırlık verdiğinden küçük “araç” sınıfındaki hataları da görünür kılar.
Tıbbi görüntülerde lezyon genellikle görüntünün çok küçük bir kısmını kaplar. Lezyonun piksellerin %1’ini oluşturduğu bir MR kesitinde hiçbir şeyi işaretlemeyen bir model %99 piksel doğruluğu elde eder; ancak Dice ve IoU değerleri sıfırdır. Bu nedenle segmentasyonda başarı her zaman Dice ve/veya IoU ile raporlanır. Bu durum, 5. haftadaki doğruluk paradoksunun piksel düzeyindeki karşılığıdır.
• Piksel bazlı çapraz entropi: Her piksel için (ikili veya kategorik) çapraz entropi hesaplanır ve ortalaması alınır. Sınıflar dengesizse arka plan piksellerinin sayısal üstünlüğü nedeniyle model
küçük nesneleri ihmal edebilir; bu durumda sınıflara ağırlık verilir. • Dice kaybı (Dice loss): Dice katsayısının olasılıklarla hesaplanan “yumuşak” sürümü doğrudan
yen küçük bir sayıdır. Dice kaybı yalnızca nesne bölgesinin örtüşmesine odaklandığı için sınıf dengesizliğine karşı dayanıklıdır. • Pratikte ikisinin toplamı (çapraz entropi + Dice) sıkça kullanılır: Çapraz entropi kararlı gradyanlar, Dice ise doğrudan değerlendirme metriğine uygunluk sağlar.
Ağın ürettiği olasılık haritası eşiklenerek maskeye dönüştürüldükten sonra genellikle birkaç küçük düzeltme adımı uygulanır. 8. haftadaki morfolojik açma ve kapama, maskedeki tek tük gürültü piksellerini ve küçük delikleri temizler. Bağlantılı bileşen analizi (connected component analysis), maskedeki birbirine değen piksel gruplarını ayrı nesneler olarak etiketler; böylece nesneler sayılabilir, her birinin alanı ölçülebilir ve belirli bir alanın altındaki parçalar gürültü olarak atılabilir. İki pikselin “komşu” sayılması için 4-komşuluk (yalnızca yatay/dikey) veya 8-komşuluk (çaprazlar dahil) kullanılır ve seçim sonucu değiştirir.
rine çapraz olarak bağlıdır ve tek bir bileşen oluşturur. Toplam 2 bileşen. Nesne sayımı yapan bir uygulamada komşuluk seçimi sonucu bu kadar değiştirebildiğinden, seçim görüntünün yapısına göre bilinçli yapılmalıdır.
Örnek segmentasyonunda birbirine değen nesneleri ayırmak için maskeye uzaklık dönüşümü ve bu haftanın başında gördüğümüz havza algoritması uygulanır. Sınır kalitesini artırmak için geçmişte koşullu rastgele alanlar (CRF) ile renk benzerliğine göre sınır düzeltmesi yapılırdı; günümüz ağlarında bu adım çoğunlukla gereksizdir, ancak hafif modellerde hâlâ görülür.
• Tıp: Beyin tümörü ve inme lezyonu segmentasyonu (MR), organ ve damar segmentasyonu (BT), hücre ve çekirdek segmentasyonu (patoloji), polip tespiti (endoskopi). Lezyon hacmi gibi ölçümler tedavi planlamasında kullanılır. • Tarım: Yapraklarda hastalıklı alanların oranı, yabani ot ile ürünün ayrılması, uydu ve İHA görüntülerinden tarla sınırları. • Otonom sürüş: Yol, şerit, yaya ve araç piksellerinin gerçek zamanlı belirlenmesi. • Uzaktan algılama ve şehircilik: Bina, yol ve su alanlarının haritalanması, afet sonrası hasar
✓ Segmentasyon her pikseli etiketler. Anlamsal segmentasyon sınıfları, örnek segmentasyonu tek tek nesneleri, panoptik segmentasyon ikisini birlikte ayırır. ✓ Klasik yöntemler (eşikleme, bölge büyütme, havza, k-means) düşük seviyeli bilgiye dayanır; anlamsal içerik bilgisi yoktur. ✓ Derin segmentasyon ağları piksel düzeyinde sınıflandırma yapar ve kodlayıcı–kod çözücü yapısı kullanır; büyütme transpoze evrişim veya enterpolasyonla yapılır. ✓ U-Net’te atlama bağlantıları kodlayıcının ince ayrıntılarını kod çözücüye taşır; tıbbi görün-
✓ Küçük nesnelerde piksel doğruluğu yanıltıcıdır; Dice/IoU raporlanmalı, Dice kaybı veya ağırlıklı çapraz entropi kullanılmalıdır. ✓ Çok sınıflı segmentasyonda her sınıfın IoU’su karışıklık matrisinden TP/(satır + sütun −
✓ Son işlemede morfolojik açma/kapama gürültüyü temizler; bağlantılı bileşen analizi nesneleri sayar ve küçük parçaları atar; 4ve 8-komşuluk farklı sonuç verebilir.
2. U-Net mimarisindeki atlama bağlantılarının temel işlevi nedir? A Parametre sayısını azaltmak B Kodlayıcıda kaybolan ince uzamsal ayrıntıları kod çözücüye aktarmak
6. Lezyonun piksellerin yalnızca %0.5’ini kapladığı MR görüntülerinde segmentasyon başarısını raporlamak için en uygunsuz metrik hangisidir?
8. Kodlayıcı–kod çözücü yapısında kodlayıcı ve kod çözücünün görevlerini “ne” ve “nerede” soruları üzerinden açıklayınız.
IoU’sunu ve mIoU’yu hesaplayınız. (b) Lezyon sınıfı için Dice katsayısını bulunuz ve IoU ile ilişkisini doğrulayınız. (c) Piksel doğruluğunu hesaplayıp mIoU ile karşılaştırınız.
verilen bir lezyon pikselinin ve 0.99 olasılık verilen bir arka plan pikselinin ağırlıklı kayıplarını hesaplayınız. (c) Ağırlıklı çapraz entropi yerine Dice kaybı kullanmanın avantajı nedir?
Bu haftanın sonunda öğrenciler; • doğal dil işlemenin temel görevlerini ve dilin bilgisayarlar için neden zor olduğunu açık-
• metin ön işleme adımlarını ve kelime, karakter ve alt kelime tokenizasyonunu karşılaştırabilecek; BPE birleştirmelerini elle uygulayabilecek, • kelime torbası ve TF-IDF temsillerini hesaplayabilecek, • kelime gömmelerinin mantığını açıklayıp kosinüs benzerliğini hesaplayabilecek, • bir dil modelinin ne olduğunu tanımlayabilecek; bigram olasılıklarını ve karmaşıklık (perplexity) değerini hesaplayabilecek, • tekrarlayan sinir ağlarının (RNN) çalışma mantığını ve sınırlılıklarını açıklayabilecek, • karmaşıklığı çapraz entropi kaybıyla ilişkilendirebilecek, negatif örneklemenin amacını ve LSTM kapı denklemlerini açıklayabilecektir.
Doğal dil işleme (natural language processing, NLP), bilgisayarların insan dilini anlamasını, yorumlamasını ve üretmesini amaçlayan yapay zekâ alanıdır. Günlük hayatta kullandığımız birçok sistem
• Metin sınıflandırma: spam filtresi, haber kategorileme, ürün yorumlarında duygu analizi (olumlu/olumsuz), finansal haberlerin piyasa etkisi. • Bilgi çıkarımı: metindeki kişi, kurum, yer ve tarih adlarını bulma (adlandırılmış varlık tanıma); sözleşme ve hasta kayıtlarından yapılandırılmış bilgi çıkarma. • Metin üretme ve dönüştürme: makine çevirisi, özetleme, soru cevaplama, sohbet asistanları,
• Belirsizlik: “Yüz” kelimesi bir sayı, bir organ ya da yüzmek fiili olabilir. “Adam bahçede dürbünle kuşu gördü” cümlesinde dürbünü kimin taşıdığı belirsizdir. • Bağlam ve dünya bilgisi: “Bardağı masaya koydum, çünkü o kırıktı” cümlesinde “o”nun bardak mı masa mı olduğunu anlamak için dünya bilgisi gerekir. • Biçimbilim (morfoloji): Türkçe gibi eklemeli dillerde tek bir kökten çok sayıda kelime türetilir: ev, evler, evlerimiz, evlerimizden, evlerimizdekiler… Bu nedenle Türkçede farklı kelime biçimlerinin sayısı İngilizceye göre çok daha fazladır. • Mecaz, deyim ve ironi: “Ne güzel, yine kargo gecikti!” cümlesi olumlu kelimeler içerse de olum-
Metinle çalışan her modelin ilk adımı, metni küçük birimlere ayırmaktır. Bu birimlere token (token), işleme ise tokenizasyon (tokenization) denir. Tokenizasyondan önce veya sonra uygulanabilecek başlıca ön işleme adımları şunlardır: küçük harfe çevirme (Türkçede “I/ı” ve “İ/i” dönüşümüne dikkat edilmelidir), noktalama işaretlerinin ve gereksiz karakterlerin temizlenmesi, “ve, bu, bir” gibi çok sık geçen ancak az bilgi taşıyan durak kelimelerin (stop words) çıkarılması ve kelimeleri köklerine indirgeme (kök bulma, stemming; lemmatizasyon).
Klasik yöntemlerde (kelime torbası, TF-IDF) durak kelimeleri ve ekleri atmak faydalı olabilir. Ancak modern derin öğrenme modellerinde bu adımlar genellikle yapılmaz: “değil” gibi bir durak kelimesini atmak “iyi değil” ifadesinin anlamını tersine çevirir. Transformer tabanlı modeller ham metinle ve kendi tokenizerlarıyla çalışır.
• Kelime düzeyi: Metin boşluklardan ve noktalama işaretlerinden bölünür. Basittir, ancak sözlük çok büyür ve eğitimde hiç görülmemiş kelimeler (out-of-vocabulary, OOV) temsil edilemez. “evlerimizdekiler” kelimesi eğitim verisinde yoksa model onu tanımaz. • Karakter düzeyi: Her karakter bir tokendir. Sözlük küçüktür ve OOV sorunu yoktur; ancak diziler çok uzar ve tek bir karakter anlam taşımaz. • Alt kelime tokenizasyonu (subword tokenization): Sık kelimeler tek token, nadir kelimeler anlamlı
En yaygın alt kelime algoritması olan bayt çifti kodlaması (Byte Pair Encoding, BPE) basit bir fikre dayanır: Eğitim metnindeki kelimeler önce karakterlerine ayrılır; ardından en sık yan yana gelen sembol çifti bulunup tek bir yeni sembolde birleştirilir. Bu işlem, istenen sözlük boyutuna ulaşılana kadar tekrarlanır. Öğrenilen birleştirmeler sırasıyla saklanır ve yeni metinlere aynı sırayla uygula-
lime, öğrenilmiş parçalarla temsil edilebilmiştir. Dikkat çekici biçimde algoritma, Türkçedeki çoğul eki “-ler”i kendi kendine keşfetmiştir.
Kelime torbası (bag of words) modelinde bir belge, sözlükteki her kelimenin o belgede kaç kez geçtiğini gösteren bir vektörle temsil edilir. Kelime sırası tamamen göz ardı edilir: “köpek adamı ısırdı” ile “adam köpeği ısırdı” (kök biçimleriyle) aynı vektöre sahip olur. Her kelime aynı derecede bilgi taşımaz: “ve”, “bir” gibi kelimeler her belgede geçer. TF-IDF (term frequency–inverse document frequency), bir kelimenin belgedeki sıklığını, o kelimenin kaç farklı belgede geçtiğine göre ağırlıklandırır:
[kedi, süt, içer, köpek, fare, yakalar] olmak üzere her belgenin TF-IDF vektörünü hesaplayınız (doğal logaritma kullanınız).
“yakalar” kelimeleri, o belgeyi diğerlerinden ayırt ettikleri için daha yüksek ağırlık almıştır. (Kütüphaneler sıfıra bölmeyi önlemek için IDF formülünü biraz farklı uygulayabilir; ör. scikit-
TF-IDF vektörleri lojistik regresyon veya Naive Bayes gibi sınıflandırıcılarla birleştirildiğinde metin sınıflandırma için hızlı ve güçlü bir taban çizgisi oluşturur. Ancak iki temel sorunu vardır: (i) vektörler sözlük boyutunda (on binlerce boyut) ve çoğu sıfır olan seyrek vektörlerdir; (ii) kelimeler arasındaki anlam benzerliğini yakalamazlar: “güzel” ile “harika” birbirinden “güzel” ile “masa”
dan farklı) bir vektörle temsil eder. Bu vektörler, dilbilimdeki dağılımsal hipoteze dayanarak veriden öğrenilir: “Bir kelimeyi, çevresindeki kelimelerden tanırsınız.” Benzer bağlamlarda geçen kelimeler (kedi–köpek, Ankara–İstanbul) benzer vektörlere sahip olur. Mikolov ve arkadaşlarının 2013’te önerdiği Word2Vec iki basit eğitim görevi kullanır: CBOW çevredeki kelimelerden ortadaki kelimeyi, skip-gram ise ortadaki kelimeden çevredeki kelimeleri tahmin eder. Ağ bu görevleri öğrenirken ara katmanında anlamlı kelime vektörleri oluşur. Gömme
Skip-gram, her kelimeye iki vektör verir: Merkez kelime olarak vc , bağlam kelimesi olarak uo . “c merkezken o’nun pencerede görülme olasılığı” tüm sözlük üzerinde bir softmax ile tanım-
lanması çok pahalıdır. Negatif örnekleme, görevi ikili sınıflandırmaya çevirir: Gerçek (merkez, bağlam) çifti için σ(uo · vc ) büyütülürken rastgele seçilen K “sahte” bağlam kelimesi için
Word2Vec gibi yöntemler her kelimeye tek bir vektör verir. Oysa “yüz” kelimesinin “yüz lira” ve “yüzünü yıka” ifadelerindeki anlamları farklıdır. Bu soruna, aynı kelimeye bağlamına göre farklı vektör veren bağlamsal gömmeler (BERT, GPT; 13. hafta) çözüm getirmiştir. Ayrıca gömmeler eğitim verisindeki toplumsal önyargıları (ör. meslek–cinsiyet ilişkileri) da öğrenebilir.
Dil modeli (language model), bir kelime (token) dizisine olasılık atayan modeldir. Zincir kuralına göre bir cümlenin olasılığı, her kelimenin kendisinden önceki kelimeler verildiğinde koşullu olasılıklarının çarpımıdır:
Bu tanım basit görünse de son derece güçlüdür. Telefon klavyesindeki kelime önerileri, konuşma tanıma sistemlerinin “doğru duyduğu” cümleyi seçmesi ve ChatGPT gibi sohbet asistanlarının metin üretmesi aynı ilkeye dayanır: Olası devamlardan birini olasılığına göre seçmek ve tekrarlamak.
Tüm geçmişe bağlı olasılıkları tahmin etmek imkânsızdır; çünkü uzun cümlelerin çoğu eğitim metninde hiç geçmez. N-gram modelleri, Markov varsayımı ile her kelimenin yalnızca önceki n − 1 keli-
Karmaşıklık uygulamada çarpımla değil, logaritmaların ortalamasıyla hesaplanır; böylece çok küçük sayıların çarpımının sıfıra yuvarlanması (underflow) önlenir:
Parantez içindeki nicelik, tam olarak token başına ortalama çapraz entropi kaybı L̄’dir. Yani karmaşıklık, sinir ağı dil modellerinin eğitimde en küçüklediği kaybın üstel hâlidir: Token başına kaybı 2
N-gram modellerinin temel sorunları; uzun bağlamı yakalayamamaları (n büyüdükçe sayımlar seyrekleşir), kelimeler arası benzerliği kullanamamaları (“kedi süt içer” görüldüğünde “köpek süt içer” hakkında bir şey öğrenilmez) ve sözlük büyüdükçe tablo boyutunun patlamasıdır. Sinir ağı tabanlı dil modelleri bu sorunları gömmeler ve paylaşılan parametrelerle çözer.
Metin bir dizidir: Kelimelerin sırası anlam için belirleyicidir ve diziler farklı uzunlukta olabilir. Tekrarlayan sinir ağları (recurrent neural networks, RNN), diziyi adım adım işler ve o ana kadar okunan bilginin özetini bir gizli durum (hidden state) vektöründe taşır (Şekil 12.2):
Burada xt t. tokenın gömme vektörü, ht ise gizli durumdur. Aynı Wh , W x ve Wy ağırlıkları her zaman adımında kullanılır (zaman boyunca ağırlık paylaşımı). Dil modeli olarak kullanıldığında her adımda ŷt , bir sonraki tokenın olasılık dağılımıdır.
RNN’ler zaman boyunca açıldığında çok derin bir ağa dönüşür; geri yayılım her adımda Wh ile çarpılarak geriye aktığı için 7. haftadaki gradyan sönümlenmesi sorunu burada çok belirgindir. Sonuç olarak standart RNN’ler uzun cümlelerin başındaki bilgiyi “unutur”. Örneğin “Ali, sabah erkenden kalkıp kahvaltısını yaptıktan ve otobüse bindikten sonra okula vardığında … o” cümlesinde “o”nun Ali olduğunu hatırlamak zordur. LSTM (Long Short-Term Memory, 1997) ve onun sadeleştirilmiş sürümü GRU, gizli duruma ek olarak bir hücre durumu ve bu durumu kontrol eden kapılar kullanır: unutma kapısı hangi bilginin silineceğine, girdi kapısı hangi yeni bilginin yazılacağına, çıkış kapısı ise hücre durumunun ne kadarının dışarı verileceğine karar verir. Kapılar, sigmoid ile 0–1 arasında “ne kadar geçsin” oranları üretir. Hücre durumu toplama işlemleriyle güncellendiği için gradyan uzun mesafeler boyunca daha iyi korunur. 2014–2017 yılları arasında makine çevirisi, konuşma tanıma ve metin üretiminde LSTM’ler
xt girdi, ht−1 önceki gizli durum, ct−1 önceki hücre durumu olmak üzere bir LSTM hücresi şu hesapları yapar (⊙: eleman eleman çarpım):
Hücre durumu, eski bilginin %73’ünü korumuş ve yeni bilginin yarısını eklemiştir. ct güncellemesindeki toplama, gradyanın zaman boyunca çarpılmadan akmasını sağlayan “otoyol”dur;
Makine çevirisi gibi görevlerde girdi ve çıktı farklı uzunlukta dizilerdir. Diziden diziye (sequenceto-sequence, seq2seq) modeller iki RNN kullanır (Şekil 12.3): Kodlayıcı kaynak cümleyi okuyup son gizli durumunda sabit uzunlukta bir bağlam vektörü oluşturur; kod çözücü bu vektörden başlayarak hedef cümleyi kelime kelime üretir.
Seq2seq modelinde kaynak cümle ne kadar uzun olursa olsun tüm anlamı tek bir sabit uzunluklu vektöre sığdırılmak zorundadır. Bu darboğaz, uzun cümlelerde çeviri kalitesini belirgin biçimde düşürür. 2014’te Bahdanau ve arkadaşları, kod çözücünün her adımda kaynak cümlenin tüm gizli durumlarına bakıp hangisine ne kadar “dikkat” edeceğine karar vermesini önerdi. Bu dikkat mekanizması fikri, 2017’de RNN’leri tamamen ortadan kaldıran Transformer mi-
✓ NLP; belirsizlik, bağlam, zengin biçimbilim ve mecaz nedeniyle zordur. Türkçe eklemeli bir dil olduğundan çok sayıda kelime biçimi vardır. ✓ Tokenizasyon metni birimlere ayırır. Modern modeller alt kelime tokenizasyonu (BPE) kullanır; BPE en sık sembol çiftlerini art arda birleştirir.
✓ RNN gizli durumla diziyi adım adım işler; uzun bağımlılıklarda gradyan sönümlenmesi yaşar. LSTM/GRU kapılarla bu sorunu azaltır. ✓ Seq2seq modellerindeki sabit uzunluklu bağlam vektörü darboğazı, dikkat mekanizmasının ve Transformer’ın doğuşuna yol açmıştır.
A Kelime sırasını tamamen göz ardı etmesi B Eğitimde görülmemiş kelimeleri bilinen parçalarla temsil edebilmesi C Her zaman daha uzun diziler üretmesi D Durak kelimelerini otomatik silmesi E Gömme vektörlerine ihtiyaç duymaması
4. Bir dil modelinin temel görevi aşağıdakilerden hangisidir? A Metni gri tonlu görüntüye çevirmek B Önceki tokenlar verildiğinde sonraki tokenın olasılık dağılımını tahmin etmek C Belgeleri yalnızca uzunluklarına göre sıralamak D Kelimelerin kökünü bulmak
7. Standart RNN’lerin uzun cümlelerde başarısız olmasının temel nedeni nedir? A Parametrelerinin her zaman adımında farklı olması B Zaman boyunca geri yayılımda gradyanların sönümlenmesi
larını vermiştir. (a) Dizinin olasılığını ve karmaşıklığını hesaplayınız. (b) Aynı sonucu Denklem (12.5) ile, logaritmalar üzerinden elde ediniz. (c) İnce ayar sonrası modelin token başına ortalama kaybı 1.0 nata düşmüştür; yeni karmaşıklık nedir ve bu sayı nasıl yorumlanır?
LSTM’in parametre sayılarını hesaplayınız (bias dahil). (c) LSTM’in hücre durumu neden “gradyan otoyolu” olarak adlandırılır? Basit RNN’de gradyan T adım geriye giderken neyle
• sorgu, anahtar ve değer (Q, K, V) kavramlarını yorumlayabilecek ve ölçekli nokta çarpım dikkatini küçük matrislerle elle hesaplayabilecek, • öz-dikkat, çok başlı dikkat ve nedensel maskeleme kavramlarını açıklayabilecek, • konum kodlamasının neden gerekli olduğunu açıklayıp sinüzoidal konum kodlamasını
• bir Transformer bloğunun bileşenlerini sayabilecek ve parametre sayısını hesaplayabilecek, • kodlayıcı (BERT), kod çözücü (GPT) ve kodlayıcı–kod çözücü (T5) modellerini karşılaştırabilecek; Vision Transformer’ın görüntüleri nasıl işlediğini açıklayabilecek, • katman normalizasyonunu hesaplayabilecek, öz-dikkat ile RNN’in hesaplama maliyetlerini karşılaştırabilecek ve maskeli/nedensel dil modeli kayıplarını ayırt edebilecektir.
Geçen hafta, RNN tabanlı çeviri modellerinde kaynak cümlenin tüm anlamının tek bir sabit uzunluklu vektöre sıkıştırılmasının bir darboğaz yarattığını gördük. Bir insan çevirmen ise uzun bir cümleyi çevirirken her hedef kelimeyi yazarken kaynak cümlenin ilgili kısmına tekrar bakar. “school” kelimesini yazarken gözü “okula” kelimesindedir. Dikkat mekanizması (attention mechanism) bu davranışı modeller: Kod çözücü her adımda kaynak cümlenin tüm konumlarına bakar ve her birine ne kadar “dikkat” edeceğini belirleyen ağırlıklar hesaplar (Şekil 13.1).
Dikkat mekanizması bir yumuşak sözlük araması gibi düşünülebilir. Bir kütüphanede kitap aradığınızı varsayın: Aklınızdaki konu bir sorgudur (query); raflardaki kitapların sırtındaki etiketler anahtarlardır (key); kitapların içeriği ise değerlerdir (value). Sorgunuzu her etiketle karşılaştırırsınız; en çok eşleşen kitaplara daha fazla zaman ayırarak bilgilerini birleştirirsiniz. Klasik bir sözlükte yal-
nızca tam eşleşen anahtarın değeri döner; dikkat mekanizmasında ise tüm değerlerin, benzerlikle orantılı ağırlıklarla bir karışımı döner.
3. Softmax (her satıra ayrı): Skorlar, toplamı 1 olan dikkat ağırlıklarına dönüşür. 4. V ile çarpım: Her token için değer vektörlerinin ağırlıklı ortalaması alınır.
Birinci token, kendisine ve kendisine benzeyen üçüncü tokena daha fazla dikkat etmiştir. Her tokenın yeni temsili artık dizideki diğer tokenlardan gelen bilgiyi de içerir.
Seq2seq modellerinde dikkat, kod çözücü ile kodlayıcı arasında kullanılıyordu. Transformer’ın devrimci fikri, dikkati bir dizinin kendi içinde kullanmaktır: Öz-dikkatte (self-attention) her token, aynı cümledeki diğer tüm tokenlara bakarak kendi temsilini günceller. Sorgu, anahtar ve değer matrislerinin üçü de aynı girdi X’ten (token gömmeleri) öğrenilebilir ağırlık matrisleriyle elde edilir:
Öz-dikkat, kelimelerin bağlama göre anlam kazanmasını sağlar. “Hayvan caddeyi geçmedi çünkü o çok yorgundu” cümlesinde “o” zamirinin “hayvan”ı mı “cadde”yi mi işaret ettiği, öz-dikkatin “o” kelimesine verdiği ağırlıklardan anlaşılabilir (Şekil 13.3). Aynı şekilde “yüz” kelimesinin temsili, cümlede “lira” geçiyorsa sayıya, “yıkamak” geçiyorsa organa yaklaşır; yani öz-dikkat bağlamsal
dikkat edilen kelime (anahtar) Şekil 13.3 Bir öz-dikkat başlığının ağırlıkları (temsili). Turuncu çerçeveli satır, “o” kelimesinin sorgusunu gösterir: En yüksek dikkat “Hayvan” kelimesine verilmiştir.
RNN’lerle karşılaştırıldığında öz-dikkatin iki büyük avantajı vardır: (i) Cümlenin başındaki ve sonundaki iki kelime arasındaki bağlantı tek adımda kurulur; bilgi onlarca adım boyunca taşınmak zorunda değildir. (ii) Tüm tokenlar için hesaplama aynı anda, matris çarpımlarıyla paralel yapılır;
katte (multi-head attention) h adet dikkat “başı”, kendi WQ , WK , WV matrisleriyle paralel olarak çalışır. Bir baş özne–yüklem ilişkisine, diğeri zamirlerin gönderdiği kelimelere, bir diğeri yan yana kelimelere odaklanabilir. Başların çıktıları birleştirilir (concatenate) ve bir çıkış matrisi WO ile
Öz-dikkat işlemi, tokenların sırasından habersizdir: “Ali Ayşe’yi aradı” ile “Ayşe Ali’yi aradı” cümlelerindeki kelimeler aynı olduğu için dikkat hesabı (konum bilgisi eklenmezse) aynı sonucu verir. Bu nedenle her token gömmesine, konumunu belirten bir konum kodlaması (positional encoding)
Düşük boyutlar hızlı, yüksek boyutlar yavaş salınır; böylece her konum benzersiz bir “parmak izine” sahip olur (Şekil 13.4). Günümüzün büyük dil modellerinde öğrenilebilir konum gömmeleri veya döndürmeli konum kodlaması (RoPE) gibi gelişmiş yöntemler de kullanılmaktadır.
İlk iki boyut konumdan konuma belirgin biçimde değişirken son iki boyut çok yavaş değişir. Bu vektörler token gömmeleriyle toplanarak modele verilir.
Vaswani ve arkadaşlarının 2017’de yayımladığı “Attention Is All You Need” makalesi, RNN ve evrişim kullanmadan yalnızca dikkat mekanizmasına dayanan Transformer (Transformer) mimarisini tanıttı. Bir Transformer, aynı yapıdaki blokların art arda dizilmesiyle oluşur (Şekil 13.5). Her blokta
Her alt katmanın etrafında ResNet’ten tanıdığımız bir artık bağlantı ve ardından (ya da öncesinde) katman normalizasyonu bulunur: x ← LayerNorm(x + AltKatman(x)). Artık bağlantılar sayesinde onlarca hatta yüzlerce blok üst üste konulabilir.
7. haftadaki toplu normalizasyon bir özniteliği yığındaki örnekler üzerinden normalleştirir; dizi uzunlukları değişken olduğunda ve küçük yığınlarda bu kararsızdır. Katman normalizasyonu (layer normalization) bunun yerine tek bir tokenın dmodel boyutlu vektörünü kendi elemanları üzerinden nor-
eğitim ve çıkarımda aynı biçimde çalışır. Orijinal Transformer normalizasyonu artık toplamından sonra uygular (post-LN); günümüz modellerinin çoğu, derin ağlarda daha kararlı eğitim sağladığı
dikkatin, n > d iken RNN’in daha az işlem yapmasıdır; dikkatin n2 maliyeti, uzun belgeler ve yüksek çözünürlüklü görüntüler için ana darboğazdır ve seyrek dikkat, FlashAttention gibi yöntemler bu maliyeti düşürmeye çalışır. Bilgi yolunun O(1) olması da dikkatin asıl üstünlüğüdür: Dizinin başındaki bir token sondakiyle tek adımda ilişkilenir, oysa RNN’de bilgi n adım boyunca taşınmak zorundadır. Metin üretirken her yeni token için önceki tokenların K ve V matrisleri yeniden hesaplanmaz, bellekte saklanır (KV önbelleği); bunun bellek maliyetini 14. haftada hesaplayacağız.
Orijinal Transformer, makine çevirisi için bir kodlayıcı ve bir kod çözücüden oluşuyordu. Kod çözücü, metni soldan sağa kelime kelime üretirken henüz üretilmemiş gelecekteki kelimelere bakmamalıdır; aksi hâlde eğitimde cevabı “kopya çeker”. Bunun için dikkat skorlarına bir nedensel maske (causal mask) uygulanır: Gelecekteki konumların skorları −∞ yapılır ve softmax sonrasında
önceki tokenları görebildiğinden değişmez. Kod çözücüde ayrıca çapraz dikkat (cross-attention) katmanı bulunur: Sorgular kod çözücüden, anahtar ve değerler kodlayıcının çıktısından gelir. Böylece çeviri sırasında hedef cümle kaynak cümleye “bakar”; bu, 12. haftadaki Bahdanau dikkatinin Transformer’daki karşılığıdır.
BERT’in ön eğitiminde “Bugün hava çok [MASK]” girdisi için model, maskeli konumda sözlükteki tüm tokenlar üzerinde bir dağılım üretir. Basitlik için sözlük üç tokendan oluşsun ve
konumlarda hesaplanır; seçilenlerin %80’i [MASK] ile, %10’u rastgele bir tokenla değiştirilir, %10’u olduğu gibi bırakılır (böylece model, [MASK] görmediği konumları da temsil etmeyi
hepsi tek bir ileri geçişte paralel olarak hesaplanır ve kayıp tüm konumların ortalamasıdır. Her tokendan sinyal alındığı için üretici modellerin ön eğitimi daha verimlidir; buna karşılık BERT her konumda iki yönlü bağlam kullanabilir.
BERT’in Türkçe metinlerle ön eğitilmiş sürümü BERTurk, Türkçe duygu analizi, haber sınıflandırma ve varlık tanıma gibi görevlerde yaygın olarak kullanılan açık kaynaklı bir modeldir. Kodlayıcı modeller, metin üretmeyen “anlama” görevlerinde büyük üretici modellere göre çok
Transformer’ın başarısı yalnızca dille sınırlı kalmadı. 2020’de önerilen Vision Transformer (ViT), bir görüntüyü kelimelerden oluşan bir cümle gibi ele alır (Şekil 13.7): Görüntü sabit boyutlu yamalara (patch) bölünür; her yama düzleştirilip doğrusal bir katmanla gömme vektörüne dönüştürülür; konum kodlaması eklenir ve dizinin başına sınıflandırma için özel bir [CLS] tokenı konur. Bu dizi
oluşur ve dikkat matrisi yaklaşık 16 kat büyür: Dizi uzunluğunun karesiyle büyüme, Transformer’ların temel hesaplama sınırlamasıdır.
ViT’ler CNN’lerdeki yerel bağlantı ve öteleme eşdeğerliği gibi “hazır” varsayımlara (tümevarımsal yanlılık) sahip olmadığı için küçük veri setlerinde CNN’lerden daha kötü performans gösterebilir; ancak çok büyük veriyle önceden eğitildiklerinde CNN’leri geride bırakırlar. Günümüzde ViT tabanlı modeller sınıflandırma, segmentasyon (SAM), tıbbi görüntü analizi ve metin–görüntü modellerinde (CLIP) yaygın olarak kullanılmaktadır.
tokenlara tek adımda bakar ve bağlamsal temsil kazanır. ✓ Çok başlı dikkat farklı ilişki türlerini paralel öğrenir. Öz-dikkat sırayı bilmediği için konum
✓ Transformer bloğu: çok başlı dikkat + FFN, her birinde artık bağlantı ve katman normalizasyonu. Kod çözücüde nedensel maske gelecekteki tokenları gizler. ✓ BERT yalnızca kodlayıcı (maskeli dil modeli), GPT yalnızca kod çözücü (sonraki token tahmini), T5 kodlayıcı–kod çözücüdür. ✓ ViT görüntüyü yamalara bölerek Transformer’a verir; dikkatin maliyeti dizi uzunluğunun
✓ Katman normalizasyonu tek bir tokenın vektörünü kendi elemanları üzerinden standartlaştırır; pre-LN düzeni derin modellerde daha kararlıdır.
rumda paralel çalışır ve tokenlar arası bilgi yolu O(1)’dir. BERT kaybı yalnızca maskeli konumlarda, GPT kaybı her konumda hesaplanır.
A Dikkat ağırlıklarının toplamını 1 yapmak B Büyük boyutlarda nokta çarpımların büyüyüp softmax’ı aşırı sivriltmesini ve gradyanların küçülmesini önlemek C Parametre sayısını azaltmak
2. Öz-dikkatte sorgu, anahtar ve değer matrisleri nereden elde edilir? A Üçü de farklı veri setlerinden B Üçü de aynı girdi dizisinden, farklı öğrenilebilir ağırlık matrisleriyle C Sorgu kodlayıcıdan, anahtar ve değer kod çözücüden D Rastgele üretilir ve eğitilmez
4. Transformer’da konum kodlamasına neden ihtiyaç duyulur? A Öz-dikkat işlemi tokenların sırasından bağımsız olduğu için B Parametre sayısını azaltmak için C Softmax’ı hızlandırmak için D Sözlük boyutunu küçültmek için E Gradyan patlamasını önlemek için
5. GPT gibi metin üreten modellerde nedensel maskenin işlevi nedir? A Durak kelimeleri silmek B Her tokenın yalnızca kendisine ve önceki tokenlara dikkat etmesini sağlamak C Rastgele kelimeleri gizleyip tahmin ettirmek D Görüntüyü yamalara bölmek E Başların sayısını belirlemek
toplam parametre sayısını (dikkat + FFN + iki katman normalizasyonu) ve 12 bloğun toplamını bulunuz. (b) Token gömme matrisi ve konum gömme matrisinin parametre sayılarını ekleyerek modelin toplam parametre sayısını tahmin ediniz (çıkış katmanı gömme matrisini
Bu haftanın sonunda öğrenciler; • büyük dil modellerinin sonraki token tahmini ilkesiyle nasıl eğitildiğini ve metin ürettiğini
• sıcaklık, top-k ve top-p örnekleme yöntemlerini sayısal örneklerle uygulayabilecek, • ön eğitim, talimat ince ayarı ve insan geri bildirimli pekiştirmeli öğrenme (RLHF) aşama-
• temel istem mühendisliği tekniklerini kullanabilecek, • halüsinasyon gibi sınırlılıkları ve bunlara karşı geri getirme destekli üretim (RAG) yakla-
• çıkarım maliyetini ve KV önbelleğinin boyutunu hesaplayabilecek; RLHF/DPO amaç fonksiyonlarını, uzman karışımı mimarisini ve LLM değerlendirme yöntemlerini açıklayabile-
Büyük dil modeli (large language model, LLM), çok büyük metin derlemleri üzerinde sonraki tokenı tahmin etme göreviyle eğitilmiş, milyarlarca parametreye sahip, genellikle yalnızca kod çözücüden oluşan bir Transformer modelidir.
GPT, Claude, Gemini ve Llama gibi modeller bu tanıma uyar. Önceki iki haftada gördüğümüz parçalar burada bir araya gelir: Metin alt kelime tokenlarına ayrılır (12. hafta), tokenlar gömme vektörlerine dönüştürülür, nedensel maskeli Transformer blokları (13. hafta) bu vektörleri işler ve
Günümüzün açık modelleri birkaç milyar ile yüzlerce milyar parametre arasındadır. • Veri: Ön eğitimde web sayfaları, kitaplar, makaleler, kod depoları gibi kaynaklardan derlenmiş binlerce milyar (trilyonlarca) token kullanılır. • Hesaplama: Eğitim, binlerce GPU üzerinde haftalar ila aylar sürer.
Ön eğitimin amacı şaşırtıcı derecede basittir: Bir metin parçasındaki her konumda, o ana kadarki tokenlara bakarak bir sonraki tokenı tahmin etmek. Kayıp, 7. haftadaki kategorik çapraz entropidir ve tüm konumlar üzerinden ortalaması alınır:
Bu bir öz-denetimli öğrenmedir (2. hafta): Etiketler (sonraki tokenlar) metnin kendisinde bulunduğundan insan etiketlemesine gerek yoktur. Bu yüzden internet ölçeğindeki metinlerin tamamı eğitim verisi olarak kullanılabilir.
“Türkiye’nin başkenti” bağlamından sonra model “Ankara” tokenına 0.62, “İstanbul” tokenına 0.05 olasılık vermektedir. Metindeki gerçek sonraki token “Ankara” ise bu konumdaki kayıp nedir? Model “İstanbul”a 0.62 olasılık vermiş olsaydı kayıp ne olurdu?
konumda bu kaybı küçültmek, modeli dilbilgisini, olguları, akıl yürütme örüntülerini ve hatta kod yazmayı “öğrenmeye” zorlar; çünkü bunlar sonraki kelimeyi iyi tahmin etmenin en etkili
OpenAI araştırmacıları 2020’de önemli bir gözlem yayımladı: Bir dil modelinin test kaybı, parametre sayısı (N), eğitim verisi (D) ve hesaplama miktarı (C) arttıkça öngörülebilir biçimde, bir kuvvet yasasıyla azalır (Şekil 14.1). Bu ölçekleme yasaları (scaling laws), daha büyük modellere yapılan yatırımların bilimsel gerekçesi oldu. 2022’de DeepMind’ın “Chinchilla” çalışması, belirli bir hesaplama bütçesinde en iyi sonuç için model büyüklüğü ile veri miktarının birlikte artırılması gerektiğini gösterdi; kabaca parametre başına yaklaşık 20 token kullanılmalıdır. Bir Transformer’ı eğit-
Model ve veri büyüdükçe, küçük modellerde hiç görülmeyen bazı yeteneklerin (çok adımlı aritmetik, birkaç örnekten yeni bir görevi öğrenme, kod yazma) belirli bir ölçekten sonra ortaya çıktığı gözlenmiştir. Bu yeteneklerin gerçekten “ani” mi ortaya çıktığı, yoksa ölçüm yöntemlerinden mi kaynaklandığı araştırmacılar arasında tartışma konusudur.
Bir LLM metni özbağlanımlı (autoregressive) olarak üretir: Model girdi için sonraki tokenın olasılık dağılımını hesaplar, bu dağılımdan bir token seçilir, seçilen token girdinin sonuna eklenir ve işlem bir bitiş tokenı üretilene veya uzunluk sınırına ulaşılana kadar tekrarlanır (Şekil 14.2).
Her adımda dağılımdan hangi tokenın seçileceğini kod çözme (decoding) stratejisi belirler: • Açgözlü seçim (greedy): Her zaman en olası token seçilir. Belirleyicidir ama tekrara düşmeye ve sıkıcı metinlere yatkındır.
Özbağlanımlı metin üretimi. Her adımda model sonraki tokenın olasılık dağılımını hesaplar ve bu dağılımdan bir token örneklenir; üretilen token (turuncu) girdinin sonuna eklenir ve model yeniden çalıştırılır.
• Top-p (çekirdek) örnekleme: Olasılıklar büyükten küçüğe sıralanır ve toplam olasılığı p’ye ulaşan en küçük token kümesi içinden örnekleme yapılır. Model kendinden eminse küme küçük,
Bir LLM’in tek seferde işleyebildiği en fazla token sayısına bağlam penceresi (context window) denir. Bu sınır, istemi (prompt), konuşma geçmişini, eklenen belgeleri ve üretilen cevabı birlikte kapsar. Öz-dikkatin maliyeti dizi uzunluğunun karesiyle arttığı için uzun bağlam pahalıdır. İlk GPT modellerinde birkaç bin token olan bağlam pencereleri, günümüz modellerinde yüz binlerce hatta milyonlarca tokena ulaşmıştır. Ticari LLM hizmetleri de genellikle işlenen token sayısı üzerinden ücretlendirilir. Türkçe gibi eklemeli dillerde aynı içerik İngilizceye göre çoğu zaman daha fazla to-
lıkları en fazla 70 kez okuyabilir, yani üst sınır yaklaşık 70 token/s’dir. Çıkarım bu yüzden bellek bant genişliği sınırlıdır; birden çok isteği aynı anda işlemek (batching) ve ağırlıkları 8 veya 4 bite
Önbellek, bağlam uzunluğuyla doğrusal büyür ve eşzamanlı her istek için ayrıdır: 10 kullanıcı aynı anda uzun belgelerle çalışırsa yalnızca önbellek 20 GB bellek ister. Uzun bağlam pencerelerinin asıl maliyeti budur; modern modeller bu yüzden başlar arasında K ve V’yi paylaşan gruplanmış sorgu dikkati (GQA) gibi yöntemlerle önbelleği küçültür.
Yalnızca ön eğitimden geçmiş bir model (temel model) metni devam ettirmeyi bilir, ancak talimatları izlemeyi bilmez. “Türkiye’nin başkenti neresidir?” sorusuna cevap vermek yerine benzer sorulardan oluşan bir liste üretmeye devam edebilir. Kullanışlı ve güvenli bir asistan elde etmek için ek
• Talimat ince ayarı (instruction tuning, supervised fine-tuning, SFT): Model, insanlar tarafından yazılmış yüksek kaliteli “talimat–cevap” çiftleriyle denetimli olarak eğitilir. Böylece soruları yanıtlamayı, özet çıkarmayı, kod yazmayı ve talimatların biçimine uymayı öğrenir. • İnsan geri bildirimli pekiştirmeli öğrenme (reinforcement learning from human feedback, RLHF): Aynı soruya verilen birkaç cevap insanlar tarafından iyiden kötüye sıralanır. Bu tercihlerden, bir cevabın ne kadar iyi olduğunu puanlayan bir ödül modeli eğitilir. Ardından dil modeli, ödül modelinden yüksek puan alacak cevaplar üretmek üzere pekiştirmeli öğrenmeyle (2. hafta) güncellenir. DPO (Direct Preference Optimization) gibi yöntemler, ayrı bir ödül modeli eğitmeden tercih verisini doğrudan kullanır. Yapay zekâ geri bildirimiyle ya da yazılı ilkeler kümesiyle hizalama yapan yaklaşımlar da geliştirilmiştir.
KL terimi olmadan model, ödül modelinin zayıflıklarını sömüren anlamsız ama yüksek puanlı metinler üretmeye başlar (ödül hilesi). Bu amaç fonksiyonunun en iyi çözümünün π ∗ (y |
Sezgisi basittir: Tercih edilen cevabın olasılığını referansa göre artır, edilmeyeninkini azalt. DPO denetimli bir kayıp gibi doğrudan tercih verisiyle eğitildiğinden RLHF’ten çok daha basit ve kararlıdır; açık kaynaklı modellerin çoğu bu ya da benzeri yöntemlerle hizalanır.
Bu aşamaların amacı modelin yardımcı, dürüst ve zararsız olmasını sağlamaktır; bu genel hedefe hizalama (alignment) denir.
LLM’lerin en önemli özelliklerinden biri, ağırlıkları değiştirilmeden yalnızca girdideki talimat ve örneklerle yeni görevlere uyum sağlayabilmeleridir. Buna bağlam içi öğrenme (in-context learning) denir. Modele verilen girdiyi (istemi) etkili biçimde tasarlamaya istem mühendisliği (prompt engi-
• Sıfır örnekli (zero-shot): Görev yalnızca tarif edilir. “Aşağıdaki yorumun duygusunu olumlu/olumsuz olarak sınıflandır: …” • Az örnekli (few-shot): İstemde birkaç çözülmüş örnek verilir; model örüntüyü yakalar. • Düşünce zinciri (chain-of-thought): Modelden cevaba geçmeden önce adım adım akıl yürütmesi istenir; çok adımlı matematik ve mantık problemlerinde doğruluğu belirgin biçimde artırır. Güncel “akıl yürütme modelleri” bu süreci eğitim yoluyla içselleştirmiştir. • Rol ve bağlam: Modelin rolü, hedef kitle, üslup ve kısıtlar açıkça belirtilir (sistem istemi). • Yapılandırılmış çıktı: Cevabın biçimi (tablo, madde listesi, JSON) açıkça tanımlanır; bu, LLM çıktısını programlarda kullanmayı kolaylaştırır.
Sen bir e-ticaret sitesinin müşteri yorumlarını inceleyen bir asistansın. Her yorumu "olumlu", "olumsuz" veya "nötr" olarak sınıflandır ve
LLM’ler bazen akıcı ve kendinden emin bir dille yanlış bilgi üretir: var olmayan makaleler, hatalı tarihler, uydurma kaynakça. Buna halüsinasyon (hallucination) denir. Nedeni, modelin “doğru” olanı değil “olası” olanı üretmek üzere eğitilmiş olmasıdır. Bu nedenle LLM çıktıları, özellikle sağlık, hukuk, finans ve akademik çalışmalarda mutlaka doğrulanmalıdır. Ödevlerinizde LLM kullandıysanız, ürettiği her kaynağı kendiniz kontrol etmelisiniz.
• Bilgi kesim tarihi: Model, eğitim verisinin toplandığı tarihten sonraki olayları bilmez (araç kullanımı veya arama ile desteklenmedikçe). • Önyargı: Eğitim verisindeki toplumsal önyargılar modele yansıyabilir (1. hafta). • Güvenlik: Kötü niyetli istemlerle modelin kurallarını aşmaya çalışma (jailbreak) ve dış belgelerin içine gizlenmiş talimatlarla modeli yönlendirme (prompt injection) önemli güvenlik riskler-
• Gizlilik ve telif: Kişisel verilerin ve telifli içeriğin eğitimde ve kullanımda nasıl ele alındığı hukuki ve etik bir sorudur. • Maliyet ve enerji: Büyük modellerin eğitimi ve çalıştırılması yüksek enerji ve donanım maliyeti
Halüsinasyonu azaltmanın ve modele güncel ya da kuruma özel bilgi kazandırmanın en yaygın yolu geri getirme destekli üretimdir (retrieval-augmented generation, RAG). Fikir şudur: Modelin “ezberine” güvenmek yerine, soruyla ilgili belgeleri bir bilgi tabanından bulup isteme eklemek ve modelden yalnızca bu belgelere dayanarak cevap vermesini istemek (Şekil 14.5). Bu, kapalı kitap sınavını açık kitap sınavına çevirmeye benzer.
En benzer parça sınav yönetmeliğidir. İstem şöyle kurulur: “Aşağıdaki yönetmelik metnine dayanarak soruyu cevapla; metinde yoksa bilmediğini söyle. [yönetmelik parçası] Soru: Bütünleme sınavına kimler girebilir?”
Bir LLM’i belirli bir alana (ör. hukuk metinleri, tıbbi raporlar, kurum içi yazışma üslubu) uyarlamak için ince ayar yapılabilir. Ancak milyarlarca parametrenin hepsini güncellemek (tam ince ayar) çok büyük GPU belleği gerektirir: Her parametre için ağırlığın yanında gradyan ve Adam optimizasyon algoritmasının iki moment değeri de saklanmalıdır. Bu nedenle yalnızca küçük bir parametre kümesini eğiten parametre-verimli ince ayar (parameter-eﬀicient fine-tuning, PEFT) yöntemleri geliştirilmiştir. Bunların en yaygını LoRA’dır.
Yalnızca A ve B eğitilir. r rank, α ise ölçekleme katsayısıdır. B sıfırla başlatıldığı için eğitimin başında model orijinal modelle aynıdır.
nıyor. (a) Tam ince ayarda ve LoRA’da bu matris için eğitilecek parametre sayılarını karşılaştırınız. (b) 7 milyar parametreli bir modelin ağırlıkları 16 bitlik (2 bayt) sayılarla kaç GB yer kaplar? 4 bitlik nicemlemede ne olur?
Model 4 bitte dondurulup üzerine LoRA uygulanmasına QLoRA denir; bu sayede milyarlarca parametreli modellere tek bir tüketici GPU’sunda bile ince ayar yapılabilir.
LoRA’nın pratik avantajları şunlardır: (i) Eğitim belleği ve süresi büyük ölçüde azalır. (ii) Her görev için yalnızca birkaç megabaytlık A ve B matrisleri saklanır; aynı temel model, farklı görevler için farklı LoRA “adaptörleriyle” kullanılabilir. (iii) Eğitim sonrasında BA çarpımı W0 ’a eklenerek birleştirilebilir; böylece çıkarım sırasında ek gecikme oluşmaz. LoRA yalnızca dil modellerinde değil, görüntü sınıflandırmada kullanılan ViT modellerinde ve metinden görüntü üreten difüzyon modellerinde de yaygın olarak kullanılmaktadır.
• Görev yalnızca talimatla tarif edilebiliyorsa: istem mühendisliği (en ucuz, en hızlı). • Model güncel veya kuruma özel bilgiye ihtiyaç duyuyorsa: RAG. • Model belirli bir davranışı, üslubu veya çıktı biçimini tutarlı olarak öğrenmeliyse: ince ayar
Uzman karışımı (Mixture of Experts, MoE), Transformer bloğundaki tek FFN’i birbirine paralel E adet FFN “uzmanı” ile değiştirir; küçük bir yönlendirici (router) ağ her token için bu uzmanlardan
toplam parametre sayısı çok büyürken her tokenın etkin olarak kullandığı parametre (ve işlem) sayısı küçük kalır. Örneğin bloktaki FFN 2.1 milyon parametreliyse 8 uzmanlı bir MoE bloğu 16.8 milyon
Tüm uzmanlar bellekte tutulmak zorundadır ve yönlendiricinin tokenları dengeli dağıtması için ek bir yük dengeleme kaybı gerekir. Mixtral, DeepSeek-V3 ve birçok güncel büyük model bu mimariyi
Değerlendirme. LLM’lerin başarımı, standart soru kümeleri (benchmark) üzerinde ölçülür: çok alanlı çoktan seçmeli bilgi soruları (MMLU), ilkokul düzeyi matematik problemleri (GSM8K), programlama görevleri (HumanEval) ve daha zor güncel kümeler. Üç sorun bilinmelidir: (i) Veri kirlenmesi: Soru kümesi modelin eğitim verisine sızmışsa skor ezberi ölçer; (ii) üretici görevlerde (özetleme, sohbet) tek doğru cevap olmadığından ya insan değerlendiricilere ya da bir başka LLM’in “hakem” olarak kullanıldığı otomatik puanlamaya başvurulur; (iii) kullanıcıların iki anonim cevaptan birini seçtiği karşılaştırmalı arenalar (Elo puanı) gerçek kullanım kalitesini daha iyi yansıtır ama manipülasyona açıktır. Türkçe için de çeviri ve özgün soru kümeleri geliştirilmektedir; bir model için “en iyi” kararı, hedef görevde kendi verinizle yapılan değerlendirmeyle verilmelidir.
Güncel modeller yalnızca metinle sınırlı değildir. Çok kipli (multimodal) modeller görüntü, ses ve videoyu da girdi olarak alabilir: Görüntüler ViT benzeri bir kodlayıcıyla tokenlara dönüştürülür ve metin tokenlarıyla aynı Transformer’da işlenir. Bir röntgen görüntüsünü açıklamak, bir grafikteki verileri okumak veya bir el çizimini koda dönüştürmek bu sayede mümkün olur. Yapay zekâ ajanları ise LLM’i 1. haftada gördüğümüz ajan çerçevesine yerleştirir: Model bir hedefe ulaşmak için plan yapar; arama motoru, hesap makinesi, kod çalıştırma ortamı veya veritabanı gibi araçları çağırır; sonuçları gözlemler ve bir sonraki adıma karar verir. Bu yaklaşım, LLM’leri yalnızca soru cevaplayan sistemlerden çok adımlı işleri yürütebilen sistemlere dönüştürmektedir. Ajanların güvenilirliği, denetimi ve güvenliği ise günümüzün en aktif araştırma konularındandır.
✓ LLM, büyük metin derlemlerinde sonraki token tahminiyle (öz-denetimli) eğitilmiş, yalnızca kod çözücülü bir Transformer’dır; kayıp, doğru tokenın negatif log-olasılığıdır. ✓ Ölçekleme yasaları: Kayıp N, D ve C ile kuvvet yasasına göre azalır; Chinchilla kuralı pa-
✓ İstem mühendisliği: sıfır/az örnekli istem, düşünce zinciri, rol ve yapılandırılmış çıktı. ✓ Halüsinasyon temel bir risktir; RAG, ilgili belgeleri kosinüs benzerliğiyle bulup isteme ekleyerek cevapları kaynaklara dayandırır.
önbelleği token başına 2Ldmodel sayı tutar ve bağlam uzunluğuyla doğrusal büyür. ✓ Ödül modeli Bradley–Terry ile tercihlerden öğrenilir; RLHF, KL cezalı ödül en büyüklemesidir; DPO aynı amacı doğrudan tercih çiftleriyle çözer. MoE, her token için uzmanların yalnızca bir kısmını çalıştırarak toplam parametreyi büyütürken işlem maliyetini sabit tu-
1. Bir büyük dil modelinin ön eğitim görevi aşağıdakilerden hangisidir? A Görüntüleri sınıflandırmak B Önceki tokenlar verildiğinde sonraki tokenı tahmin etmek C İnsan tercihlerine göre cevapları sıralamak D Belgeleri kümelere ayırmak E Soruları veritabanında aramak
A Dağılım düzleşir, metin daha rastgele olur B Dağılım sivrileşir, model en olası tokenları daha çok seçer C Bağlam penceresi büyür D Modelin parametre sayısı azalır E Halüsinasyon tamamen ortadan kalkar
5. Bir şirket, çalışanların kurum içi yönetmeliklerle ilgili sorularını güncel belgelere dayanarak cevaplayan bir asistan istiyor. En uygun yaklaşım hangisidir? A Modeli sıfırdan eğitmek
7. Aşağıdakilerden hangisi “halüsinasyon”u en iyi tanımlar? A Modelin aşırı yavaş çalışması B Modelin akıcı ve kendinden emin bir dille gerçeğe aykırı bilgi üretmesi C Modelin bağlam penceresinin dolması D Modelin aynı cevabı tekrar etmesi E Modelin talimatları reddetmesi
9. 13 milyar parametreli bir model için Chinchilla kuralına göre gereken token sayısını ve yaklaşık eğitim hesaplamasını (FLOP) bulunuz.
10. İstem mühendisliği, RAG ve LoRA ile ince ayar yaklaşımlarını hangi durumlarda tercih edeceğinizi birer örnekle açıklayınız.
düşününüz. (a) Token başına KV önbelleği kaç bayttır? 8192 tokenlık bir bağlamda toplam önbellek kaç GB olur? (b) 200 tokenlık bir cevap üretmek yaklaşık kaç FLOP ister? (c) Ağırlıklar 16 bitten 4 bite nicemlenirse model boyutu ve bant genişliğiyle sınırlı üretim hızı nasıl değişir?
miştir. Bradley–Terry olasılığını ve kaybı hesaplayınız; gradyan adımı iki puanı hangi yönde değiştirir? (b) RLHF’te KL cezası kaldırılırsa ne olması beklenir? (c) 50 milyon dikkat para-
Bu ekte, ders boyunca kullanılan temel matematik kavramları kısaca hatırlatılmaktadır. Konuların ayrıntılı anlatımı için doğrusal cebir, analiz ve olasılık ders kitaplarına başvurulabilir.
bir kelimenin gömmesi veya bir nöronun ağırlıkları vektörlerle temsil edilir. Aksi belirtilmedikçe vektörler sütun vektörü olarak düşünülür.
zincir kuralının sistematik bir uygulamasıdır. Kısmi türev ve gradyan: Çok değişkenli bir fonksiyonda kısmi türev, diğer değişkenler sabit tu-
Bu ek, sınavlara hazırlanırken hızlı tekrar için derste kullanılan başlıca formülleri haftalara göre özetler. Formüllerin anlamı ve kullanımı için ilgili haftalara başvurunuz.
Aşağıdaki sözlük, notlarda kalın yazılarak tanıtılan terimleri ve İngilizce karşılıklarını alfabetik sırayla vermektedir. Parantez içindeki sayı, terimin ilk tanıtıldığı haftayı gösterir.
çekirdek hilesi kernel trick (H4) aktivasyon fonksiyonu activation function (H6) çoklu doğrusal regresyon multiple linear regression
enterpolasyon interpolation (H8) ikili çapraz entropi binary cross-entropy (H4) en küçük kareler least squares (H3) ikili sınıflandırma binary classification (H4)
one-hot kodlama one-hot encoding (H2) siluet skoru silhouette score (H5) optimizasyon algoritması optimizer (H7) Sobel operatörü Sobel operator (H8) ortalama kare hata (MSE) mean squared error, MSE
ortalama kesinlik average precision, AP (H5) ortalama kesinlik ortalaması mean Average Precision, stokastik gradyan iniş stochastic gradient descent,
transpoze evrişim transposed convolution (H11) yerel bağlantı local connectivity (H9) Turing testi Turing test (H1) yığın boyutu batch size (H7)
Bu ekte her haftanın sonundaki değerlendirme sorularının cevapları verilmiştir. Çoktan seçmeli sorular için doğru şık ve kısa bir gerekçe, hesaplama soruları için çözümün ana adımları yer almaktadır. Cevaplara bakmadan önce soruları kendiniz çözmeye çalışınız.
kayıtları; P: önerilen ürünlerin tıklanma/satın 4. C – Başarımı ölçen nicelik performans ölçütüdür; alınma oranı. Yalnızca tıklama oranını en büyüketiketli e-postalar deneyimdir. lemek, dikkat çekici ama düşük kaliteli ürün-
GPU’ların sağladığı paralel hesaplama gücü; (3) ReLU, dropout, iyi ağırlık başlatma ve gelişmiş Hafta 2 · Makine Öğrenmesinin Temelleri optimizasyon yöntemleri gibi algoritmik ilerle-
10. Modelin eğitim verisindeki önyargıları öğrenerek belirli gruplar aleyhine sistematik hatalı kararlar 3. C – Ajan, ödül/ceza sinyaliyle deneme–yanılma
risini dengeli ve temsil edici hâle getirmek; modelin başarımını farklı gruplar için ayrı ayrı ölçmek 6. B – Test verisinin istatistikleri eğitim sürecine ka-
dağılım değiştirilirse raporlanan başarım gerçeği yansıtmaz. Ayrıca sentetik örnekler eğitim bilgisi taşıdığından test kümesine sızmaları veri sızıntısı
liğinden etkilenmez ve doğrudan değerlendirme Model her adımda ortalama yaklaşık 2.7 eşit olası metriğini (Dice) en büyük yapmaya çalışır. seçenek arasında kararsızmış gibi davranır; dü-
Aşağıda, notların hazırlanmasında yararlanılan ve konuları derinleştirmek isteyen öğrencilere önerilen temel kitaplar ile her haftanın dayandığı özgün makaleler listelenmiştir. Makalelerin büyük çoğunluğuna arXiv veya yayıncı sayfalarından açık erişimle ulaşılabilir."""
open("ders_notlari.txt", "w", encoding="utf-8").write(NOTLAR)
print(f"ders_notlari.txt hazır: {len(NOTLAR):,} karakter, {NOTLAR.count(chr(10)) + 1} paragraf")

ders_notlari.txt hazır: 149,118 karakter, 528 paragraf


### Ders notlarıyla kendi BPE tokenizer'ımızı eğitmek ve şablonu uygulamak
Bu hücre bir önceki hücrenin yazdığı `ders_notlari.txt` dosyasını kullanır.

In [11]:
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
dosya = "ders_notlari.txt"           # yoksa küçük örnek metin yazılır
if not os.path.exists(dosya): open(dosya, "w").write("Parametreler güncellenir. " * 200)
tok = Tokenizer(models.BPE(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train([dosya], trainers.BpeTrainer(vocab_size=3000, show_progress=False,
          special_tokens=["<|im_start|>", "<|im_end|>", "<unk>"],
          initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
hf = PreTrainedTokenizerFast(tokenizer_object=tok, unk_token="<unk>")
hf.chat_template = ("{% for m in messages %}<|im_start|>{{ m['role'] }}\n"
    "{{ m['content'] }}<|im_end|>\n{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}")
metin = hf.apply_chat_template(mesajlar, tokenize=False,
                               add_generation_prompt=True)
print("elle kurduğumuzla aynı mı?", metin == chatml(mesajlar),
      "|", len(hf(metin)["input_ids"]), "token")
c = "Öğrenilebilir parametreler eğitim sırasında güncellenir."
print([hf.convert_tokens_to_string([t]) for t in hf.tokenize(c)])

elle kurduğumuzla aynı mı? True | 32 token
['Ö', 'ğren', 'ilebilir', ' parametreler', ' eğitim', ' sırasında', ' güncel', 'lenir', '.']


### Az örnekli istemi kodla kurmak (14.5)

In [12]:
ornekler = [("Kargo çok hızlıydı, ürün tam beklediğim gibi.", "olumlu",
             "hızlı kargo ve beklentiyi karşılayan ürün"),
            ("Ürün idare eder, fiyatına göre normal.", "nötr",
             "ne övgü ne şikâyet var")]
istem = ('Sen bir e-ticaret sitesinin müşteri yorumlarını inceleyen bir asistansın.\n'
         'Her yorumu "olumlu", "olumsuz" veya "nötr" olarak sınıflandır ve\n'
         'cevabı yalnızca {"duygu": ..., "gerekce": ...} biçiminde JSON olarak ver.\n')
for yorum, duygu, gerekce in ornekler:
    istem += f'\nYorum: "{yorum}"\nCevap: {{"duygu": "{duygu}", "gerekce": "{gerekce}"}}\n'
istem += '\nYorum: "İki gün kullandım, şarj aleti bozuldu."\nCevap:'
print(istem)

Sen bir e-ticaret sitesinin müşteri yorumlarını inceleyen bir asistansın.
Her yorumu "olumlu", "olumsuz" veya "nötr" olarak sınıflandır ve
cevabı yalnızca {"duygu": ..., "gerekce": ...} biçiminde JSON olarak ver.

Yorum: "Kargo çok hızlıydı, ürün tam beklediğim gibi."
Cevap: {"duygu": "olumlu", "gerekce": "hızlı kargo ve beklentiyi karşılayan ürün"}

Yorum: "Ürün idare eder, fiyatına göre normal."
Cevap: {"duygu": "nötr", "gerekce": "ne övgü ne şikâyet var"}

Yorum: "İki gün kullandım, şarj aleti bozuldu."
Cevap:


## P4 · Ders notlarıyla RAG: “notlara soru soran asistan”
Adım 1–2: Metni parçalara böl (burada her paragraf bir parça, ortalama ≈ 270 karakter), her parçayı TF-IDF vektörüne çevir. Türkçe ekler yüzünden kelime yerine **karakter 3–5'lileri** kullanıyoruz (“parametre”, “parametreler”, “parametrelerin” ortak parçalar paylaşır).

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
metin = open("ders_notlari.txt", encoding="utf-8").read()
P = [p for p in metin.split("\n") if len(p) > 80]   # parçalar
vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                      sublinear_tf=True).fit(P)
X = vek.transform(P)                                 # parça vektörleri
print(len(P), "parça | vektör boyutu:", X.shape[1])

528 parça | vektör boyutu: 35774


Adım 3: Soruyu aynı uzaya çevir, kosinüs benzerliğiyle en yakın parçaları getir.

In [14]:
def ara(soru, k=3):
    b = cosine_similarity(vek.transform([soru]), X)[0]
    return [(b[i], P[i]) for i in b.argsort()[::-1][:k]]

for soru in ["Dropout ne işe yarar?", "Veri sızıntısı nedir?",
             "LoRA neden daha az bellek kullanır?"]:
    skor, parca = ara(soru, k=1)[0]
    print(f"{soru:36s}{skor:.2f} | {parca[:44]}...")

Dropout ne işe yarar?               0.24 | Dropout (dropout), eğitim sırasında her iter...
Veri sızıntısı nedir?               0.32 | Test verisine ait herhangi bir bilginin eğit...
LoRA neden daha az bellek kullanır? 0.19 | mış gradyanı bulunuz. (c) Toplu normalizasyo...


Adım 4: Getirilen parçalarla istemi kur. Modelden **yalnızca bu parçalara dayanmasını** ve kaynak numarası vermesini istiyoruz.

In [15]:
import textwrap
def rag_istemi(soru, k=3):
    kaynak = "\n".join(f"[{i+1}] {p[:150]}..."
                        for i, (_, p) in enumerate(ara(soru, k)))
    return ("Aşağıdaki ders notu parçalarına dayanarak soruyu cevapla. "
            "Kaynak numarasını ver; parçalarda yoksa 'bilmiyorum' de.\n"
            f"{kaynak}\nSoru: {soru}\nCevap:")
istem = rag_istemi("Otsu yöntemi eşiği nasıl seçer?", k=2)
for satir in istem.split("\n"):
    print(textwrap.fill(satir, 92))

Aşağıdaki ders notu parçalarına dayanarak soruyu cevapla. Kaynak numarasını ver; parçalarda
yoksa 'bilmiyorum' de.
[1] ✓ Histogram ton dağılımını gösterir; histogram eşitleme kontrastı artırır. Eşikleme
ikili görüntü üretir; Otsu yöntemi eşiği otomatik seçer. ✓ Evrişim...
[2] Eşikleme, nesneyi arka plandan ayırmanın en basit yoludur ve 11. haftada göreceğimiz
segmentasyonun en temel biçimidir. Eşiği elle seçmek zordur. Otsu...
Soru: Otsu yöntemi eşiği nasıl seçer?
Cevap:


### Adım 5 (Colab, internet gerekir): istemi açık ağırlıklı küçük bir modele ver
`Qwen/Qwen2.5-0.5B-Instruct` (Apache 2.0, ≈ 1 GB) indirilir. GPU (T4) seçerseniz daha hızlı çalışır. Daha iyi arama için TF-IDF yerine çok dilli bir gömme modeli (ör. `intfloat/multilingual-e5-small`) deneyin.

In [ ]:
from transformers import pipeline
uret = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
soru = "Otsu yöntemi eşiği nasıl seçer?"
mesaj = [{"role": "user", "content": rag_istemi(soru)}]
cevap = uret(mesaj, max_new_tokens=150, do_sample=False)
print(cevap[0]["generated_text"][-1]["content"])

# İsteğe bağlı: anlamsal arama ile karşılaştırın
# !pip install -q sentence-transformers
# from sentence_transformers import SentenceTransformer
# gm = SentenceTransformer("intfloat/multilingual-e5-small")
# E = gm.encode(["passage: " + p for p in P], normalize_embeddings=True)
# q = gm.encode(["query: " + soru], normalize_embeddings=True)
# print(P[int((E @ q.T).argmax())][:200])

## P5 · LoRA / QLoRA ince ayar tarifi
Önce eğitilecek parametre sayısını gerçek modelin mimarisinde (meta cihazda) görelim: yalnızca dikkat projeksiyonlarına mı, tüm doğrusal katmanlara mı LoRA eklemeli?

In [16]:
from peft import LoraConfig, get_peft_model
dikkat = ["q_proj", "k_proj", "v_proj", "o_proj"]
for etiket, hedef in [("yalnızca dikkat", dikkat), ("tüm doğrusal", "all-linear")]:
    with torch.device("meta"):
        m = AutoModelForCausalLM.from_config(LlamaConfig(**cfg))
    ayar = LoraConfig(r=16, lora_alpha=32, target_modules=hedef,
                      task_type="CAUSAL_LM")
    pm = get_peft_model(m, ayar)
    egit, hepsi = pm.get_nb_trainable_parameters()
    print(f"{etiket:16s}{egit/1e6:6.1f} M  (%{100*egit/hepsi:.2f})")

yalnızca dikkat   13.6 M  (%0.17)


tüm doğrusal      41.9 M  (%0.52)


### Eğitim belleği: tam ince ayar mı, LoRA mı, QLoRA mı? (8 milyar parametre, kaba hesap)
Tam ince ayar (karışık hassasiyet + Adam): parametre başına ≈ 16 bayt (ağırlık, gradyan, Adam'ın iki momenti ve 32 bitlik ana kopya). Aktivasyonlar hariç.

In [17]:
N, lora = 8.03e9, 42e6      # LoRA: tüm doğrusal katmanlar, r = 16
for ad, b in [("tam ince ayar", N * 16),        # 16 bayt/parametre
              ("LoRA (BF16)", N * 2 + lora * 16),   # donuk model 2 bayt
              ("QLoRA (4 bit)", N * 0.5 + lora * 16)]:
    print(f"{ad:15s} ≈ {b/1e9:6.1f} GB")

tam ince ayar   ≈  128.5 GB
LoRA (BF16)     ≈   16.7 GB
QLoRA (4 bit)   ≈    4.7 GB


### Tam QLoRA eğitim kodu (Colab T4 GPU, internet gerekir)
Küçük bir modele (Qwen2.5-0.5B-Instruct) kendi soru–cevap örneklerinizle ince ayar. `bitsandbytes`, `trl` ve `datasets` kurulur. Gerçek bir projede yüzlerce–binlerce kaliteli örnek gerekir; burada yalnızca işleyişi görüyoruz.

In [ ]:
# !pip install -q -U transformers peft trl datasets bitsandbytes accelerate
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ad = "Qwen/Qwen2.5-0.5B-Instruct"
dort_bit = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(ad, quantization_config=dort_bit,
                                             device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(ad)

ornekler = [  # kendi verinizle değiştirin (sohbet biçiminde)
    {"messages": [{"role": "user", "content": "Dropout nedir?"},
                  {"role": "assistant", "content": "Eğitimde nöronların bir kısmını rastgele kapatarak aşırı öğrenmeyi azaltan yöntemdir."}]},
    {"messages": [{"role": "user", "content": "IoU nedir?"},
                  {"role": "assistant", "content": "İki kutunun kesişim alanının birleşim alanına oranıdır."}]},
] * 50
veri = Dataset.from_list(ornekler)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules="all-linear", task_type="CAUSAL_LM")
ayar = SFTConfig(output_dir="qlora-ders", num_train_epochs=1,
                 per_device_train_batch_size=4, learning_rate=2e-4,
                 logging_steps=5, fp16=True, report_to="none")
egitici = SFTTrainer(model=model, args=ayar, train_dataset=veri,
                     peft_config=lora, processing_class=tokenizer)
egitici.train()
egitici.model.save_pretrained("qlora-ders/adaptor")   # yalnızca birkaç MB

## P6 · Modeli kendi bilgisayarınızda çalıştırmak (Colab'da değil, kendi terminalinizde)
Ollama, llama.cpp üzerine kurulu bir araçtır: GGUF biçiminde nicemlenmiş modeli indirir ve `localhost:11434` adresinde OpenAI uyumlu bir API açar. Model adları ollama.com/library sayfasındadır.

In [ ]:
# 1) Kurulum (Linux/macOS): https://ollama.com/download
# !curl -fsSL https://ollama.com/install.sh | sh
# 2) Modeli indirip sohbet etmek (≈ 5 GB, 4 bit):
# !ollama run llama3.1:8b
# 3) Kendi Python kodunuzdan, OpenAI uyumlu yerel API ile:
import requests
cevap = requests.post("http://localhost:11434/v1/chat/completions", json={
    "model": "llama3.1:8b",
    "messages": [{"role": "user", "content": "GQA nedir? İki cümle."}],
    "temperature": 0.2})
print(cevap.json()["choices"][0]["message"]["content"])
# llama.cpp ile doğrudan:  llama-cli -m model-Q4_K_M.gguf -p "Merhaba"
# GPU sunucuda çok kullanıcı için:  vllm serve Qwen/Qwen2.5-7B-Instruct

### Alıştırma
Önce kâğıt üzerinde çözün, sonra hücreyi çalıştırıp karşılaştırın.

(a) Qwen2.5-7B'nin parametrelerini P1'deki formülle elle hesaplayın (dikkat: Qwen'de Q, K, V projeksiyonlarının bias'ı vardır). (b) 32 milyar parametreli bir model 4 bitte kaç GB tutar, T4'e sığar mı? (c) Soru 12(c)'deki MoE bloğunda k = 4 olsaydı aktif parametre ne olurdu?

In [ ]:
V, d, ff, L, h, kv = 152064, 3584, 18944, 28, 28, 4
hd = d // h
dikkat = 2*d*d + 2*d*kv*hd + (d + 2*kv*hd)       # + Q, K, V bias
blok = dikkat + 3*d*ff + 2*d
print("(a) Qwen2.5-7B:", f"{2*V*d + L*blok + d:,}".replace(",", " "))
print("(b) 32B, 4 bit:", 32e9 * 0.5 / 1e9, "GB -> T4 (16 GB): sığar ama bağlama az yer kalır")
print("(c) aktif:", (50e6 + 4 * 100e6) / 1e6, "milyon")